# 🗺️ Climate Model Evaluation: Spatial Mean & Bias Visualization (4x3 Grid)

### 📌 Overview
This script generates a high-quality, publication-ready (Nature Communications style) **4x3 spatial map matrix** to evaluate and compare precipitation downscaling models against observational data over India. It is structured into two main components:
1. **Rows 1-2 (Mean Precipitation):** Compares time-mean precipitation maps of observations and 5 distinct baseline/deep-learning models (`CA`, `WT`, `SRCNN`, `DDPM`, and `Flow Matching`).
2. **Rows 3-4 (Spatial Bias):** Visualizes the systematic errors (Model minus Observation) alongside specialized statistical distributions.

---

### 🎨 Key Design Features
* **Perfect 4x3 Symmetrical Layout:** Uses `matplotlib.gridspec` to construct an optimized grid with zero dead space. One slot in row 3 is cleanly reserved for a prominent text label.
* **Shared Discrete Colorbars:** Placed vertically on the right margin to maximize layout efficiency and maintain identical scales across models.
* **Publication Typography:** Standardized on *Times New Roman* font parameters with explicit high-DPI scaling constraints.

---

### 📊 Metric & Validation Pipelines

The notebook computes and overlays complex statistical validations directly onto the visual layouts:

#### 1. Spatial Error Metrics (Mean Maps)
For every model evaluation panel, the script dynamically computes performance layers against the target observations:
* **Root Mean Squared Error (RMSE):** Captures total error magnitude.
* **Peak Signal-to-Noise Ratio (PSNR):** Evaluates the spatial clarity of downscaled fields.
* **Structural Similarity Index (SSIM):** Leverages `scikit-image` to evaluate how well spatial structural patterns (like topography-driven rain bands) match reality.

#### 2. Bias Statistical Distributions (Bias Maps)
Each systematic error map is accompanied by a localized bounding card detailing spatial core characteristics:
* Mean spatial bias ($\mu$)
* Median spatial bias ($\tilde{x}$)
* Standard deviation of the bias ($\sigma$)

---

### 📂 Pipeline Steps Execution
1. **Data Loading:** Lazy-loads NetCDF (`.nc`) climatology datasets using `xarray` and subsets temporal (`2011–2014`) and geographic zones.
2. **Interpolation Check:** Assures structural resolution matching by auto-interpolating mismatching model lattices to the exact grid of the observation profile via nearest-neighbor mapping.
3. **Statistical Extraction:** Reduces multi-year arrays down to 2D time-mean surfaces and computes localized metrics.
4. **Layout Assembly:** Draws borders using a custom-provided shapefile boundary of India (`geopandas`) and overlays the analytics labels.

In [ ]:
"""
plot_mean_bias_maps_4x3.py
==========================
Nature Communications-style 4x3 spatial map figure.
  Rows 1-2 : Time-mean precipitation (Obs + 5 models) with RMSE, PSNR, SSIM
  Rows 3-4 : Spatial bias (5 models + 1 title cell) with Mean, Median, Std
  Right    : Shared vertical discrete colorbars

Design principles
  • 4x3 grid creates a perfectly symmetrical layout with no empty spaces.
  • Times New Roman, panel labels a–l, 300 dpi PDF + PNG
  • Scikit-Image used for robust structural similarity (SSIM) calculation
"""

import os
import warnings
import numpy as np
import xarray as xr
import geopandas as gpd
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import matplotlib.ticker as ticker
import matplotlib.colors as mcolors
from skimage.metrics import structural_similarity as ssim

warnings.filterwarnings("ignore")

matplotlib.rcParams.update({
    'font.family':       'serif',
    'font.serif':        ['Times New Roman', 'Times', 'DejaVu Serif'],
    'font.size':         8,
    'axes.labelsize':    8,
    'axes.titlesize':    9,
    'xtick.labelsize':   7,
    'ytick.labelsize':   7,
    'figure.dpi':        150,
    'axes.linewidth':    0.5,
    'pdf.fonttype':      42,
    'ps.fonttype':       42,
})

# ============================================================
# PATHS
# ============================================================
obs_path     = "/path/to/data/MSWX_Pcp_Daily_Ind_HR.nc"
ddpm_path    = ("/path/to/project/PhD_Precipitation/"
                "03_Code/precipitation-ddpm-india/scripts/ddpm/diffusr_climate/"
                "results/precip_ddpm_v5b/inference/ddpm_v5b_precipitation_test_set.nc")
srcnn_path   = ("/path/to/project/PhD_Precipitation/"
                "03_Code/results/srcnn_baseline/srcnn_pred_2011-2014.nc")
fm_path      = ("/path/to/project/Flow_matching_downscaling/"
                "results/precip_fm_v1/precip_fm_v1_heun50_merged_test.nc")
ca_path      = ("/path/to/project/PhD_Precipitation/"
                "03_Code/results/ca_baseline/ca_pred_2011-2014.nc")
wt_path      = ("/path/to/project/PhD_Precipitation/"
                "03_Code/results/weather_typing_baseline/wt_pred_2011-2014.nc")
shp_path     = ("/path/to/project/raw-data/shapefiles/ne_10m_admin_0_countries_ind/"
                "ne_10m_admin_0_countries_ind.shp")
output_dir   = ("/path/to/project/PhD_Precipitation/"
                "03_Code/notebooks/results/precip_ddpm_v5b")

time_slice = slice('2011-01-01', '2014-12-31')
lat_slice  = slice(39.95, 5.05)
lon_slice  = slice(65.05, 99.95)

# List of models to evaluate
MODEL_ORDER = ['CA', 'WT', 'SRCNN', 'DDPM', 'Flow Matching']

MODELS_PATHS = {
    'DDPM':          (ddpm_path,  'precipitation'),
    'SRCNN':         (srcnn_path, 'pr_srcnn'),
    'Flow Matching': (fm_path,    'precipitation'),
    'CA':            (ca_path,    'pr_ca'),
    'WT':            (wt_path,    'pr_wt'),
}

CMAP_MEAN = 'YlGnBu'
CMAP_BIAS = 'RdBu_r'


# ============================================================
# DATA LOADING
# ============================================================
def load_data():
    print("Loading shapefile...")
    gdf = gpd.read_file(shp_path)

    print("Loading observations...")
    ds_obs = xr.open_dataset(obs_path, chunks='auto')
    lat_min, lat_max = sorted([lat_slice.start, lat_slice.stop])
    obs_lat_slice = (slice(lat_max, lat_min) if ds_obs.lat[0] > ds_obs.lat[-1] else slice(lat_min, lat_max))
    da_obs   = ds_obs.sel(time=time_slice, lat=obs_lat_slice, lon=lon_slice)['precipitation']
    mean_obs = da_obs.mean(dim='time').compute()

    if mean_obs.lat[0] > mean_obs.lat[-1]:
        mean_obs = mean_obs.sortby('lat')

    lat = mean_obs.lat.values
    lon = mean_obs.lon.values
    extent = [lon.min(), lon.max(), lat.min(), lat.max()]

    vmax_mean = float(np.nanpercentile(mean_obs.values, 99))
    vmax_bias = 0.0

    mean_maps = {'Observation': mean_obs.values}
    bias_maps = {}

    print("Processing models...")
    for mname in MODEL_ORDER:
        path, varname = MODELS_PATHS[mname]
        print(f"  {mname}...")
        ds_m   = xr.open_dataset(path, chunks='auto')
        da_m   = ds_m.sel(time=time_slice)[varname]
        try:
            xr.testing.assert_allclose(mean_obs.lat, da_m.lat)
        except AssertionError:
            da_m = da_m.interp(lat=mean_obs.lat, lon=mean_obs.lon, method='nearest')
        mean_m = da_m.mean(dim='time').compute().sortby('lat')
        ds_m.close()

        mean_maps[mname] = mean_m.values
        bias             = mean_m.values - mean_obs.values
        bias_maps[mname] = bias

        vm = float(np.nanpercentile(mean_m.values, 99))
        if vm > vmax_mean:
            vmax_mean = vm
        bm = float(np.nanpercentile(np.abs(bias[np.isfinite(bias)]), 99))
        if bm > vmax_bias:
            vmax_bias = bm

    ds_obs.close()
    return gdf, lat, lon, extent, mean_maps, bias_maps, vmax_mean, vmax_bias


# ============================================================
# MAP AND STATS UTILITIES
# ============================================================
def draw_map(ax, data_2d, lat, lon, gdf, cmap, norm, extent):
    im = ax.imshow(
        data_2d, origin='lower', extent=extent,
        cmap=cmap, norm=norm,
        aspect='auto', interpolation='nearest', rasterized=True
    )
    gdf.boundary.plot(ax=ax, color='#1A1A1A', linewidth=0.45, zorder=5)
    ax.set_xlim(extent[0], extent[1])
    ax.set_ylim(extent[2], extent[3])
    ax.set_xticks([]); ax.set_yticks([])
    for spine in ax.spines.values():
        spine.set_linewidth(0.4)
    return im

def spatial_metrics_annotation(ax, model_2d, obs_2d, fontsize=6):
    """Calculates and annotates RMSE, PSNR, and SSIM."""
    valid = np.isfinite(model_2d) & np.isfinite(obs_2d)
    m1 = model_2d[valid]
    m2 = obs_2d[valid]
    
    if len(m1) > 1:
        # 1. RMSE
        mse = np.mean((m1 - m2)**2)
        rmse_val = np.sqrt(mse)
        
        # 2. PSNR
        data_range = np.max(m2) - np.min(m2)
        psnr_val = 10 * np.log10((data_range**2) / mse) if mse > 0 else float('inf')
        
        # 3. SSIM (Requires 2D shape, so we fill NaNs with 0)
        m1_fill = np.nan_to_num(model_2d, nan=0.0)
        m2_fill = np.nan_to_num(obs_2d, nan=0.0)
        ssim_val = ssim(m2_fill, m1_fill, data_range=data_range)
        
        # Format the text
        txt = (f"RMSE = {rmse_val:.2f}\n"
               f"PSNR = {psnr_val:.1f}\n"
               f"SSIM = {ssim_val:.3f}")
        
        ax.text(0.97, 0.97, txt, transform=ax.transAxes,
                fontsize=fontsize, va='top', ha='right', linespacing=1.4,
                bbox=dict(boxstyle='round,pad=0.25', fc='white', ec='#888888', lw=0.5, alpha=0.88),
                zorder=10)

def stats_annotation(ax, bias_2d, fontsize=6):
    """Calculates and annotates Bias Statistics."""
    flat = bias_2d[np.isfinite(bias_2d)]
    if len(flat) == 0: return
    b_mean = np.mean(flat)
    b_std  = np.std(flat)
    b_med  = np.median(flat)
    sign   = '+' if b_mean >= 0 else ''
    
    txt = (f"$\\mu$={sign}{b_mean:.2f}\n"
           f"$\\tilde{{x}}$={'+' if b_med>=0 else ''}{b_med:.2f}\n"
           f"$\\sigma$={b_std:.2f}")
    
    ax.text(0.97, 0.97, txt, transform=ax.transAxes,
            fontsize=fontsize, va='top', ha='right', linespacing=1.4,
            bbox=dict(boxstyle='round,pad=0.25', fc='white', ec='#888888', lw=0.5, alpha=0.88),
            zorder=10)


# ============================================================
# FIGURE (4x3 Layout + Discrete Colormaps)
# ============================================================
def make_figure(gdf, lat, lon, extent, mean_maps, bias_maps, vmax_mean, vmax_bias, output_dir):
    os.makedirs(output_dir, exist_ok=True)

    # Figure slightly narrower to accommodate 3 columns instead of 4
    FIG_W = 6.0
    FIG_H = 8.5
    fig = plt.figure(figsize=(FIG_W, FIG_H))
    fig.patch.set_facecolor('white')

    # PERFECT 4x3 GRID
    gs = gridspec.GridSpec(
        4, 3, figure=fig,
        hspace=0.25, wspace=0.05,
        left=0.02, right=0.82, top=0.92, bottom=0.05
    )

    levels_mean = ticker.MaxNLocator(nbins=10).tick_values(0, vmax_mean)
    cmap_mean = plt.get_cmap(CMAP_MEAN, len(levels_mean))
    norm_mean = mcolors.BoundaryNorm(levels_mean, cmap_mean.N, extend='max')

    levels_bias = ticker.MaxNLocator(nbins=10, symmetric=True).tick_values(-vmax_bias, vmax_bias)
    cmap_bias = plt.get_cmap(CMAP_BIAS, len(levels_bias) + 1)
    norm_bias = mcolors.BoundaryNorm(levels_bias, cmap_bias.N, extend='both')

    labels = list('abcdefghijkl')
    label_idx = 0
    all_mean_keys = ['Observation'] + MODEL_ORDER

    im_mean = None
    im_bias = None

    # --- ROWS 1 & 2: MEAN MAPS (6 panels total) ---
    for i, cname in enumerate(all_mean_keys):
        row = i // 3
        col = i % 3
        ax = fig.add_subplot(gs[row, col])
        
        im = draw_map(ax, mean_maps[cname], lat, lon, gdf, cmap_mean, norm_mean, extent)
        if im_mean is None: im_mean = im

        if cname != 'Observation':
            spatial_metrics_annotation(ax, mean_maps[cname], mean_maps['Observation'], fontsize=5.8)

        weight = 'bold' if cname == 'Observation' else 'normal'
        ax.set_title(cname, fontsize=8, fontweight=weight, color='#1A1A1A', pad=4)
        
        # Moved inside: y=0.96, va='top', added format "()", added a bbox for readability
        ax.text(0.03, 0.96, f"({labels[label_idx]})", transform=ax.transAxes,
                fontsize=9, fontweight='bold', va='top', ha='left',
                bbox=dict(boxstyle='round,pad=0.15', fc='white', ec='none', alpha=0.85),
                zorder=10)
        label_idx += 1

    # --- ROW 3, COL 0: TEXT LABEL ---
    ax_label = fig.add_subplot(gs[2, 0])
    ax_label.set_axis_off()
    ax_label.text(0.5, 0.5, 'Spatial Bias\n(Model $-$ Obs)',
                  ha='center', va='center', fontsize=10, fontweight='bold',
                  color='#444444', linespacing=1.6)

    # --- ROWS 3 & 4: BIAS MAPS (5 panels total) ---
    for i, mname in enumerate(MODEL_ORDER):
        idx = i + 1 
        row = 2 + (idx // 3)
        col = idx % 3
        ax = fig.add_subplot(gs[row, col])
        
        im = draw_map(ax, bias_maps[mname], lat, lon, gdf, cmap_bias, norm_bias, extent)
        if im_bias is None: im_bias = im
        
        stats_annotation(ax, bias_maps[mname], fontsize=5.8)

        ax.set_title(f"{mname} Bias", fontsize=8, fontweight='normal', color='#1A1A1A', pad=4)

        # Moved inside: y=0.96, va='top', added format "()", added a bbox for readability
        ax.text(0.03, 0.96, f"({labels[label_idx]})", transform=ax.transAxes,
                fontsize=9, fontweight='bold', va='top', ha='left',
                bbox=dict(boxstyle='round,pad=0.15', fc='white', ec='none', alpha=0.85),
                zorder=10)
        label_idx += 1

    # --- VERTICAL DISCRETE COLORBARS ---
    cax_mean = fig.add_axes([0.85, 0.54, 0.02, 0.35]) 
    cb_mean = plt.colorbar(im_mean, cax=cax_mean, orientation='vertical', extend='max', ticks=levels_mean)
    cb_mean.set_label('Time-mean precip.\n(mm day\u207b\u00b9)', fontsize=8, fontweight='bold', labelpad=10)
    cb_mean.ax.tick_params(labelsize=7, length=3)

    cax_bias = fig.add_axes([0.85, 0.07, 0.02, 0.35])
    cb_bias = plt.colorbar(im_bias, cax=cax_bias, orientation='vertical', extend='both', ticks=levels_bias)
    cb_bias.set_label('Spatial bias\n(mm day\u207b\u00b9)', fontsize=8, fontweight='bold', labelpad=10)
    cb_bias.ax.tick_params(labelsize=7, length=3)

    # --- Save ---
    # --- Save ---
    base_filename = 'precip_mean_bias_evaluation_2011_2014'
    out_png = os.path.join(output_dir, f'{base_filename}.png')
    out_pdf = os.path.join(output_dir, f'{base_filename}.pdf')
    
    plt.savefig(out_png, dpi=300, bbox_inches='tight', facecolor='white')
    plt.savefig(out_pdf, dpi=300, bbox_inches='tight', facecolor='white')
    print(f"\nSaved evaluation maps to:\n  {out_png}")
    
    plt.show()

# ============================================================
# MAIN
# ============================================================
if __name__ == '__main__':
    gdf, lat, lon, extent, mean_maps, bias_maps, vmax_mean, vmax_bias = load_data()
    make_figure(gdf, lat, lon, extent, mean_maps, bias_maps, vmax_mean, vmax_bias, output_dir)
    print("Done.")

In [ ]:
"""
plot_mean_bias_maps_4x3.py
==========================
Nature Communications-style 4x3 spatial map figure.
  Rows 1-2 : Time-mean precipitation (Obs + 5 models) with RMSE, PSNR, SSIM
  Rows 3-4 : Spatial % bias (5 models + 1 title cell) with Mean, Median, Std
  Right    : Shared vertical discrete colorbars

Design principles
  • 4x3 grid creates a perfectly symmetrical layout with no empty spaces.
  • Times New Roman, panel labels a–l, 300 dpi PDF + PNG
  • Scikit-Image used for robust structural similarity (SSIM) calculation
"""

import os
import warnings
import numpy as np
import xarray as xr
import geopandas as gpd
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import matplotlib.ticker as ticker
import matplotlib.colors as mcolors
from skimage.metrics import structural_similarity as ssim

warnings.filterwarnings("ignore")

matplotlib.rcParams.update({
    'font.family':       'serif',
    'font.serif':        ['Times New Roman', 'Times', 'DejaVu Serif'],
    'font.size':         8,
    'axes.labelsize':    8,
    'axes.titlesize':    9,
    'xtick.labelsize':   7,
    'ytick.labelsize':   7,
    'figure.dpi':        150,
    'axes.linewidth':    0.5,
    'pdf.fonttype':      42,
    'ps.fonttype':       42,
})

# ============================================================
# PATHS
# ============================================================
obs_path     = "/path/to/data/MSWX_Pcp_Daily_Ind_HR.nc"
ddpm_path    = ("/path/to/project/PhD_Precipitation/"
                "03_Code/precipitation-ddpm-india/scripts/ddpm/diffusr_climate/"
                "results/precip_ddpm_v5b/inference/ddpm_v5b_precipitation_test_set.nc")
srcnn_path   = ("/path/to/project/PhD_Precipitation/"
                "03_Code/results/srcnn_baseline/srcnn_pred_2011-2014.nc")
fm_path      = ("/path/to/project/Flow_matching_downscaling/"
                "results/precip_fm_v1/precip_fm_v1_heun50_merged_test.nc")
ca_path      = ("/path/to/project/PhD_Precipitation/"
                "03_Code/results/ca_baseline/ca_pred_2011-2014.nc")
wt_path      = ("/path/to/project/PhD_Precipitation/"
                "03_Code/results/weather_typing_baseline/wt_pred_2011-2014.nc")
shp_path     = ("/path/to/project/raw-data/shapefiles/ne_10m_admin_0_countries_ind/"
                "ne_10m_admin_0_countries_ind.shp")
output_dir   = ("/path/to/project/PhD_Precipitation/"
                "03_Code/notebooks/results/precip_ddpm_v5b")

time_slice = slice('2011-01-01', '2014-12-31')
lat_slice  = slice(39.95, 5.05)
lon_slice  = slice(65.05, 99.95)

# List of models to evaluate
MODEL_ORDER = ['CA', 'WT', 'SRCNN', 'DDPM', 'Flow Matching']

MODELS_PATHS = {
    'DDPM':          (ddpm_path,  'precipitation'),
    'SRCNN':         (srcnn_path, 'pr_srcnn'),
    'Flow Matching': (fm_path,    'precipitation'),
    'CA':            (ca_path,    'pr_ca'),
    'WT':            (wt_path,    'pr_wt'),
}

CMAP_MEAN = 'YlGnBu'
CMAP_BIAS = 'RdBu_r'


# ============================================================
# DATA LOADING
# ============================================================
def load_data():
    print("Loading shapefile...")
    gdf = gpd.read_file(shp_path)

    print("Loading observations...")
    ds_obs = xr.open_dataset(obs_path, chunks='auto')
    lat_min, lat_max = sorted([lat_slice.start, lat_slice.stop])
    obs_lat_slice = (slice(lat_max, lat_min) if ds_obs.lat[0] > ds_obs.lat[-1] else slice(lat_min, lat_max))
    da_obs   = ds_obs.sel(time=time_slice, lat=obs_lat_slice, lon=lon_slice)['precipitation']
    mean_obs = da_obs.mean(dim='time').compute()

    if mean_obs.lat[0] > mean_obs.lat[-1]:
        mean_obs = mean_obs.sortby('lat')

    lat = mean_obs.lat.values
    lon = mean_obs.lon.values
    extent = [lon.min(), lon.max(), lat.min(), lat.max()]

    vmax_mean = float(np.nanpercentile(mean_obs.values, 99))
    vmax_bias = 0.0

    mean_maps = {'Observation': mean_obs.values}
    bias_maps = {}

    print("Processing models...")
    for mname in MODEL_ORDER:
        path, varname = MODELS_PATHS[mname]
        print(f"  {mname}...")
        ds_m   = xr.open_dataset(path, chunks='auto')
        da_m   = ds_m.sel(time=time_slice)[varname]
        try:
            xr.testing.assert_allclose(mean_obs.lat, da_m.lat)
        except AssertionError:
            da_m = da_m.interp(lat=mean_obs.lat, lon=mean_obs.lon, method='nearest')
        mean_m = da_m.mean(dim='time').compute().sortby('lat')
        ds_m.close()

        mean_maps[mname] = mean_m.values
        
        # Calculate Percentage Bias safely with a 1 mm/day threshold
        diff = mean_m.values - mean_obs.values
        pct_bias = np.full_like(diff, np.nan)
        
        # Mask where observations are > 1 mm/day to avoid extreme outliers in dry regions
        valid_mask = mean_obs.values > 1.0
        pct_bias[valid_mask] = (diff[valid_mask] / mean_obs.values[valid_mask]) * 100
        bias_maps[mname] = pct_bias

        vm = float(np.nanpercentile(mean_m.values, 99))
        if vm > vmax_mean:
            vmax_mean = vm
            
        # Ignore NaNs when computing the max scale for the bias colorbar
        bm = float(np.nanpercentile(np.abs(pct_bias[np.isfinite(pct_bias)]), 99))
        if bm > vmax_bias:
            vmax_bias = bm

    ds_obs.close()
    return gdf, lat, lon, extent, mean_maps, bias_maps, vmax_mean, vmax_bias


# ============================================================
# MAP AND STATS UTILITIES
# ============================================================
def draw_map(ax, data_2d, lat, lon, gdf, cmap, norm, extent):
    im = ax.imshow(
        data_2d, origin='lower', extent=extent,
        cmap=cmap, norm=norm,
        aspect='auto', interpolation='nearest', rasterized=True
    )
    gdf.boundary.plot(ax=ax, color='#1A1A1A', linewidth=0.45, zorder=5)
    ax.set_xlim(extent[0], extent[1])
    ax.set_ylim(extent[2], extent[3])
    ax.set_xticks([]); ax.set_yticks([])
    for spine in ax.spines.values():
        spine.set_linewidth(0.4)
    return im

def spatial_metrics_annotation(ax, model_2d, obs_2d, fontsize=6):
    """Calculates and annotates RMSE, PSNR, and SSIM."""
    valid = np.isfinite(model_2d) & np.isfinite(obs_2d)
    m1 = model_2d[valid]
    m2 = obs_2d[valid]
    
    if len(m1) > 1:
        # 1. RMSE
        mse = np.mean((m1 - m2)**2)
        rmse_val = np.sqrt(mse)
        
        # 2. PSNR
        data_range = np.max(m2) - np.min(m2)
        psnr_val = 10 * np.log10((data_range**2) / mse) if mse > 0 else float('inf')
        
        # 3. SSIM (Requires 2D shape, so we fill NaNs with 0)
        m1_fill = np.nan_to_num(model_2d, nan=0.0)
        m2_fill = np.nan_to_num(obs_2d, nan=0.0)
        ssim_val = ssim(m2_fill, m1_fill, data_range=data_range)
        
        # Format the text
        txt = (f"RMSE = {rmse_val:.2f}\n"
               f"PSNR = {psnr_val:.1f}\n"
               f"SSIM = {ssim_val:.3f}")
        
        ax.text(0.97, 0.97, txt, transform=ax.transAxes,
                fontsize=fontsize, va='top', ha='right', linespacing=1.4,
                bbox=dict(boxstyle='round,pad=0.25', fc='white', ec='#888888', lw=0.5, alpha=0.88),
                zorder=10)

def stats_annotation(ax, bias_2d, fontsize=6):
    """Calculates and annotates Bias Statistics."""
    flat = bias_2d[np.isfinite(bias_2d)]
    if len(flat) == 0: return
    b_mean = np.mean(flat)
    b_std  = np.std(flat)
    b_med  = np.median(flat)
    sign   = '+' if b_mean >= 0 else ''
    
    txt = (f"$\\mu$={sign}{b_mean:.2f}%\n"
           f"$\\tilde{{x}}$={'+' if b_med>=0 else ''}{b_med:.2f}%\n"
           f"$\\sigma$={b_std:.2f}%")
    
    ax.text(0.97, 0.97, txt, transform=ax.transAxes,
            fontsize=fontsize, va='top', ha='right', linespacing=1.4,
            bbox=dict(boxstyle='round,pad=0.25', fc='white', ec='#888888', lw=0.5, alpha=0.88),
            zorder=10)


# ============================================================
# FIGURE (4x3 Layout + Discrete Colormaps)
# ============================================================
def make_figure(gdf, lat, lon, extent, mean_maps, bias_maps, vmax_mean, vmax_bias, output_dir):
    os.makedirs(output_dir, exist_ok=True)

    # Figure slightly narrower to accommodate 3 columns instead of 4
    FIG_W = 6.0
    FIG_H = 8.5
    fig = plt.figure(figsize=(FIG_W, FIG_H))
    fig.patch.set_facecolor('white')

    # PERFECT 4x3 GRID
    gs = gridspec.GridSpec(
        4, 3, figure=fig,
        hspace=0.25, wspace=0.05,
        left=0.02, right=0.82, top=0.92, bottom=0.05
    )

    levels_mean = ticker.MaxNLocator(nbins=10).tick_values(0, vmax_mean)
    cmap_mean = plt.get_cmap(CMAP_MEAN, len(levels_mean))
    norm_mean = mcolors.BoundaryNorm(levels_mean, cmap_mean.N, extend='max')

    levels_bias = ticker.MaxNLocator(nbins=10, symmetric=True).tick_values(-vmax_bias, vmax_bias)
    cmap_bias = plt.get_cmap(CMAP_BIAS, len(levels_bias) + 1)
    norm_bias = mcolors.BoundaryNorm(levels_bias, cmap_bias.N, extend='both')

    labels = list('abcdefghijkl')
    label_idx = 0
    all_mean_keys = ['Observation'] + MODEL_ORDER

    im_mean = None
    im_bias = None

    # --- ROWS 1 & 2: MEAN MAPS (6 panels total) ---
    for i, cname in enumerate(all_mean_keys):
        row = i // 3
        col = i % 3
        ax = fig.add_subplot(gs[row, col])
        
        im = draw_map(ax, mean_maps[cname], lat, lon, gdf, cmap_mean, norm_mean, extent)
        if im_mean is None: im_mean = im

        if cname != 'Observation':
            spatial_metrics_annotation(ax, mean_maps[cname], mean_maps['Observation'], fontsize=5.8)

        weight = 'bold' if cname == 'Observation' else 'normal'
        ax.set_title(cname, fontsize=8, fontweight=weight, color='#1A1A1A', pad=4)
        
        # Moved inside: y=0.96, va='top', added format "()", added a bbox for readability
        ax.text(0.03, 0.96, f"({labels[label_idx]})", transform=ax.transAxes,
                fontsize=9, fontweight='bold', va='top', ha='left',
                bbox=dict(boxstyle='round,pad=0.15', fc='white', ec='none', alpha=0.85),
                zorder=10)
        label_idx += 1

    # --- ROW 3, COL 0: TEXT LABEL ---
    ax_label = fig.add_subplot(gs[2, 0])
    ax_label.set_axis_off()
    
    # Updated text label to reflect the 1 mm/day threshold
    ax_label.text(0.5, 0.5, 'Percentage Bias\n(Where Obs > 1 mm/day)\n((Model $-$ Obs) / Obs $\\times$ 100)',
                  ha='center', va='center', fontsize=9, fontweight='bold',
                  color='#444444', linespacing=1.6)

    # --- ROWS 3 & 4: BIAS MAPS (5 panels total) ---
    for i, mname in enumerate(MODEL_ORDER):
        idx = i + 1 
        row = 2 + (idx // 3)
        col = idx % 3
        ax = fig.add_subplot(gs[row, col])
        
        im = draw_map(ax, bias_maps[mname], lat, lon, gdf, cmap_bias, norm_bias, extent)
        if im_bias is None: im_bias = im
        
        stats_annotation(ax, bias_maps[mname], fontsize=5.8)

        ax.set_title(f"{mname} % Bias", fontsize=8, fontweight='normal', color='#1A1A1A', pad=4)

        # Moved inside: y=0.96, va='top', added format "()", added a bbox for readability
        ax.text(0.03, 0.96, f"({labels[label_idx]})", transform=ax.transAxes,
                fontsize=9, fontweight='bold', va='top', ha='left',
                bbox=dict(boxstyle='round,pad=0.15', fc='white', ec='none', alpha=0.85),
                zorder=10)
        label_idx += 1

    # --- VERTICAL DISCRETE COLORBARS ---
    cax_mean = fig.add_axes([0.85, 0.54, 0.02, 0.35]) 
    cb_mean = plt.colorbar(im_mean, cax=cax_mean, orientation='vertical', extend='max', ticks=levels_mean)
    cb_mean.set_label('Time-mean precip.\n(mm day\u207b\u00b9)', fontsize=8, fontweight='bold', labelpad=10)
    cb_mean.ax.tick_params(labelsize=7, length=3)

    cax_bias = fig.add_axes([0.85, 0.07, 0.02, 0.35])
    cb_bias = plt.colorbar(im_bias, cax=cax_bias, orientation='vertical', extend='both', ticks=levels_bias)
    cb_bias.set_label('Percentage bias\n(%)', fontsize=8, fontweight='bold', labelpad=10)
    cb_bias.ax.tick_params(labelsize=7, length=3)

    # --- Save ---
    base_filename = 'precip_mean_pct_bias_evaluation_2011_2014_masked'
    out_png = os.path.join(output_dir, f'{base_filename}.png')
    out_pdf = os.path.join(output_dir, f'{base_filename}.pdf')
    
    plt.savefig(out_png, dpi=300, bbox_inches='tight', facecolor='white')
    plt.savefig(out_pdf, dpi=300, bbox_inches='tight', facecolor='white')
    print(f"\nSaved evaluation maps to:\n  {out_png}")
    
    plt.show()

# ============================================================
# MAIN
# ============================================================
if __name__ == '__main__':
    gdf, lat, lon, extent, mean_maps, bias_maps, vmax_mean, vmax_bias = load_data()
    make_figure(gdf, lat, lon, extent, mean_maps, bias_maps, vmax_mean, vmax_bias, output_dir)
    print("Done.")

# 🎻 Climate Zone Bias Distribution: Hybrid Half-Violin & Box Plots

### 📌 Overview
This script creates a highly specialized, multi-panel diagnostic graphic evaluating the distribution of systematic errors (spatial bias) across India's top 7 **Köppen-Geiger climate zones** for the target years 2011–2014. The plot validates four baseline models—`CA`, `WT`, `SRCNN`, and `DDPM`—alongside the primary proposed model, `Flow Matching`, to reveal geographic specificities in downscaling performance.

---

### 🎨 Hybrid Data Visualization Strategy
Instead of traditional, overlapping violin plots that hide descriptive statistics, this script builds a **custom hybrid configuration** for each model evaluation channel:
* **Left Half (Asymmetric Violin):** Generates a probability density curve showing the overall shape and modality of the bias data.
* **Right Half (Box-and-Whisker):** Integrates an exact interquartile range box patch (`FancyBboxPatch`) detailing explicit structural bounds:
  * **White Solid Bar:** Represents the median score.
  * **Diamond Indicator ($D$):** Pinpoints the computed arithmetic mean.
  * **Whisker Extensions:** Bounds regional variance within 1.5 times the Interquartile Range (IQR).

---

### 🗺️ Target Köppen-Geiger Regionalization
Data points are spatially extracted and isolated based on the following seven dominant climatological signatures over the Indian subcontinent:
* **Aw:** Tropical savanna climate (Wet/Dry)
* **Cwa:** Humid subtropical climate (Dry winter, hot summer)
* **BSh:** Hot semi-arid climate
* **ET:** Tundra / Alpine climate
* **BWh:** Hot desert climate
* **Am:** Tropical monsoon climate
* **Cwb:** Subtropical highland climate (Dry winter)

---

### 📊 Data Integration & Alignment Pipeline
1. **Mask Extraction:** Opens a legacy Köppen-Geiger classification GeoTIFF raster map file via `rioxarray` and projects spatial coordinates onto your standard grid resolution.
2. **Dynamic Clipping:** Utilizes a standard EPSG:4326 shapefile boundary of India to discard geographic noise outside country limits.
3. **Outlier Filtering & Compression:** Computes regional anomalies between model output and ground truth, then applies a strict statistical truncation algorithm ($1^{\text{st}}$ to $99^{\text{th}}$ percentile caps) to isolate outliers and optimize layout readability without skewing variance.
4. **Layout Resolution:** Arranges a multi-panel sub-grid via `gridspec`, leveraging the final available grid cell to anchor a detailed model registry legend map.

In [ ]:
"""
plot_zone_bias_violin.py
========================
Nature Communications-style spatial bias violin plot across
India's top 7 Köppen-Geiger climate zones (2011-2014).

Refactored from the dashboard script with:
  - Half-violin (left) + box-and-whisker (right) per model
  - Per-model colour palette
  - 7-panel layout (one per zone) in a single figure column
  - Ultra-minimalist layout: Only Köppen short codes (e.g., "Aw") as titles
  - Times New Roman, 300 dpi PDF + PNG output
  - Evaluates models: CA, WT, SRCNN, DDPM, Flow Matching
"""

import os
import warnings
import numpy as np
import xarray as xr
import rioxarray
import geopandas as gpd
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.gridspec as gridspec
from matplotlib.lines import Line2D

warnings.filterwarnings("ignore")

matplotlib.rcParams.update({
    'font.family':      'serif',
    'font.serif':       ['Times New Roman', 'Times', 'DejaVu Serif'],
    'font.size':        8,
    'axes.labelsize':   8.5,
    'axes.titlesize':   10.0,
    'xtick.labelsize':  7.5,
    'ytick.labelsize':  7.5,
    'legend.fontsize':  7.5,
    'figure.dpi':       150,
    'axes.linewidth':   0.6,
    'xtick.major.width': 0.6,
    'ytick.major.width': 0.6,
    'xtick.major.size': 3,
    'ytick.major.size': 3,
    'pdf.fonttype':     42,
    'ps.fonttype':      42,
})

# ============================================================
# PATHS
# ============================================================
obs_path     = "/path/to/data/MSWX_Pcp_Daily_Ind_HR.nc"
ddpm_path    = ("/path/to/project/PhD_Precipitation/"
                "03_Code/precipitation-ddpm-india/scripts/ddpm/diffusr_climate/"
                "results/precip_ddpm_v5b/inference/ddpm_v5b_precipitation_test_set.nc")
srcnn_path   = ("/path/to/project/PhD_Precipitation/"
                "03_Code/results/srcnn_baseline/srcnn_pred_2011-2014.nc")
fm_path      = ("/path/to/project/Flow_matching_downscaling/"
                "results/precip_fm_v1/precip_fm_v1_heun50_merged_test.nc")
ca_path      = ("/path/to/project/PhD_Precipitation/"
                "03_Code/results/ca_baseline/ca_pred_2011-2014.nc")
wt_path      = ("/path/to/project/PhD_Precipitation/"
                "03_Code/results/weather_typing_baseline/wt_pred_2011-2014.nc")
shp_path     = ("/path/to/project/raw-data/"
                "shapefiles/India_Boundary.shp")
tiff_path    = ("/path/to/project/raw-data/"
                "nc-files/Koppen_Geiger/koppen_geiger_0p1_1991_2020.tif")
output_dir   = ("/path/to/project/PhD_Precipitation/"
                "03_Code/notebooks/results/precip_ddpm_v5b")

time_slice = slice('2011-01-01', '2014-12-31')
lat_slice  = slice(39.95, 5.05)
lon_slice  = slice(65.05, 99.95)

# ============================================================
# CLIMATE ZONES  (top 7 by area)
# ============================================================
TARGET_ZONES = {
    3:  "Aw",
    11: "Cwa",
    6:  "BSh",
    29: "ET",
    4:  "BWh",
    2:  "Am",
    12: "Cwb",
}
ZONE_IDS = list(TARGET_ZONES.keys())

# ============================================================
# MODEL REGISTRY
# ============================================================
MODEL_ORDER = ['CA', 'WT', 'SRCNN', 'DDPM', 'Flow Matching']

PALETTE = {
    'CA':            '#A0C4C8',
    'WT':            '#78B7BC',
    'SRCNN':         '#C68B59',
    'DDPM':          '#8E44AD', 
    'Flow Matching': '#C0292B', 
}

MODELS_PATHS = {
    'Flow Matching': (fm_path,    'precipitation'),
    'DDPM':          (ddpm_path,  'precipitation'),
    'SRCNN':         (srcnn_path, 'pr_srcnn'),
    'CA':            (ca_path,    'pr_ca'),
    'WT':            (wt_path,    'pr_wt'),
}

ACTIVE_MODELS = [m for m in MODEL_ORDER if m in MODELS_PATHS]


# ============================================================
# DATA LOADING
# ============================================================
def load_data():
    print("Loading shapefile...")
    gdf = gpd.read_file(shp_path)

    print("Loading observations...")
    ds_obs = xr.open_dataset(obs_path, chunks='auto')
    lat_min, lat_max = sorted([lat_slice.start, lat_slice.stop])
    obs_lat_slice = (slice(lat_max, lat_min)
                     if ds_obs.lat[0] > ds_obs.lat[-1]
                     else slice(lat_min, lat_max))
    da_hr = (ds_obs.sel(time=time_slice, lat=obs_lat_slice, lon=lon_slice)['precipitation']
             .rio.set_spatial_dims(x_dim="lon", y_dim="lat")
             .rio.write_crs("epsg:4326")
             .rio.clip(gdf.geometry, gdf.crs, drop=False))
    mean_hr = da_hr.mean(dim='time').compute()

    print("Aligning Köppen-Geiger map...")
    da_kg = (rioxarray.open_rasterio(tiff_path)
             .squeeze().drop_vars('band')
             .rename({'y': 'lat', 'x': 'lon'}))
    da_kg_aligned = da_kg.interp(lat=mean_hr.lat, lon=mean_hr.lon,
                                  method='nearest')
    kg_flat      = da_kg_aligned.values.flatten()
    obs_mask     = np.isfinite(mean_hr.values.flatten())
    kg_valid     = np.where(obs_mask, kg_flat, 0)

    print("Computing per-zone bias arrays...")
    zone_bias = {zid: {} for zid in ZONE_IDS}

    for mname in ACTIVE_MODELS:
        path, varname = MODELS_PATHS[mname]
        print(f"  {mname}...")
        ds_m    = xr.open_dataset(path, chunks='auto')
        da_m    = ds_m.sel(time=time_slice)[varname]
        try:
            xr.testing.assert_allclose(mean_hr.lat, da_m.lat)
        except AssertionError:
            da_m = da_m.interp(lat=mean_hr.lat, lon=mean_hr.lon, method='nearest')
        mean_m  = da_m.mean(dim='time').compute()
        bias_f  = (mean_m - mean_hr).values.flatten()
        ds_m.close()

        for zid in ZONE_IDS:
            mask = (kg_valid == zid) & np.isfinite(bias_f)
            zone_bias[zid][mname] = bias_f[mask]

    ds_obs.close()
    return zone_bias


# ============================================================
# HALF-VIOLIN + BOX DRAWING UTILITY
# ============================================================
def draw_half_violin_box(ax, data_arr, x_pos, color, clip_sym=None):
    """
    Draw a half-violin (left) + box-and-whisker (right) at x_pos.
    """
    arr = data_arr[np.isfinite(data_arr)]
    if len(arr) < 5:
        return

    arr_plot = (arr[np.abs(arr) <= clip_sym] if clip_sym is not None else arr)
    if len(arr_plot) < 5:
        arr_plot = arr   # fall back to unclipped if too few points

    alpha_v = 0.28
    lw_box  = 0.7
    ec_box  = color
    bw      = 0.13   # box half-width (right side)

    # ── Violin (left half) ─────────────────────────────────
    vp = ax.violinplot([arr_plot], positions=[x_pos], widths=0.70,
                        showmeans=False, showmedians=False, showextrema=False)
    body = vp['bodies'][0]
    body.set_facecolor(color); body.set_alpha(alpha_v)
    body.set_edgecolor(color); body.set_linewidth(0.35)
    
    # Clip to left side
    verts = body.get_paths()[0].vertices
    verts[:, 0] = np.clip(verts[:, 0], -np.inf, x_pos)
    body.get_paths()[0].vertices = verts

    # ── Box (right half) ───────────────────────────────────
    q1, q2, q3 = np.percentile(arr_plot, [25, 50, 75])
    iqr  = q3 - q1
    wlo  = max(arr_plot.min(), q1 - 1.5 * iqr)
    whi  = min(arr_plot.max(), q3 + 1.5 * iqr)

    rect = mpatches.FancyBboxPatch(
        (x_pos, q1), bw, q3 - q1,
        boxstyle='square,pad=0',
        linewidth=lw_box, edgecolor=ec_box,
        facecolor=color,
        alpha=0.65,
        zorder=3)
    ax.add_patch(rect)

    # Median line
    ax.plot([x_pos, x_pos + bw], [q2, q2],
            color='white', lw=1.1,
            zorder=4, solid_capstyle='butt')
            
    # Whiskers
    xc = x_pos + bw / 2
    ax.plot([xc, xc], [wlo, q1], color=color, lw=0.6, zorder=2)
    ax.plot([xc, xc], [q3, whi], color=color, lw=0.6, zorder=2)
    for y_end in [wlo, whi]:
        ax.plot([x_pos + bw/4, x_pos + 3*bw/4], [y_end, y_end],
                color=color, lw=0.6, zorder=2)
                
    # Mean diamond
    ax.scatter([xc], [np.nanmean(arr_plot)],
               marker='D',
               s=9,
               color=color,
               edgecolors='dimgrey',
               linewidths=0.7, zorder=5)


# ============================================================
# MAIN FIGURE
# ============================================================
def plot_zone_bias(zone_bias, output_dir):
    os.makedirs(output_dir, exist_ok=True)
    n_zones = len(ZONE_IDS)
    n_col   = 4
    n_row   = 2   # 7 zones + 1 legend cell

    fig = plt.figure(figsize=(7.2, 5.5))
    fig.patch.set_facecolor('white')

    gs = gridspec.GridSpec(n_row, n_col, figure=fig,
                           hspace=0.70, wspace=0.35,
                           left=0.07, right=0.98,
                           top=0.90, bottom=0.15)

    panel_labels = list('abcdefg')
    axes = []
    for row in range(n_row):
        for col in range(n_col):
            axes.append(fig.add_subplot(gs[row, col]))

    n_models = len(ACTIVE_MODELS)
    positions = np.arange(n_models)

    for idx, zid in enumerate(ZONE_IDS):
        ax   = axes[idx]
        short_code = TARGET_ZONES[zid]

        # Determine symmetric clip (P1–P99 across all models, capped at 5)
        all_vals = np.concatenate([zone_bias[zid][m]
                                   for m in ACTIVE_MODELS
                                   if len(zone_bias[zid][m]) > 0])
        if len(all_vals) > 0:
            p1, p99 = np.percentile(np.abs(all_vals[np.isfinite(all_vals)]),
                                     [1, 99])
            clip_sym = min(p99 * 1.3, 5.0)
        else:
            clip_sym = 2.0

        # Draw each model
        for j, mname in enumerate(ACTIVE_MODELS):
            arr        = zone_bias[zid][mname]
            draw_half_violin_box(ax, arr, x_pos=j, color=PALETTE[mname], clip_sym=clip_sym)

        # Zero reference
        ax.axhline(0, color='#333333', lw=0.8, ls='--',
                   dashes=(4, 3), alpha=0.7, zorder=1)

        # Panel label
        ax.text(-0.15, 1.18, panel_labels[idx],
                transform=ax.transAxes,
                fontsize=10, fontweight='bold', va='top')

        # Clean title using just the short code
        ax.set_title(f'$\\bf{{{short_code}}}$', fontsize=10.5, pad=12, color='#1A1A1A')

        # Axes
        ax.set_xlim(-0.55, n_models - 0.45)
        ax.set_ylim(-clip_sym, clip_sym)
        ax.set_xticks(positions)
        ax.set_xticklabels(ACTIVE_MODELS, rotation=35, ha='right', fontsize=6.8)

        if idx % n_col == 0:
            ax.set_ylabel('Bias (mm day\u207b\u00b9)', fontsize=8)
        ax.yaxis.grid(True, linestyle=':', lw=0.4, color='#CCCCCC', zorder=0)
        ax.set_axisbelow(True)
        ax.spines['top'].set_visible(False)
        ax.spines['right'].set_visible(False)
        ax.spines['left'].set_linewidth(0.6)
        ax.spines['bottom'].set_linewidth(0.6)

    # ── Legend cell (8th subplot) ─────────────────────────────
    ax_leg = axes[7]
    ax_leg.set_axis_off()

    handles = []
    for mname in ACTIVE_MODELS:
        col  = PALETTE[mname]
        handles.append(mpatches.Patch(
            facecolor=col, edgecolor=col,
            linewidth=0.5, alpha=0.80, label=mname))
            
    handles += [
        Line2D([0], [0], marker='D', color='w',
               markerfacecolor='white', markeredgecolor='grey',
               markersize=5, linewidth=0, label='Mean'),
        Line2D([0], [0], color='white', lw=2,
               label='Median (white line)'),
        Line2D([0], [0], color='#333333', lw=0.9, ls='--',
               dashes=(4, 3), label='Zero bias'),
    ]
    ax_leg.legend(handles=handles, loc='center',
                  frameon=True, framealpha=0.95,
                  edgecolor='#CCCCCC', fancybox=False,
                  fontsize=7, handlelength=1.3,
                  handletextpad=0.5, labelspacing=0.55,
                  title='Models & markers', title_fontsize=7.5)


    # ── Save ─────────────────────────────────────────────────
    base_filename = 'precip_bias_by_climate_zone_2011_2014'
    out_pdf = os.path.join(output_dir, f'{base_filename}.pdf')
    out_png = os.path.join(output_dir, f'{base_filename}.png')
    
    # Uncomment to automatically save output
    fig.savefig(out_pdf, dpi=300, bbox_inches='tight', facecolor='white')
    fig.savefig(out_png, dpi=300, bbox_inches='tight', facecolor='white')
    print(f"\nSaved plots to:\n  {out_pdf}\n  {out_png}")
    plt.show()
    plt.close(fig)


# ============================================================
# MAIN
# ============================================================
if __name__ == '__main__':
    zone_bias = load_data()
    plot_zone_bias(zone_bias, output_dir)
    print("Done.")

# 📐 Spatial Texture Analysis: Structural Sharpness & Variance Distribution

### 📌 Overview
This script executes a computer vision pipeline to evaluate **spatial texture metrics** across 1,461 days (`2011–2014`). It quantifies whether downscaled fields look like realistic, high-resolution weather structures or artificially blurry averages. The resulting figure is an 183 mm wide, 2-panel publication visual mapping distributions of structural roughness and variance metrics across the 4 core models.

---

### 🧠 Mathematical Texture Profiling

To look past standard point-by-point errors, the code uses a 2D **Sobel Derivative Filter** along the spatial axes ($x, y$) to isolate structural gradients. For an observation or prediction surface $I$, the spatial gradient magnitude is computed as:

$$\nabla I = \sqrt{\left(\frac{\partial I}{\partial x}\right)^2 + \left(\frac{\partial I}{\partial y}\right)^2}$$

Using these spatial gradients, the script calculates two critical structural distributions:

#### Panel a: Roughness Ratio ($RR$)
Evaluates the absolute sharpness of boundaries (e.g., localized convective rainfall fronts). It compares the average gradient magnitude of the predicted field against the observation field:

$$RR = \frac{\mathbb{E}[\nabla I_{\text{pred}}]}{\mathbb{E}[\nabla I_{\text{obs}}]}$$

*   **$RR = 1.0$:** Perfect reconstruction of spatial sharpness.
*   **$RR < 1.0$:** The model is structurally oversmoothed or blurry (typical of conventional interpolation).
*   **$RR > 1.0$:** The model contains artificial spatial noise or artifacts.

#### Panel b: Variance Ratio ($VR$)
Measures the preservation of true spatial variability across the geographic area without spatial averaging degradation:

$$VR = \frac{\text{Var}(I_{\text{pred}})}{\text{Var}(I_{\text{obs}})}$$

---

### 🎨 Visual Layout & Safeguards
* **Asymmetric Data Splitting:** Implements the custom left-hand violin density plot balanced by a right-hand box-and-whisker structure (`FancyBboxPatch`).
* **DDPM Structural Highlight:** Places a custom crimson tone background band across the `DDPM v5b` channel to clearly highlight our core generative method's capacity to maintain spatial texture.
* **Outlier Truncation Guard:** Leverages a `clip_upper` parameter to cap extreme rendering ranges (e.g., $RR > 2.2$; $VR > 2.0$) dynamically, ensuring extreme single-day outliers do not squish or warp the primary visualization bounds.

In [ ]:
"""
plot_texture_violin.py
======================
Nature Communications-style violin + box plot for spatial texture metrics.

Reads the daily metric arrays produced by the Version 2 (Sobel) texture
script and generates a 4-panel publication figure.

Evaluates Models: CA, WT, SRCNN, DDPM, Flow Matching

Usage:
  Run AFTER the Sobel texture script has finished — this script simply
  collects the daily_rr, daily_tc, daily_grmse, daily_vr lists and plots them.

  Option A: Paste this into the texture script and call plot_texture_violin()
            after the main loop.
  Option B: Run standalone by re-computing the daily arrays (takes ~same time).
"""

import os
import numpy as np
import xarray as xr
import geopandas as gpd
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.gridspec as gridspec
from matplotlib.lines import Line2D
from scipy.ndimage import sobel
import warnings

warnings.filterwarnings("ignore")
matplotlib.rcParams.update({
    'font.family':        'serif',
    'font.serif':         ['Times New Roman', 'Times', 'DejaVu Serif'],
    'font.size':          8,
    'axes.labelsize':     9,
    'axes.titlesize':     9,
    'xtick.labelsize':    8,
    'ytick.labelsize':    8,
    'legend.fontsize':    7.5,
    'figure.dpi':         150,
    'axes.linewidth':     0.6,
    'xtick.major.width':  0.6,
    'ytick.major.width':  0.6,
    'xtick.major.size':   3,
    'ytick.major.size':   3,
    'pdf.fonttype':       42,   # embed fonts in PDF
    'ps.fonttype':        42,
})

# ============================================================
# PATHS  — edit to match your server
# ============================================================
obs_path     = "/path/to/data/MSWX_Pcp_Daily_Ind_HR.nc"
ddpm_path    = ("/path/to/project/PhD_Precipitation/"
                "03_Code/precipitation-ddpm-india/scripts/ddpm/diffusr_climate/"
                "results/precip_ddpm_v5b/inference/ddpm_v5b_precipitation_test_set.nc")
srcnn_path   = ("/path/to/project/PhD_Precipitation/"
                "03_Code/results/srcnn_baseline/srcnn_pred_2011-2014.nc")
fm_path      = ("/path/to/project/Flow_matching_downscaling/"
                "results/precip_fm_v1/precip_fm_v1_heun50_merged_test.nc")
ca_path      = ("/path/to/project/PhD_Precipitation/"
                "03_Code/results/ca_baseline/ca_pred_2011-2014.nc")
wt_path      = ("/path/to/project/PhD_Precipitation/"
                "03_Code/results/weather_typing_baseline/wt_pred_2011-2014.nc")
shp_path     = ("/path/to/project/raw-data/"
                "shapefiles/India_Boundary.shp")
output_dir   = ("/path/to/project/PhD_Precipitation/"
                "03_Code/notebooks/results/precip_ddpm_v5b")

time_slice = slice('2011-01-01', '2014-12-31')
lat_slice  = slice(39.95, 5.05)
lon_slice  = slice(65.05, 99.95)


# ============================================================
# MODEL REGISTRY
# ============================================================
MODEL_ORDER = ['CA', 'WT', 'SRCNN', 'DDPM', 'Flow Matching']

PALETTE = {
    'CA':            '#A0C4C8',
    'WT':            '#78B7BC',
    'SRCNN':         '#C68B59',
    'DDPM':          '#8E44AD',
    'Flow Matching': '#C0292B',
}

MODELS_PATHS = {
    'DDPM':          (ddpm_path,   'precipitation'),
    'Flow Matching': (fm_path,     'precipitation'),
    'SRCNN':         (srcnn_path,  'pr_srcnn'),
    'CA':            (ca_path,     'pr_ca'),
    'WT':            (wt_path,     'pr_wt'),
}


# ============================================================
# TEXTURE METRIC COMPUTATION  (verbatim from Version 2 script)
# ============================================================
def texture_metrics_precip(ref_2d, pred_2d, mask_crop):
    ir = ref_2d.astype(float).copy()
    ip = pred_2d.astype(float).copy()
    gr = np.sqrt(sobel(ir, axis=0)**2 + sobel(ir, axis=1)**2)
    gp = np.sqrt(sobel(ip, axis=0)**2 + sobel(ip, axis=1)**2)
    valid = mask_crop & ~np.isnan(gr) & ~np.isnan(gp)
    gr_f  = gr[valid]; gp_f = gp[valid]
    raw_r = ir[mask_crop & ~np.isnan(ir)]
    raw_p = ip[mask_crop & ~np.isnan(ip)]
    if len(gr_f) == 0 or len(raw_r) == 0:
        return {k: np.nan for k in
                ['Roughness ratio', 'Texture corr.', 'Gradient RMSE', 'Variance ratio']}
    rr     = float(np.mean(gp_f) / (np.mean(gr_f) + 1e-12))
    g_rmse = float(np.sqrt(np.mean((gp_f - gr_f)**2)))
    vr     = float(np.var(raw_p) / (np.var(raw_r) + 1e-12))
    tc     = (float(np.corrcoef(gr_f, gp_f)[0, 1])
              if np.std(gr_f) > 1e-6 and np.std(gp_f) > 1e-6 else np.nan)
    return {'Roughness ratio': rr, 'Texture corr.': tc,
            'Gradient RMSE': g_rmse, 'Variance ratio': vr}


def compute_all_daily_metrics():
    """Load data and compute daily texture metrics for all models."""
    print("Loading observations...")
    gdf    = gpd.read_file(shp_path)
    ds_obs = xr.open_dataset(obs_path)
    lat_min, lat_max = sorted([lat_slice.start, lat_slice.stop])
    obs_lat_slice = (slice(lat_max, lat_min) if ds_obs.lat[0] > ds_obs.lat[-1]
                     else slice(lat_min, lat_max))
    da_obs = ds_obs.sel(time=time_slice, lat=obs_lat_slice, lon=lon_slice)['precipitation']
    da_obs = da_obs.rio.set_spatial_dims(x_dim="lon", y_dim="lat")
    da_obs = da_obs.rio.write_crs("epsg:4326")
    da_obs = da_obs.rio.clip(gdf.geometry, gdf.crs, drop=False)
    da_obs = da_obs.compute()

    obs_vals  = da_obs.values
    mask_crop = ~np.isnan(obs_vals[0])
    T         = da_obs.sizes['time']

    all_metrics = {}   # model_name → dict of lists

    for mname in MODEL_ORDER:
        path, varname = MODELS_PATHS[mname]
        print(f"  Computing metrics for {mname}...")
        ds_m    = xr.open_dataset(path)
        da_m    = ds_m.sel(time=time_slice)[varname]
        try:
            xr.testing.assert_allclose(da_obs.lat, da_m.lat)
        except AssertionError:
            da_m = da_m.interp(lat=da_obs.lat, lon=da_obs.lon, method='nearest')
        da_m    = da_m.where(da_obs.notnull()).compute()
        mvals   = da_m.values

        rr_list, tc_list, gr_list, vr_list = [], [], [], []
        for t in range(T):
            m = texture_metrics_precip(obs_vals[t], mvals[t], mask_crop)
            rr_list.append(m['Roughness ratio'])
            tc_list.append(m['Texture corr.'])
            gr_list.append(m['Gradient RMSE'])
            vr_list.append(m['Variance ratio'])

        all_metrics[mname] = {
            'Roughness ratio': np.array(rr_list, dtype=float),
            'Texture corr.':   np.array(tc_list, dtype=float),
            'Gradient RMSE':   np.array(gr_list, dtype=float),
            'Variance ratio':  np.array(vr_list, dtype=float),
        }
        ds_m.close()

    ds_obs.close()
    return all_metrics


# ============================================================
# PLOTTING
# ============================================================
def violin_box_panel(ax, data_dict, metric_key,
                     ref_line=None, ref_label=None,
                     panel_label='', ylabel='', ylim=None,
                     clip_upper=None):
    """
    Draw one panel: half-violin (left) + box (right) per model,
    ordered as MODEL_ORDER, coloured by PALETTE.

    Parameters
    ----------
    ref_line   : float or None — horizontal reference line (e.g. 1.0 for ratios)
    clip_upper : float or None — clip display outliers above this for readability
                 (data not removed from stats, only from violin/box extent)
    """
    n      = len(MODEL_ORDER)
    pos    = np.arange(n)

    for i, mname in enumerate(MODEL_ORDER):
        raw = data_dict[mname][metric_key]
        arr = raw[~np.isnan(raw)]

        if clip_upper is not None:
            arr_plot = arr[arr <= clip_upper]
        else:
            arr_plot = arr

        col = PALETTE[mname]

        # ── Violin (left half only) ─────────────────────────
        if len(arr_plot) > 10:
            vp = ax.violinplot([arr_plot], positions=[i],
                               widths=0.65,
                               showmeans=False, showmedians=False,
                               showextrema=False)
            body = vp['bodies'][0]
            body.set_facecolor(col)
            body.set_alpha(0.30)
            body.set_edgecolor(col)
            body.set_linewidth(0.4)

            # Clip to left half
            verts = body.get_paths()[0].vertices
            verts[:, 0] = np.clip(verts[:, 0], -np.inf, i)
            body.get_paths()[0].vertices = verts

        # ── Box plot (right half) ────────────────────────────
        q1, q2, q3 = np.percentile(arr_plot, [25, 50, 75])
        iqr   = q3 - q1
        wlo   = max(arr_plot.min(), q1 - 1.5 * iqr)
        whi   = min(arr_plot.max(), q3 + 1.5 * iqr)
        bw    = 0.14   # half-width of box on right side

        # Box
        rect = mpatches.FancyBboxPatch(
            (i, q1), bw, q3 - q1,
            boxstyle='square,pad=0',
            linewidth=0.8,
            edgecolor=col,
            facecolor=col,
            alpha=0.70,
            zorder=3
        )
        ax.add_patch(rect)

        # Median line
        ax.plot([i, i + bw], [q2, q2],
                color='white', lw=1.2, zorder=4, solid_capstyle='butt')

        # Whiskers
        ax.plot([i + bw/2, i + bw/2], [wlo, q1],
                color=col, lw=0.7, zorder=2)
        ax.plot([i + bw/2, i + bw/2], [q3, whi],
                color=col, lw=0.7, zorder=2)
        ax.plot([i + bw/4, i + 3*bw/4], [wlo, wlo],
                color=col, lw=0.7, zorder=2)
        ax.plot([i + bw/4, i + 3*bw/4], [whi, whi],
                color=col, lw=0.7, zorder=2)

        # Mean dot
        ax.scatter([i + bw/2], [np.nanmean(arr_plot)],
                   marker='D', s=10,
                   color=col,
                   edgecolors='grey',
                   linewidths=0.6, zorder=5)

    # Reference line
    if ref_line is not None:
        ax.axhline(ref_line, color='#333333', lw=0.8, ls='--',
                   dashes=(4, 3), zorder=1, alpha=0.7)
        ax.text(n - 0.55, ref_line, ref_label,
                va='bottom', ha='right',
                fontsize=6.5, color='#444444', style='italic')

    # Axes formatting
    ax.set_xticks(pos)
    ax.set_xticklabels(MODEL_ORDER, rotation=30, ha='right',
                       fontsize=7.5)

    ax.set_xlim(-0.55, n - 0.45)
    if ylim:
        ax.set_ylim(ylim)
    ax.set_ylabel(ylabel, fontsize=8.5)
    ax.set_xlabel('')
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.spines['left'].set_linewidth(0.6)
    ax.spines['bottom'].set_linewidth(0.6)
    ax.yaxis.grid(True, linestyle=':', linewidth=0.4, color='#CCCCCC', zorder=0)
    ax.set_axisbelow(True)

    # Panel label inside the plot
    ax.text(0.03, 0.96, f"({panel_label})",
            transform=ax.transAxes,
            fontsize=10, fontweight='bold', va='top', ha='left',
            bbox=dict(boxstyle='round,pad=0.15', fc='white', ec='none', alpha=0.85),
            zorder=10)


def plot_texture_violin(all_metrics, output_dir):
    os.makedirs(output_dir, exist_ok=True)

    fig = plt.figure(figsize=(7.2, 4.2))   # 183 mm wide — Nature 2-column, single row
    fig.patch.set_facecolor('white')

    gs = gridspec.GridSpec(1, 2, figure=fig,
                           wspace=0.38,
                           left=0.10, right=0.97,
                           top=0.88, bottom=0.18)

    ax_rr = fig.add_subplot(gs[0, 0])
    ax_vr = fig.add_subplot(gs[0, 1])

    # ── Panel a: Roughness Ratio ─────────────────────────────
    violin_box_panel(
        ax_rr, all_metrics,
        metric_key='Roughness ratio',
        ref_line=1.0, ref_label='Perfect = 1',
        panel_label='a',
        ylabel='Roughness ratio',
        ylim=(0.0, 2.2),
        clip_upper=2.2,
    )
    ax_rr.set_title('Roughness ratio\n(spatial gradient magnitude, rel. to obs)',
                     fontsize=8, pad=4)

    # ── Panel b: Variance Ratio ──────────────────────────────
    violin_box_panel(
        ax_vr, all_metrics,
        metric_key='Variance ratio',
        ref_line=1.0, ref_label='Perfect = 1',
        panel_label='b',
        ylabel='Variance ratio',
        ylim=(0.0, 2.0),
        clip_upper=2.0,
    )
    ax_vr.set_title('Variance ratio\n(spatial variance, rel. to obs)',
                     fontsize=8, pad=4)

    # ── Shared legend ─────────────────────────────────────────
    legend_elements = []
    for mname in MODEL_ORDER:
        col  = PALETTE[mname]
        patch = mpatches.Patch(
            facecolor=col, edgecolor=col,
            linewidth=0.5,
            alpha=0.80, label=mname
        )
        legend_elements.append(patch)

    # Mean indicator
    legend_elements.append(
        Line2D([0], [0], marker='D', color='w',
               markerfacecolor='white', markeredgecolor='grey',
               markersize=5, label='Mean', linewidth=0)
    )
    legend_elements.append(
        Line2D([0], [0], color='white', lw=2,
               markerfacecolor='white', label='Median (white line)',
               solid_capstyle='butt')
    )

    fig.legend(handles=legend_elements,
               loc='upper center',
               bbox_to_anchor=(0.5, 1.00),
               ncol=7, frameon=False,  
               fontsize=7, handlelength=1.2,
               handletextpad=0.4, columnspacing=0.8)

    # ── Save ─────────────────────────────────────────────────
    base_filename = 'precip_spatial_texture_metrics_2011_2014'
    out_pdf = os.path.join(output_dir, f'{base_filename}.pdf')
    out_png = os.path.join(output_dir, f'{base_filename}.png')
    
    fig.savefig(out_pdf, dpi=300, bbox_inches='tight', facecolor='white')
    fig.savefig(out_png, dpi=300, bbox_inches='tight', facecolor='white')
    print(f"\nSaved:\n  {out_pdf}\n  {out_png}")
    plt.show()
    plt.close(fig)


# ============================================================
# MAIN
# ============================================================
if __name__ == '__main__':
    print("Computing daily spatial texture metrics (Sobel)...")
    all_metrics = compute_all_daily_metrics()

    print("\nGenerating figure...")
    plot_texture_violin(all_metrics, output_dir)
    print("Done.")

# 📡 Spatial Power Spectra Analysis: Radially Averaged Fast Fourier Transforms

### 📌 Overview
This script executes a 2D **Fast Fourier Transform (FFT)** diagnostic pipeline to compute the **Radially Averaged Power Spectra (RAPS)** across the precipitation grids. This allows us to verify how accurately each downscaling model reproduces the kinetic spatial energy of rain fields over different scale dimensions—ranging from massive synoptic systems down to fine-scale, high-frequency kinetic textures.

---

### 🧠 Mathematical Signal Transformation & Windowing

To perform spatial frequency domain analysis over non-periodic, bounded regional datasets without introducing artificial spectral leakage, the script implements a detailed preparation sequence on daily time frames:

1. **Spatial Detrending:** Isolates localized variance by subtracting the background spatial frame mean, dampening the $k=0$ zero-frequency energy spike.
2. **2D Hanning Tapering:** Multiplies each spatial surface against an isotropic spatial cosine taper window function to suppress edge boundary discontinuities before signal transformation:
   $$W(x, y) = \left[\frac{1}{2}\left(1 - \cos\left(\frac{2\pi x}{N_x-1}\right)\right)\right] \times \left[\frac{1}{2}\left(1 - \cos\left(\frac{2\pi y}{N_y-1}\right)\right)\right]$$
3. **2D Spectral Extraction:** Projects the windowed matrix into discrete spatial wavenumbers ($k_x, k_y$) using a 2D Forward FFT:
   $$F(k_x, k_y) = \mathcal{F}\big\{I(x,y) \cdot W(x,y)\big\}$$
4. **Radial Averaging:** Shifts zero-frequency components to the grid center, converts coordinate arrays to absolute distance wavenumbers $k = \sqrt{k_x^2 + k_y^2}$, and integrates the 2D energy map omnidirectionally into a 1D Power Spectral Density ($PSD$) profile.

---

### 📊 Dashboard Plot Architecture

The visualization constructs a 2-panel stack linking wave mechanics directly to geographic physical units:

#### 1. Top Panel: Power Spectral Density ($PSD$) Log-Log Graph
*   Maps spatial scale distribution ($x$-axis) directly against absolute spectral density amplitude ($y$-axis).
*   **Dual X-Axes:** Includes an inverted top axis showing the exact physical spatial wavelengths ($\text{Wavelength} = 1/k$) in kilometers ($\text{km}$).
*   **The Baseline Curve:** Incorporates a prominent solid black baseline trajectory mapping the ground truth `MSWX` observation profile.

#### 2. Bottom Panel: Spectral Ratio (Sim/Obs) Semilog Graph
*   Normalizes the structural energy distribution by mapping the exact scaling quotient ($\text{Ratio} = PSD_{\text{Model}} / PSD_{\text{Obs}}$).
*   **Interpretation Metric:** An ideal spatial reconstruction anchors perfectly along the horizontal line at $\text{Ratio} = 1.0$.
    *   **$\text{Ratio} < 1.0$ (Under-correction):** Highlights an energy deficit where models generate artificially soft, smoothed spatial trends.
    *   **$\text{Ratio} > 1.0$ (Over-correction):** Identifies regional frequency bands polluted by structural model artifacts or unconstrained noise.

In [ ]:
"""
plot_spatial_power_spectra_v2.py
================================
Two-panel spatial power spectrum figure for the precipitation downscaling
manuscript, using the isotropic (physically correct) RAPS.

  (a) Time-mean radially averaged power spectra, model vs MSWX, with top
      wavenumber axis.
  (b) Spectral ratio P_model / P_obs against spatial wavelength. The horizontal
      line at 1.0 is perfect agreement; the line at 0.5 is the effective-resolution
      threshold. A marker and vertical dotted drop-line indicate L_eff on the
      bottom spatial wavelength axis. Flow Matching carries no marker as it keeps
      >= half the observed power to the grid limit.

No in-panel text or annotation boxes: L_eff values live in Table 2 and the caption.
FM field is the Heun-50 merged field, consistent with Sect. 4.3.
"""

import os
import warnings
import numpy as np
import xarray as xr
import matplotlib
import matplotlib.pyplot as plt
from scipy.interpolate import interp1d

warnings.filterwarnings("ignore")

matplotlib.rcParams.update({
    'font.family':     'serif',
    'font.serif':      ['Times New Roman', 'Times', 'DejaVu Serif'],
    'font.size':       9,
    'axes.labelsize':  10,
    'xtick.labelsize': 9,
    'ytick.labelsize': 9,
    'legend.fontsize': 8.5,
    'figure.dpi':      150,
    'axes.linewidth':  0.8,
    'pdf.fonttype':    42,
    'ps.fonttype':     42,
})

# ============================================================ paths
obs_path   = "/path/to/data/MSWX_Pcp_Daily_Ind_HR.nc"
ddpm_path  = ("/path/to/project/PhD_Precipitation/"
              "03_Code/precipitation-ddpm-india/scripts/ddpm/diffusr_climate/"
              "results/precip_ddpm_v5b/inference/ddpm_v5b_precipitation_test_set.nc")
srcnn_path = ("/path/to/project/PhD_Precipitation/"
              "03_Code/results/srcnn_baseline/srcnn_pred_2011-2014.nc")
fm_path    = ("/path/to/project/Flow_matching_downscaling/"
              "results/precip_fm_v1/precip_fm_v1_heun50_merged_test.nc")
ca_path    = ("/path/to/project/PhD_Precipitation/"
              "03_Code/results/ca_baseline/ca_pred_2011-2014.nc")
wt_path    = ("/path/to/project/PhD_Precipitation/"
              "03_Code/results/weather_typing_baseline/wt_pred_2011-2014.nc")
output_dir = ("/path/to/project/PhD_Precipitation/"
              "03_Code/notebooks/results/precip_ddpm_v5b")

time_slice = slice('2011-01-01', '2014-12-31')
lat_slice  = slice(39.95, 5.05)
lon_slice  = slice(65.05, 99.95)
dx_km      = 11.0

# ============================================================ config
MODEL_ORDER = ['CA', 'WT', 'SRCNN', 'DDPM', 'Flow Matching']
OBS_C = '#1A1A1A'
PALETTE = {'CA': '#A0C4C8', 'WT': '#5B9AA0', 'SRCNN': '#C68B59',
           'DDPM': '#8E44AD', 'Flow Matching': '#C0292B'}
LINESTYLE = {'CA': (0, (5, 2)), 'WT': (0, (2, 2)), 'SRCNN': (0, (5, 2, 1, 2)),
             'DDPM': (0, (4, 1.5)), 'Flow Matching': 'solid'}
LINEWIDTH = {'CA': 1.3, 'WT': 1.3, 'SRCNN': 1.3, 'DDPM': 1.6, 'Flow Matching': 1.9}
ZORDER    = {'CA': 5, 'WT': 5, 'SRCNN': 6, 'DDPM': 8, 'Flow Matching': 9}
MODELS_PATHS = {
    'Flow Matching': (fm_path, 'precipitation'), 'DDPM': (ddpm_path, 'precipitation'),
    'SRCNN': (srcnn_path, 'pr_srcnn'), 'CA': (ca_path, 'pr_ca'), 'WT': (wt_path, 'pr_wt'),
}

# ============================================================ RAPS
def compute_time_mean_raps(data_3d, dx_km=11.0):
    """Time-mean radially averaged power spectrum, isotropic (Nyquist circle)."""
    n_time, ny, nx = data_3d.shape
    window_2d = np.hanning(ny)[:, None] * np.hanning(nx)[None, :]
    fy = np.fft.fftshift(np.fft.fftfreq(ny, d=dx_km))
    fx = np.fft.fftshift(np.fft.fftfreq(nx, d=dx_km))
    FX, FY = np.meshgrid(fx, fy)
    K_2d = np.sqrt(FX**2 + FY**2)
    dk = 1.0 / (min(ny, nx) * dx_km)
    k_bins = (K_2d / dk).astype(int)
    max_bin = min(ny, nx) // 2
    valid = (k_bins <= max_bin)
    mean_psd = np.zeros(max_bin + 1)
    valid_time_steps = 0
    for t in range(n_time):
        frame = data_3d[t]
        if np.isnan(frame).all():
            continue
        fmean = np.nanmean(frame)
        frame = (np.where(np.isnan(frame), fmean, frame) - fmean) * window_2d
        power_2d = np.abs(np.fft.fftshift(np.fft.fft2(frame))) ** 2
        tbin = np.bincount(k_bins[valid], power_2d[valid], minlength=max_bin + 1)
        nr = np.bincount(k_bins[valid], minlength=max_bin + 1)
        with np.errstate(invalid='ignore'):
            mean_psd += np.nan_to_num(tbin / nr, nan=0.0)
        valid_time_steps += 1
    mean_psd /= max(valid_time_steps, 1)
    physical_k = np.arange(max_bin + 1) * dk
    return physical_k[1:], mean_psd[1:]

def crossover_k(k, ratio, thr=0.5):
    """Wavenumber at which the ratio first drops below thr (None if it never does)."""
    idx = np.where(ratio < thr)[0]
    if len(idx) == 0:
        return None
    i = idx[0]
    if i == 0:
        return float(k[0])
    f = interp1d(ratio[i-1:i+1], k[i-1:i+1])
    return float(f(thr))

def process():
    ds_obs = xr.open_dataset(obs_path)
    lo, hi = sorted([lat_slice.start, lat_slice.stop])
    obs_lat = slice(hi, lo) if ds_obs.lat[0] > ds_obs.lat[-1] else slice(lo, hi)
    da_obs = ds_obs.sel(time=time_slice, lat=obs_lat, lon=lon_slice)['precipitation']
    k, p_obs = compute_time_mean_raps(da_obs.values, dx_km)
    models = {}
    for m in MODEL_ORDER:
        path, var = MODELS_PATHS[m]
        ds_m = xr.open_dataset(path).sel(time=time_slice)[var]
        try:
            xr.testing.assert_allclose(da_obs.lat, ds_m.lat)
        except AssertionError:
            ds_m = ds_m.interp(lat=da_obs.lat, lon=da_obs.lon, method='nearest')
        _, p = compute_time_mean_raps(ds_m.values, dx_km)
        models[m] = p
    ds_obs.close()
    return k, p_obs, models

# ============================================================ figure
def plot(k, p_obs, models, save=True):
    os.makedirs(output_dir, exist_ok=True)
    wl = 1.0 / k  # Convert coordinate vector to Spatial Wavelength (km)
    
    fig, (ax1, ax2) = plt.subplots(
        2, 1, figsize=(7.2, 7.4), sharex=True,
        gridspec_kw={'height_ratios': [2.3, 1], 'hspace': 0.09})
    fig.patch.set_facecolor('white')

    # -- (a) spectra --
    ax1.loglog(wl, p_obs, color=OBS_C, lw=2.4, zorder=10, label='MSWX (observed)')
    for m in MODEL_ORDER:
        ax1.loglog(wl, models[m], color=PALETTE[m], ls=LINESTYLE[m],
                   lw=LINEWIDTH[m], zorder=ZORDER[m], label=m)
    ax1.set_ylabel('Power spectral density')
    ax1.grid(True, which='both', ls=':', lw=0.4, alpha=0.45, color='#CCCCCC')
    for s in ('top', 'right'): ax1.spines[s].set_visible(False)
    ax1.legend(loc='lower left', frameon=True, framealpha=0.92,
               edgecolor='#CCCCCC', fancybox=False, handlelength=2.6)
    ax1.text(-0.10, 1.02, 'a', transform=ax1.transAxes, fontweight='bold', fontsize=12)

    # Top Secondary Axis: Wavenumber k (km^-1)
    secax = ax1.secondary_xaxis('top', functions=(lambda x: 1/np.where(x==0, np.nan, x),
                                                  lambda x: 1/np.where(x==0, np.nan, x)))
    secax.set_xlabel(r'Wavenumber $k$ (km$^{-1}$)', labelpad=8)

    # -- (b) ratio / bias --
    ax2.axhline(1.0, color=OBS_C, lw=1.6, zorder=7)
    ax2.axhline(0.5, color='#888888', lw=1.1, ls=(0, (4, 3)), zorder=6)
    
    for m in MODEL_ORDER:
        ratio = models[m] / (p_obs + 1e-30)
        ax2.semilogx(wl, ratio, color=PALETTE[m], ls=LINESTYLE[m],
                     lw=LINEWIDTH[m], zorder=ZORDER[m])
        kc = crossover_k(k, ratio, 0.5)
        if kc is not None and kc <= k.max():
            wlc = 1.0 / kc
            # Vertical dotted drop-line connecting marker down to bottom x-axis
            ax2.plot([wlc, wlc], [0.0, 0.5], color=PALETTE[m], ls=':', 
                     lw=1.2, alpha=0.85, zorder=ZORDER[m] - 1)
            ax2.scatter([wlc], [0.5], s=42, color=PALETTE[m], edgecolor='white',
                        linewidth=0.8, zorder=ZORDER[m] + 3)

    # Orient large scales (1000 km) on left and fine scales (22 km) on right
    ax2.set_xlim(wl.max(), wl.min())
    ax2.set_ylim(0.0, 1.3)
    
    # Bottom Primary Axis: Spatial Wavelength (km)
    ax2.set_xlabel('Spatial wavelength (km)')
    ax2.set_xticks([1000, 500, 200, 100, 50, 25])
    ax2.xaxis.set_major_formatter(plt.ScalarFormatter())
    
    ax2.set_ylabel(r'Ratio  $P_{\mathrm{model}}\,/\,P_{\mathrm{obs}}$')
    ax2.grid(True, which='both', ls=':', lw=0.4, alpha=0.45, color='#CCCCCC')
    for s in ('top', 'right'): ax2.spines[s].set_visible(False)
    ax2.text(-0.10, 1.05, 'b', transform=ax2.transAxes, fontweight='bold', fontsize=12)

    fig.tight_layout()
    if save:
        for ext in ('pdf', 'png'):
            fig.savefig(os.path.join(output_dir, f'fig_spatial_power_spectra.{ext}'),
                        dpi=300, bbox_inches='tight', facecolor='white')
    return fig

# ============================================================ main
if __name__ == '__main__':
    k, p_obs, models = process()
    plot(k, p_obs, models, save=True)
    print("Done.")

# 📈 Climatological Extremes: Side-by-Side Exceedance Probability & PDF Analysis

### 📌 Overview
This script builds a double-column ($180\text{ mm}$ width) diagnostic dashboard to analyze how well downscaling models simulate intense, extreme weather events. By combining an **Exceedance Probability** curve and a **Probability Density Function (PDF)** into a single execution loop, it evaluates whether downscaled predictions preserve real-world rainfall distribution tails or collapse into under-predicted, soft averages.

---

### 📊 Structural Layout & Statistical Foundations

Both panels rely on semi-logarithmic vertical axes ($\text{semilogy}$) to clearly expose structural behaviors at the extreme tails of the distribution where probabilities are low but risks are high:

#### Panel a: Exceedance Probability ($1-\text{CDF}$)
Maps the probability that rainfall on any given wet day will exceed a specific precipitation value ($x$):
$$P(X > x) = 1 - F(x)$$
*   **Purpose:** Measures systemic errors in predicting the duration and frequency of intense rainfall events. A steep drop-off indicates a model that structurally fails to generate extreme storm events.

#### Panel b: Probability Density Function ($\text{PDF}$)
Plots the relative likelihood of distinct rainfall amounts across discrete, uniform $3\text{ mm}$ bin intervals:
*   **Purpose:** Tracks how well the models preserve structural volume metrics across the full spectrum of wet days (defined as daily accumulation $\ge 1.0\text{ mm}$).

---

### 🏷️ Standardized Rainfall Intensity Reference Bands
To bridge statistical observations with meteorological risk assessment, the background of each panel is divided into five color-coded intensity categories. These match the official classification standards of the **India Meteorological Department (IMD)**:

| Rainfall Range ($\text{mm day}^{-1}$) | Background Color | Intensity Classification |
| :--- | :--- | :--- |
| **$1.0 \rightarrow 7.5$** | Blue (`#EBF5FB`) | **Light Rainfall** |
| **$7.5 \rightarrow 35.5$** | Green (`#D5F5E3`) | **Moderate Rainfall** |
| **$35.5 \rightarrow 64.5$** | Orange (`#FDEBD0`) | **Heavy Rainfall** |
| **$64.5 \rightarrow 115.5$** | Light Red (`#FADBD8`) | **Very Heavy Rainfall** |
| **$115.5 \rightarrow 300.0$** | Dark Red (`#F1948A`) | **Extremely Heavy Rainfall** |

---

### 🛠️ Optimization Features
* **Single Data Pass:** Loops through huge NetCDF dimensional layers exactly once to calculate both metrics simultaneously. This dramatically cuts down memory consumption and runtime.


In [ ]:
"""
plot_distribution_tail_v2.py
============================
Three-panel precipitation distribution / extremes figure for the downscaling
manuscript, replacing the earlier two-panel exceedance+PDF plot.

  (a) Exceedance probability P(X > x | wet)          -- distributional shape
  (b) Tail-frequency ratio  P_model / P_obs          -- tail error, isolates the
                                                        generative-vs-baseline gap
                                                        and separates FM from DDPM
  (c) Wet-day quantile-quantile vs observed          -- per-quantile fidelity,
                                                        reads off each baseline's
                                                        tail-truncation intensity
"""

import os
import warnings
import numpy as np
import xarray as xr
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import matplotlib.patches as mpatches

warnings.filterwarnings("ignore")

# ============================================================
# ENHANCED LEGIBILITY CONFIGURATION
# ============================================================
matplotlib.rcParams.update({
    'font.family':       'serif',
    'font.serif':        ['Times New Roman', 'Times', 'DejaVu Serif'],
    'font.size':         11,
    'axes.labelsize':    11.5,
    'axes.titlesize':    13,
    'xtick.labelsize':   10.5,
    'ytick.labelsize':   10.5,
    'legend.fontsize':   10,
    'figure.dpi':        150,
    'axes.linewidth':    0.8,
    'xtick.major.width': 0.8,
    'ytick.major.width': 0.8,
    'xtick.minor.width': 0.6,
    'xtick.major.size':  4.5,
    'ytick.major.size':  4.5,
    'xtick.minor.size':  2.5,
    'pdf.fonttype':      42,
    'ps.fonttype':       42,
})

# ============================================================
# PATHS & CONFIG
# ============================================================
obs_path   = "/path/to/data/MSWX_Pcp_Daily_Ind_HR.nc"
ddpm_path  = ("/path/to/project/PhD_Precipitation/"
              "03_Code/precipitation-ddpm-india/scripts/ddpm/diffusr_climate/"
              "results/precip_ddpm_v5b/inference/ddpm_v5b_precipitation_test_set.nc")
srcnn_path = ("/path/to/project/PhD_Precipitation/"
              "03_Code/results/srcnn_baseline/srcnn_pred_2011-2014.nc")
fm_path    = ("/path/to/project/Flow_matching_downscaling/results/precip_fm_v1/precip_fm_v1_heun50_merged_test.nc")
ca_path    = ("/path/to/project/PhD_Precipitation/"
              "03_Code/results/ca_baseline/ca_pred_2011-2014.nc")
wt_path    = ("/path/to/project/PhD_Precipitation/"
              "03_Code/results/weather_typing_baseline/wt_pred_2011-2014.nc")
output_dir = ("/path/to/project/PhD_Precipitation/"
              "03_Code/notebooks/results/precip_ddpm_v5b")

time_slice = slice('2011-01-01', '2014-12-31')
lat_slice  = slice(39.95, 5.05)
lon_slice  = slice(65.05, 99.95)
WET_THR    = 1.0     # mm/day, ETCCDI wet-day threshold
X_MAX      = 300.0   # display / grid ceiling, mm/day

MODEL_ORDER = ['CA', 'WT', 'SRCNN', 'DDPM', 'Flow Matching']

PALETTE = {
    'CA':            '#A0C4C8',
    'WT':            '#5B9AA0',   
    'SRCNN':         '#C68B59',
    'DDPM':          '#8E44AD',
    'Flow Matching': '#C0292B',
}
LINESTYLE = {
    'CA':            (0, (4, 2)),
    'WT':            (0, (2, 2)),
    'SRCNN':         (0, (4, 2, 1, 2)),
    'DDPM':          (0, (1, 1)),
    'Flow Matching': 'solid',
}
# Slightly boosted line widths for better visibility against larger fonts
LINEWIDTH = {'CA': 1.1, 'WT': 1.1, 'SRCNN': 1.1, 'DDPM': 1.3, 'Flow Matching': 1.8}

MODELS_PATHS = {
    'Flow Matching': (fm_path,    'precipitation'),
    'DDPM':          (ddpm_path,  'precipitation'),
    'SRCNN':         (srcnn_path, 'pr_srcnn'),
    'CA':            (ca_path,    'pr_ca'),
    'WT':            (wt_path,    'pr_wt'),
}

INTENSITY_BANDS = [
    (WET_THR, 7.5,   '#EBF5FB', 'Light'),
    (7.5,     35.5,  '#D5F5E3', 'Moderate'),
    (35.5,    64.5,  '#FDEBD0', 'Heavy'),
    (64.5,    115.5, '#FADBD8', 'Very Heavy'),
    (115.5,   X_MAX, '#F1948A', 'Extremely Heavy'),
]

# shared grids
X_GRID = np.logspace(np.log10(WET_THR), np.log10(X_MAX), 260)
Q_LEVELS = np.sort(1.0 - np.logspace(-4, np.log10(0.5), 220))

REPORT_X = 200.0     
REPORT_Q = 0.999     


# ============================================================
# COMPUTATION
# ============================================================
def wet_exceedance_on_grid(values_flat, x_grid, thr=WET_THR):
    """P(X > x | X >= thr) evaluated on x_grid via a single sort + searchsorted."""
    v = np.sort(values_flat[np.isfinite(values_flat) & (values_flat >= thr)])
    n = v.size
    if n == 0:
        return np.full(x_grid.shape, np.nan)
    cnt_gt = n - np.searchsorted(v, x_grid, side='right')
    p = cnt_gt / n
    p[p <= 0] = np.nan
    return p


def wet_quantiles(values_flat, q_levels, thr=WET_THR):
    v = values_flat[np.isfinite(values_flat) & (values_flat >= thr)]
    if v.size == 0:
        return np.full(q_levels.shape, np.nan)
    return np.quantile(v, q_levels)


def compute_all(values_flat):
    return {
        "P":   wet_exceedance_on_grid(values_flat, X_GRID),
        "q":   wet_quantiles(values_flat, Q_LEVELS),
        "p999": float(wet_quantiles(values_flat, np.array([REPORT_Q]))[0]),
    }


def load_data():
    print("Loading observations...")
    ds_obs = xr.open_dataset(obs_path)
    lat_min, lat_max = sorted([lat_slice.start, lat_slice.stop])
    obs_lat = (slice(lat_max, lat_min) if ds_obs.lat[0] > ds_obs.lat[-1]
               else slice(lat_min, lat_max))
    da_obs = ds_obs.sel(time=time_slice, lat=obs_lat, lon=lon_slice)['precipitation']
    obs_pack = compute_all(da_obs.values.flatten())

    model_packs = {}
    for mname in MODEL_ORDER:
        path, varname = MODELS_PATHS[mname]
        print(f"  Loading {mname}...")
        with xr.open_dataset(path) as ds_m:
            da_m = ds_m.sel(time=time_slice)[varname]
            try:
                xr.testing.assert_allclose(da_obs.lat, da_m.lat)
                xr.testing.assert_allclose(da_obs.lon, da_m.lon)
            except AssertionError:
                da_m = da_m.sel(lat=da_obs.lat, lon=da_obs.lon, method='nearest')
            model_packs[mname] = compute_all(da_m.values.flatten())

    ds_obs.close()
    return obs_pack, model_packs


# ============================================================
# FIGURE
# ============================================================
def _bands(ax, alpha=0.55):
    """Draws the IMD background bands without adding text labels inside the plot."""
    for x_lo, x_hi, fc, lab in INTENSITY_BANDS:
        x_hi_p = min(x_hi, X_MAX)
        if x_lo >= X_MAX:
            continue
        ax.axvspan(x_lo, x_hi_p, color=fc, alpha=alpha, zorder=0, lw=0)


def plot_combined(obs_pack, model_packs, save=True):
    os.makedirs(output_dir, exist_ok=True)
    
    # Increased figure size for clear reading at 100% scale
    fig, (ax_exc, ax_ratio, ax_qq) = plt.subplots(1, 3, figsize=(12.5, 5.2))
    fig.patch.set_facecolor('white')

    # ---- (a) exceedance probability ------------------------------------
    _bands(ax_exc)
    ax_exc.semilogy(X_GRID, obs_pack["P"], color='#1A1A1A', lw=1.8,
                    label='MSWX (Observed)', zorder=10, solid_capstyle='round')
    for m in MODEL_ORDER:
        is_fm = (m == 'Flow Matching')
        ax_exc.semilogy(X_GRID, model_packs[m]["P"], color=PALETTE[m],
                        lw=LINEWIDTH[m], ls=LINESTYLE[m], label=m,
                        zorder=9 if is_fm else 5, alpha=1.0 if is_fm else 0.9)
    ax_exc.set_xlim(WET_THR, X_MAX)
    ax_exc.set_ylim(5e-5, 1.5)
    ax_exc.set_xlabel('Daily precipitation (mm day\u207b\u00b9)', labelpad=6)
    ax_exc.set_ylabel('Exceedance probability  P(X\u2009>\u2009x\u2009|\u2009wet)', labelpad=6)
    ax_exc.set_title('Exceedance probability', fontweight='bold', pad=10)

    # ---- (b) tail-frequency ratio to observed --------------------------
    _bands(ax_ratio, alpha=0.45)
    ax_ratio.axhline(1.0, color='#1A1A1A', lw=1.4, ls=(0, (5, 2)), zorder=10,
                     label='Observed baseline')
    with np.errstate(divide='ignore', invalid='ignore'):
        for m in MODEL_ORDER:
            ratio = model_packs[m]["P"] / obs_pack["P"]
            ratio[~np.isfinite(ratio)] = np.nan
            is_fm = (m == 'Flow Matching')
            ax_ratio.semilogy(X_GRID, ratio, color=PALETTE[m], lw=LINEWIDTH[m],
                              ls=LINESTYLE[m], label=m,
                              zorder=9 if is_fm else 5, alpha=1.0 if is_fm else 0.9)
    ax_ratio.set_xlim(WET_THR, X_MAX)
    ax_ratio.set_ylim(1e-3, 3.0)
    ax_ratio.set_xlabel('Daily precipitation (mm day\u207b\u00b9)', labelpad=6)
    ax_ratio.set_ylabel('Tail-frequency ratio  P$_{\\mathrm{model}}$ / P$_{\\mathrm{obs}}$', labelpad=6)
    ax_ratio.set_title('Tail-frequency ratio to observed', fontweight='bold', pad=10)

    # ---- (c) wet-day quantile-quantile ---------------------------------
    oq = obs_pack["q"]
    qmax = np.nanmax(oq) * 1.02
    _bands(ax_qq, alpha=0.30)
    ax_qq.plot([0, X_MAX], [0, X_MAX], color='#1A1A1A', lw=1.4, ls=(0, (5, 2)),
               zorder=10, label='1:1 (perfect)')
    for m in MODEL_ORDER:
        is_fm = (m == 'Flow Matching')
        ax_qq.plot(oq, model_packs[m]["q"], color=PALETTE[m], lw=LINEWIDTH[m],
                   ls=LINESTYLE[m], label=m,
                   zorder=9 if is_fm else 5, alpha=1.0 if is_fm else 0.9)
    ax_qq.set_xlim(0, qmax)
    ax_qq.set_ylim(0, qmax)
    ax_qq.set_aspect('equal', adjustable='box')
    ax_qq.set_xlabel('Observed quantile (mm day\u207b\u00b9)', labelpad=6)
    ax_qq.set_ylabel('Model quantile (mm day\u207b\u00b9)', labelpad=6)
    ax_qq.set_title('Wet-day quantile\u2013quantile', fontweight='bold', pad=10)

    # ---- shared cosmetics ----------------------------------------------
    for ax in (ax_exc, ax_ratio, ax_qq):
        ax.grid(True, which='major', ls=':', lw=0.5, color='#BBBBBB', zorder=1)
        ax.spines['top'].set_visible(False)
        ax.spines['right'].set_visible(False)
    for ax in (ax_exc, ax_ratio):
        ax.xaxis.set_major_locator(ticker.MultipleLocator(50))
        ax.xaxis.set_minor_locator(ticker.MultipleLocator(10))

    # Panel letters - enlarged to 14pt Bold for striking visibility
    for ax, lbl in zip((ax_exc, ax_ratio, ax_qq), ('a', 'b', 'c')):
        ax.text(-0.16, 1.05, lbl, transform=ax.transAxes, fontsize=14,
                fontweight='bold', va='top')

    # ---- LEGENDS -------------------------------------------------------
    # Allocated bottom 20% of figure height explicitly for the two legend rows
    plt.tight_layout(rect=(0, 0.20, 1, 1))

    # 1. Models Legend
    handles, labels = ax_exc.get_legend_handles_labels()
    fig.legend(handles, labels, loc='lower center', ncol=6, frameon=False,
               bbox_to_anchor=(0.5, 0.11), handlelength=2.5, columnspacing=1.8)
               
    # 2. IMD Classes Legend (Custom Patches)
    imd_handles = []
    for x_lo, x_hi, fc, lab in INTENSITY_BANDS:
        range_str = f"{x_lo:g}–{x_hi:g}" if x_hi < X_MAX else f"≥{x_lo:g}"
        p = mpatches.Patch(facecolor=fc, edgecolor='#999999', lw=0.8, alpha=0.7, 
                           label=f"{lab} ({range_str})")
        imd_handles.append(p)
        
    fig.legend(handles=imd_handles, loc='lower center', ncol=5, frameon=False,
               bbox_to_anchor=(0.5, 0.03), handlelength=1.8, handleheight=1.2, columnspacing=1.8)

    if save:
        for ext in ('pdf', 'png'):
            out = os.path.join(output_dir, f'fig_distribution_tail.{ext}')
            fig.savefig(out, dpi=300, bbox_inches='tight', facecolor='white')
            print(f"  saved {out}")
    return fig


def print_key_numbers(obs_pack, model_packs):
    """Report the landmarks the Results text quotes, for the caption / prose."""
    ix = int(np.argmin(np.abs(X_GRID - REPORT_X)))
    print(f"\n99.9th wet-day percentile (mm/day) and P(X>{REPORT_X:.0f})/P_obs:")
    print(f"  {'obs':<14s} p999={obs_pack['p999']:6.1f}")
    
    for m in MODEL_ORDER:
        mod_p = np.nan_to_num(model_packs[m]['P'][ix], nan=0.0)
        ratio = mod_p / obs_pack['P'][ix]
        
        print(f"  {m:<14s} p999={model_packs[m]['p999']:6.1f}   "
              f"ratio@{REPORT_X:.0f}={ratio:5.2f}")

# ============================================================
# MAIN
# ============================================================
if __name__ == '__main__':
    obs_pack, model_packs = load_data()
    print_key_numbers(obs_pack, model_packs)
    plot_combined(obs_pack, model_packs, save=True)
    print("Done.")

# 🗺️ Hydroclimatic Mass Transport: Volumetric Class Redistribution Matrix

### 📌 Overview
This script builds a publication-grade diagnostic **Volumetric Heatmap Matrix**. Instead of simply counting the number of rainy days, this diagnostic maps out where the actual physical mass of water is allocated across different intensity regimes. It compares baseline models and your core generative downscaling framework (`Flow Matching`) against observational ground truth (`MSWX`).

---

### 🧠 Scientific Utility & Diagnostics

This plot is a powerful way to evaluate hydroclimatic models because it exposes structural failures that standard summary statistics like Mean Bias or RMSE completely miss:

1. **The Drizzle Bias Trap:** Conventional downscaling methods often suffer from an artificial "drizzle bias"—where models dump too much water into frequent, low-intensity rainfall bins rather than consolidating it into realistic storms. This matrix immediately flags that issue.
2. **Extreme Event Truncation:** This matrix checks if downscaling models preserve the massive amount of water carried by extreme events. It clearly shows whether a model underpredicts heavy monsoonal events, causing a structural deficit in the upper tiers.

---

### 📊 Mathematical Formulations & Data Processing

#### 1. Volumetric Allocation Fractional Weights
For any isolated rainfall accumulation tier defined by a lower bracket ($L$) and upper threshold ($H$), the fractional metric extracts all spatial locations and dates falling inside those boundaries and calculates their percentage of the total water volume:

$$\text{Fraction}_{L \rightarrow H} = \left( \frac{\sum I(x,y,t) \quad \forall \quad L \le I(x,y,t) < H}{\sum I(x,y,t) \quad \forall \quad I(x,y,t) \ge 1.0\text{ mm}} \right) \times 100.0$$

#### 2. Mass Transport Anomaly Mapping
To make model variations immediately obvious, the plot background displays an **Observed Difference Matrix**. It subtracts the base observation weights from each model cell to reveal explicit shifts in the water budget:

$$\Delta \text{Volume Fraction} = \text{Fraction}_{\text{Model}} - \text{Fraction}_{\text{Obs}}$$

---

### 🎨 Visual Architecture & Styling Elements
* **Diverging Mapping Scale:** Anchors a `coolwarm` diverging colormap pinned between strict clipping bounds ($-15\%$ to $+15\%$). This helps reviewers instantly identify patterns of model misallocation:
  * <span style="color:#C0292B">**Warm Tones (Red):**</span> Highlight zones where models dump an unrealistic surplus of water volume.
  * <span style="color:#A0C4C8">**Cool Tones (Blue):**</span> Highlight severe water volume deficits compared to real-world observations.
* **Dual-Layer Annotations:** Each grid cell prints the absolute, raw percentage volume from the model array, while the underlying background color displays the deviation anomaly relative to ground truth.
* **Structural Break Isolation:** Uses clear, white cross-sectional grid borders to segment the different models and official IMD precipitation categories.

In [ ]:
"""
plot_heatmap_matrix.py
======================

Nature Communications–style
Hydroclimatic Volumetric Heatmap Matrix

Scientific Purpose
------------------
This figure visualizes how precipitation volume
is redistributed across rainfall intensity regimes
for different downscaling models.

Why this figure is scientifically strong:
-----------------------------------------
1. Compact multi-model comparison
2. Reviewer-friendly
3. Publication efficient
4. Immediately exposes drizzle bias
5. Highlights heavy/extreme rainfall deficits
6. Scalable to CMIP6 ensembles and regional studies

Interpretation
--------------
Cell value:
    Percentage contribution to total rainfall volume

Rows:
    Models

Columns:
    Rainfall intensity classes

A physically realistic model should reproduce:
- observed heavy-tail contribution
- extreme-event transport
- volumetric rainfall allocation

Author: user Kumar Jha
"""

import os
import warnings
import numpy as np
import xarray as xr
import matplotlib
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore")

# ============================================================
# MATPLOTLIB STYLE
# ============================================================
matplotlib.rcParams.update({
    'font.family':       'serif',
    'font.serif':        ['Times New Roman', 'Times', 'DejaVu Serif'],
    'font.size':         10,
    'axes.labelsize':    11,
    'axes.titlesize':    13,
    'xtick.labelsize':   9,
    'ytick.labelsize':   10,
    'figure.dpi':        180,
    'axes.linewidth':    0.8,
    'pdf.fonttype':      42,
    'ps.fonttype':       42,
})

# ============================================================
# PATHS
# ============================================================
obs_path = "/path/to/data/MSWX_Pcp_Daily_Ind_HR.nc"
ddpm_path = (
    "/path/to/project/PhD_Precipitation/"
    "03_Code/precipitation-ddpm-india/scripts/ddpm/diffusr_climate/"
    "results/precip_ddpm_v5b/inference/ddpm_v5b_precipitation_test_set.nc"
)
srcnn_path = (
    "/path/to/project/PhD_Precipitation/"
    "03_Code/results/srcnn_baseline/srcnn_pred_2011-2014.nc"
)
fm_path = (
    "/path/to/project/Flow_matching_downscaling/"
    "results/precip_fm_v1/precip_fm_v1_heun50_merged_test.nc"
)
ca_path = (
    "/path/to/project/PhD_Precipitation/"
    "03_Code/results/ca_baseline/ca_pred_2011-2014.nc"
)
wt_path = (
    "/path/to/project/PhD_Precipitation/"
    "03_Code/results/weather_typing_baseline/wt_pred_2011-2014.nc"
)
output_dir = (
    "/path/to/project/PhD_Precipitation/"
    "03_Code/notebooks/results/precip_ddpm_v5b"
)

# ============================================================
# SPATIAL/TEMPORAL SUBSET
# ============================================================
time_slice = slice('2011-01-01', '2014-12-31')
lat_slice  = slice(39.95, 5.05)
lon_slice  = slice(65.05, 99.95)

# ============================================================
# MODEL CONFIGURATION
# ============================================================
MODEL_ORDER = [
    'MSWX (Observed)',
    'CA',
    'WT',
    'SRCNN',
    'DDPM',
    'Flow Matching'
]

MODELS_PATHS = {
    'Flow Matching': (fm_path,    'precipitation'),
    'DDPM':          (ddpm_path,  'precipitation'),
    'SRCNN':         (srcnn_path, 'pr_srcnn'),
    'CA':            (ca_path,    'pr_ca'),
    'WT':            (wt_path,    'pr_wt'),
}

# ============================================================
# IMD RAINFALL INTENSITY CLASSIFICATION (mm day⁻¹)
# ============================================================
BANDS = [
    (1.0,    7.5,   'Light\n(1.0–7.4)'),
    (7.5,    35.5,  'Moderate\n(7.5–35.4)'),
    (35.5,   64.5,  'Heavy\n(35.5–64.4)'),
    (64.5,   115.5, 'Very Heavy\n(64.5–115.4)'),
    (115.5,  np.inf,'Extreme\n(≥ 115.5)')
]

# ============================================================
# COMPUTATION
# ============================================================
def compute_volumetric_fractions(values_flat):
    """
    Computes percentage contribution of total
    precipitation volume for each rainfall class.
    """
    wet = values_flat[np.isfinite(values_flat)]
    wet = wet[wet >= 1.0]  # Standardized wet-day threshold (≥ 1.0 mm/day)

    total_volume = np.sum(wet)
    fractions = []

    for low, high, _ in BANDS:
        band_volume = np.sum(wet[(wet >= low) & (wet < high)])
        fraction_pct = (band_volume / total_volume) * 100.0
        fractions.append(fraction_pct)

    return fractions

# ============================================================
# LOAD DATA
# ============================================================
def load_and_compute():
    print("\nLoading observations...")
    ds_obs = xr.open_dataset(obs_path)

    lat_min, lat_max = sorted([lat_slice.start, lat_slice.stop])
    obs_lat = (
        slice(lat_max, lat_min)
        if ds_obs.lat[0] > ds_obs.lat[-1]
        else slice(lat_min, lat_max)
    )

    da_obs = ds_obs.sel(time=time_slice, lat=obs_lat, lon=lon_slice)['precipitation']
    results = {}

    print("Computing observed fractions...")
    results['MSWX (Observed)'] = compute_volumetric_fractions(da_obs.values.flatten())

    # ========================================================
    # MODELS
    # ========================================================
    for mname in MODEL_ORDER[1:]:
        print(f"Computing {mname} fractions...")
        path, varname = MODELS_PATHS[mname]

        with xr.open_dataset(path) as ds_m:
            da_m = ds_m.sel(time=time_slice)[varname]
            
            # Align grids
            da_m = da_m.sel(lat=da_obs.lat, lon=da_obs.lon, method='nearest')
            results[mname] = compute_volumetric_fractions(da_m.values.flatten())

    ds_obs.close()
    return results

# ============================================================
# HEATMAP MATRIX
# ============================================================
def plot_heatmap(results):
    os.makedirs(output_dir, exist_ok=True)

    # ========================================================
    # BUILD MATRIX
    # ========================================================
    matrix = []
    for model in MODEL_ORDER:
        matrix.append(results[model])
    matrix = np.array(matrix)

    # ========================================================
    # OBSERVED DIFFERENCE MATRIX
    # ========================================================
    observed = matrix[0]
    anomaly_matrix = matrix - observed

    # ========================================================
    # FIGURE
    # ========================================================
    # Adjusted aspect ratio for 5 distinct classes
    fig, ax = plt.subplots(figsize=(10.5, 5.5))
    fig.patch.set_facecolor('white')

    # ========================================================
    # HEATMAP
    # ========================================================
    im = ax.imshow(
        anomaly_matrix,
        cmap='coolwarm',
        aspect='auto',
        vmin=-15,
        vmax=15
    )

    # ========================================================
    # TICKS
    # ========================================================
    ax.set_xticks(np.arange(len(BANDS)))
    ax.set_yticks(np.arange(len(MODEL_ORDER)))

    ax.set_xticklabels([b[2] for b in BANDS], fontsize=9)
    ax.set_yticklabels(MODEL_ORDER, fontsize=10, fontweight='bold')

    # ========================================================
    # CELL ANNOTATIONS
    # ========================================================
    for i in range(matrix.shape[0]):
        for j in range(matrix.shape[1]):
            true_value = matrix[i, j]
            anomaly = anomaly_matrix[i, j]

            # Improved visibility: Use black text for mild anomalies, white for extremes
            if abs(anomaly) < 5.0:
                text_color = 'black'
            else:
                text_color = 'white'

            ax.text(
                j, i,
                f"{true_value:.2f}%",
                ha='center', va='center',
                fontsize=9,
                color=text_color,
                fontweight='bold'
            )

    # ========================================================
    # TITLE
    # ========================================================
    ax.set_title(
        "Hydroclimatic Volumetric Redistribution Matrix (2011–2014)",
        fontsize=14,
        fontweight='bold',
        pad=16
    )

    # ========================================================
    # COLORBAR
    # ========================================================
    cbar = fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
    cbar.set_label(
        "Deviation from Observed Volume Contribution (%)",
        fontsize=10,
        fontweight='bold'
    )

    # ========================================================
    # GRIDLINES
    # ========================================================
    ax.set_xticks(np.arange(-0.5, len(BANDS), 1), minor=True)
    ax.set_yticks(np.arange(-0.5, len(MODEL_ORDER), 1), minor=True)

    ax.grid(which='minor', color='white', linestyle='-', linewidth=1.6)
    ax.tick_params(which='minor', bottom=False, left=False)

    # ========================================================
    # LAYOUT
    # ========================================================
    plt.tight_layout()

    # ========================================================
    # SAVE
    # ========================================================
    base_filename = 'precip_volumetric_redistribution_heatmap_2011_2014'
    out_pdf = os.path.join(output_dir, f'{base_filename}.pdf')
    out_png = os.path.join(output_dir, f'{base_filename}.png')

    fig.savefig(out_pdf, dpi=400, bbox_inches='tight', facecolor='white')
    fig.savefig(out_png, dpi=400, bbox_inches='tight', facecolor='white')
    print(f"\nSaved figures:\n  {out_pdf}\n  {out_png}")

    plt.show()

# ============================================================
# MAIN
# ============================================================
if __name__ == '__main__':
    print("\n======================================")
    print("Hydroclimatic Volumetric Heatmap Matrix")
    print("======================================")

    results = load_and_compute()
    plot_heatmap(results)

    print("\nProcess complete.")

# 🗺️ Intraseasonal Monsoon Dynamics: Active vs. Break Composite Mapping

### 📌 Overview
This notebook block executes a large-scale atmospheric diagnostic pipeline to analyze **Intraseasonal Oscillations (ISOs)** within the Indian Summer Monsoon (JJAS: June, July, August, September). It evaluates how accurately generative and traditional downscaling models replicate the core spatial shifts that occur when the regional weather transitions from heavy convective storms (**Active Phases**) to prolonged dry spell intervals (**Break Phases**).

---

### 🧠 Climatological Core Thresholds

To identify these macro-climate periods without picking up localized weather noise, the script targets the **Core Monsoon Zone (CMZ)** ($18^\circ\text{N}\rightarrow28^\circ\text{N}$, $65^\circ\text{E}\rightarrow88^\circ\text{E}$)—the primary region for tracking planetary-scale tropical moisture transitions.

The diagnostic processes the timelines using the following mathematical pipeline:
1. **Normalization:** It takes the area-weighted daily time series $P(t)$ of the CMZ and extracts its long-term climatological mean ($\mu_{\text{jjas}}$) and standard deviation ($\sigma_{\text{jjas}}$).
2. **Phase Filtering:** Days are flagged based on standard deviation boundary cuts:
   *   $$\text{Active Phase Candidate: } P(t) > \mu_{\text{jjas}} + \sigma_{\text{jjas}}$$
   *   $$\text{Break Phase Candidate: } P(t) < \mu_{\text{jjas}} - \sigma_{\text{jjas}}$$
3. **Consecutive Duration Check:** To separate passing storm fronts from true intraseasonal oscillations, the code runs a rolling filter that keeps only stable systems lasting **$\ge 3$ consecutive days**.
4. **Anomaly Compositing:** It stacks the filtered maps to generate a clean, long-term spatial anomaly surface:
   $$\text{Composite Anomaly} = \frac{1}{N} \sum_{t \in \text{Phase}} I(x,y,t) - \bar{I}_{\text{climo}}(x,y)$$

---

### 🗺️ Visualization Architecture & Multi-Figure Splits

The script organizes its outputs across two separate figures using `cartopy` projections to maintain correct geographic proportions over India:

*   **Main Manuscript Canvas ($2\times2$ Grid Matrix):** Pairs the `MSWX` ground-truth observation maps directly against your primary proposed method (`Flow Matching`). This layout focuses the reader on the generative model's capability to capture large-scale spatial patterns without blurry interpolation.
*   **Supplementary Materials Canvas ($4\times2$ Grid Matrix):** Segregates the four baseline models (`CA`, `WT`, `SRCNN`, `DDPM`) to keep the primary paper clean while preserving full transparency for baseline comparisons.

---

### 🎨 Key Graphical Enhancements
* **Dynamic, Shared Colorbar Range:** Instead of fixing arbitrary limits, the script calculates symmetric bounds on the fly by scanning all datasets down to the $1^{\text{st}}$ and $99^{\text{th}}$ percentiles. This locks both figures into an identical `RdBu` colorbar scale, ensuring visual comparisons remain perfectly fair.
* **Inline Statistics Tracking:** Each model panel automatically calculates and embeds a white bounding box showing total **Spatial Bias** and **Root Mean Squared Error (RMSE)** relative to the observation composites, making performance differences immediately clear.

In [ ]:
"""
plot_intraseasonal_variability_combined.py
==========================================
Identifies Active and Break monsoon phases (JJAS) based on the Core Monsoon Zone 
(18-28N, 65-88E) rainfall and plots the spatial precipitation anomaly composites.

Generates a single combined figure in a 4x3 grid to maintain a reasonable 
aspect ratio without becoming too long horizontally:
  - Row 1 & 2: Active & Break for Group 1 (MSWX, DDPM, Flow Matching)
  - Row 3 & 4: Active & Break for Group 2 (CA, WT, SRCNN)
"""

import os
import warnings
import numpy as np
import pandas as pd
import xarray as xr
import matplotlib
import matplotlib.pyplot as plt
import cartopy.crs as ccrs
import cartopy.feature as cfeature

warnings.filterwarnings("ignore")

# Nature Communications style formatting
matplotlib.rcParams.update({
    'font.family':       'serif',
    'font.serif':        ['Times New Roman', 'Times', 'DejaVu Serif'],
    'font.size':         9,
    'axes.labelsize':    10,
    'axes.titlesize':    11,
    'xtick.labelsize':   9,
    'ytick.labelsize':   9,
    'legend.fontsize':   9,
    'figure.dpi':        150,
    'pdf.fonttype':      42,
    'ps.fonttype':       42,
})

# ==========================================
# 1. Configuration & Parameters
# ==========================================
# Core Monsoon Zone (CMZ) boundaries (Used ONLY for date extraction)
CMZ_LAT_MIN, CMZ_LAT_MAX = 18.0, 28.0
CMZ_LON_MIN, CMZ_LON_MAX = 65.0, 88.0

# Full Domain Plot Extent (Entire Area)
PLOT_EXTENT = [65.0, 100.0, 5.0, 40.0]

time_slice = slice('2011-01-01', '2014-12-31')
lat_bounds = slice(39.95, 5.05)
lon_bounds = slice(65.05, 99.95)

# Paths
obs_path   = "/path/to/data/MSWX_Pcp_Daily_Ind_HR.nc"
ddpm_path  = "/path/to/project/PhD_Precipitation/03_Code/precipitation-ddpm-india/scripts/ddpm/diffusr_climate/results/precip_ddpm_v5b/inference/ddpm_v5b_precipitation_test_set.nc"
srcnn_path = "/path/to/project/PhD_Precipitation/03_Code/results/srcnn_baseline/srcnn_pred_2011-2014.nc"
fm_path    = "/path/to/project/Flow_matching_downscaling/results/precip_fm_v1/precip_fm_v1_heun50_merged_test.nc"
ca_path    = "/path/to/project/PhD_Precipitation/03_Code/results/ca_baseline/ca_pred_2011-2014.nc"
wt_path    = "/path/to/project/PhD_Precipitation/03_Code/results/weather_typing_baseline/wt_pred_2011-2014.nc"

output_dir = "/path/to/project/PhD_Precipitation/03_Code/notebooks/results/precip_ddpm_v5b"
os.makedirs(output_dir, exist_ok=True)

# Registry
MODEL_ORDER = ['CA', 'WT', 'SRCNN', 'DDPM', 'Flow Matching']
MODELS_PATHS = {
    'CA':            (ca_path, 'pr_ca'),
    'WT':            (wt_path, 'pr_wt'),
    'SRCNN':         (srcnn_path, 'pr_srcnn'),
    'DDPM':          (ddpm_path, 'precipitation'),
    'Flow Matching': (fm_path, 'precipitation')
}

GROUP_1 = ['MSWX (Observed)', 'DDPM', 'Flow Matching']
GROUP_2 = ['CA', 'WT', 'SRCNN']

# ==========================================
# 2. Helper Functions
# ==========================================
def get_jjas_data(da):
    """Filters DataArray to only include JJAS (June-Sept)."""
    return da.sel(time=da['time'].dt.month.isin([6, 7, 8, 9]))

def identify_active_break_dates(jjas_da):
    """Identifies Active and Break days based on normalized CMZ rainfall anomalies."""
    cmz_lat_slice = slice(CMZ_LAT_MAX, CMZ_LAT_MIN) if jjas_da.lat[0] > jjas_da.lat[-1] else slice(CMZ_LAT_MIN, CMZ_LAT_MAX)
    cmz_lon_slice = slice(CMZ_LON_MIN, CMZ_LON_MAX)
    
    cmz_ts = jjas_da.sel(lat=cmz_lat_slice, lon=cmz_lon_slice).mean(dim=['lat', 'lon'])
    
    df = cmz_ts.to_dataframe(name='precip')
    mean_val = df['precip'].mean()
    std_val = df['precip'].std()
    
    active_mask = df['precip'] > (mean_val + std_val)
    break_mask = df['precip'] < (mean_val - std_val)
    
    def filter_consecutive(mask, min_days=3):
        m = mask.astype(int)
        runs = m.groupby((m != m.shift()).cumsum()).transform('size')
        return mask & (runs >= min_days)
    
    final_active = filter_consecutive(active_mask)
    final_break = filter_consecutive(break_mask)
    
    return final_active[final_active].index, final_break[final_break].index

def calculate_composites(jjas_da, active_dates, break_dates):
    """Calculates spatial anomaly composites over the ENTIRE domain."""
    climo_mean = jjas_da.mean(dim='time')
    active_data = jjas_da.sel(time=active_dates).mean(dim='time').compute()
    break_data = jjas_da.sel(time=break_dates).mean(dim='time').compute()
    
    return active_data - climo_mean, break_data - climo_mean

# ==========================================
# 3. Process Datasets
# ==========================================
print("Loading Observations (MSWX)...")
ds_obs = xr.open_dataset(obs_path)
lat_min, lat_max = sorted([lat_bounds.start, lat_bounds.stop])
obs_lat = slice(lat_max, lat_min) if ds_obs.lat[0] > ds_obs.lat[-1] else slice(lat_min, lat_max)

da_obs = ds_obs.sel(time=time_slice, lat=obs_lat, lon=lon_bounds)['precipitation']

obs_jjas = get_jjas_data(da_obs)
obs_act_dates, obs_brk_dates = identify_active_break_dates(obs_jjas)
obs_act_comp, obs_brk_comp = calculate_composites(obs_jjas, obs_act_dates, obs_brk_dates)

print(f"MSWX (2011-2014): {len(obs_act_dates)} Active days, {len(obs_brk_dates)} Break days.")

# Store MSWX alongside models for unified plotting loop
models_data = {
    'MSWX (Observed)': {
        'act': obs_act_comp, 
        'brk': obs_brk_comp, 
        'n_act': len(obs_act_dates), 
        'n_brk': len(obs_brk_dates)
    }
}
all_data_for_cmap = [obs_act_comp.values.flatten(), obs_brk_comp.values.flatten()]

print("\nProcessing Models...")
for mname in MODEL_ORDER:
    path, varname = MODELS_PATHS[mname]
    with xr.open_dataset(path) as ds_m:
        da_m = ds_m.sel(time=time_slice)[varname]
        da_m = da_m.sel(lat=da_obs.lat, lon=da_obs.lon, method='nearest').compute()
        
    m_jjas = get_jjas_data(da_m)
    m_act_dates, m_brk_dates = identify_active_break_dates(m_jjas)
    m_act_comp, m_brk_comp = calculate_composites(m_jjas, m_act_dates, m_brk_dates)
    
    print(f"  {mname}: {len(m_act_dates)} Active days, {len(m_brk_dates)} Break days.")
    
    models_data[mname] = {
        'act': m_act_comp, 
        'brk': m_brk_comp,
        'n_act': len(m_act_dates), 
        'n_brk': len(m_brk_dates)
    }
    
    all_data_for_cmap.extend([m_act_comp.values.flatten(), m_brk_comp.values.flatten()])

ds_obs.close()

# ==========================================
# 4. Dynamic Colormap Range
# ==========================================
print("\nCalculating dynamic colormap bounds across ALL datasets...")
all_arr = np.concatenate(all_data_for_cmap)
all_arr = all_arr[~np.isnan(all_arr)]

perc_1 = np.percentile(all_arr, 1)
perc_99 = np.percentile(all_arr, 99)
v_max_abs = max(abs(perc_1), abs(perc_99))

vmin, vmax = -v_max_abs, v_max_abs
levels = np.linspace(vmin, vmax, 11)
cmap = plt.get_cmap('RdBu')

print(f"Shared symmetric limits set to: {vmin:.2f} to {vmax:.2f} mm/day")

# ==========================================
# 5. Plotting Utility
# ==========================================
def plot_panel(ax, data, title, n_days=None):
    ax.set_extent(PLOT_EXTENT, crs=ccrs.PlateCarree())
    ax.add_feature(cfeature.COASTLINE, linewidth=0.6, alpha=0.8)
    
    cf = ax.contourf(data.lon, data.lat, data, levels=levels, cmap=cmap, 
                     extend='both', transform=ccrs.PlateCarree())
    
    ax.set_title(title, fontsize=12, loc='left', color='#1A1A1A', fontweight='normal')
    
    if n_days is not None:
        stat_text = f"Days: {n_days}"
        ax.text(0.97, 0.96, stat_text, transform=ax.transAxes, ha='right', va='top',
                fontsize=9, bbox=dict(boxstyle='round,pad=0.3', fc='white', ec='gray', alpha=0.85))
        
    return cf

# ==========================================
# 6. Figure Generation (4x3 Grid)
# ==========================================
print("\nGenerating Combined Plot...")
# 4 Rows (Active, Break, Active, Break) and 3 Columns
fig, axes = plt.subplots(nrows=4, ncols=3, figsize=(14, 15), 
                         subplot_kw={'projection': ccrs.PlateCarree()})
fig.patch.set_facecolor('white')

# fig.suptitle("Intraseasonal Monsoon Variability (Active vs. Break Composites)", 
#              fontsize=18, fontweight='bold', y=0.94)

# Plot Group 1 (Rows 0 and 1)
for col, mname in enumerate(GROUP_1):
    m_data = models_data[mname]
    plot_panel(axes[0, col], m_data['act'], f"{mname} - Active", n_days=m_data['n_act'])
    plot_panel(axes[1, col], m_data['brk'], f"{mname} - Break", n_days=m_data['n_brk'])

# Plot Group 2 (Rows 2 and 3)
for col, mname in enumerate(GROUP_2):
    m_data = models_data[mname]
    plot_panel(axes[2, col], m_data['act'], f"{mname} - Active", n_days=m_data['n_act'])
    cf_main = plot_panel(axes[3, col], m_data['brk'], f"{mname} - Break", n_days=m_data['n_brk'])

# Adjust spacing and add shared colorbar
plt.subplots_adjust(bottom=0.08, hspace=0.15, wspace=0.05)
cbar_ax = fig.add_axes([0.25, 0.04, 0.5, 0.015]) 
cbar = fig.colorbar(cf_main, cax=cbar_ax, orientation='horizontal', ticks=levels)
cbar.set_label('Precipitation Anomaly (mm/day)', fontsize=12, fontweight='bold')
cbar.ax.set_xticklabels([f"{tick:.1f}" for tick in levels])

# Save outputs
base_filename = 'precip_intraseasonal_variability_2011_2014'
fig.savefig(os.path.join(output_dir, f'{base_filename}.pdf'), dpi=300, bbox_inches='tight', facecolor='white')
fig.savefig(os.path.join(output_dir, f'{base_filename}.png'), dpi=300, bbox_inches='tight', facecolor='white')
plt.show()

print("\nDone.")

# 🗺️ Climate Extremes Suite: ETCCDI Spatial Mapping & Bias Assessment Dashboard

### 📌 Overview
This script executes a comprehensive climate extreme evaluation framework by calculating all **11 core Expert Team on Climate Change Detection and Indices (ETCCDI)** precipitation parameters over the Indian subcontinent. The notebook automatically segments outputs to decouple primary manuscript visualizations from supplementary tracking metrics, establishing a unified **2x3** geospatial diagnostic grid for every extreme index:

$$\text{[MSWX Absolute Baseline]} \quad \rightarrow \quad \text{[CA Bias]} \quad \rightarrow \quad \text{[WT Bias]} \quad \rightarrow \quad \text{[SRCNN Bias]} \quad \rightarrow \quad \text{[DDPM Bias]} \quad \rightarrow \quad \text{[Flow Matching Bias]}$$

---

### 📂 Structural Index Partitioning
To maintain an efficient publication profile, the diagnostic pipeline segments the index suite based on relevance to regional hydrologic extremes:

*   **Main Manuscript Track (Key Indicators):** 
    *   `CDD` (Consecutive Dry Days): Measures prolonged dry spells.
    *   `R20mm` (Very Heavy Precipitation Days): Counts high-intensity rainfall days.
    *   `PRCPTOT` (Annual Total Wet-Day Precipitation): Evaluates the overall moisture budget.
    *   `Rx1day` (Max 1-Day Precipitation Amount): Captures peak flash flood triggers.
*   **Supplementary Appendix Track:** Tracks the remaining seven localized thresholds (`CWD`, `R1mm`, `R10mm`, `Rx5day`, `SDII`, `R95pTOT`, `R99pTOT`).

---

### 🧠 Computational Vectorization & Geometric Optimizations

#### 1. Linear Streak Detection Engine
To process continuous multi-day temporal streaks (`CDD`/`CWD`) across thousands of spatial coordinate cells simultaneously without crashing system memory, the script uses a customized, stateful cumulative forward pass. For a binary wet/dry spatial array mask $M$ at time step $t$, the engine computes:

$$\text{Streak}_t = (\text{Streak}_{t-1} + M_t) \times M_t$$

The peak value at each pixel location is then captured across the entire time block using an optimized `np.maximum` reduction pass.

#### 2. Vector Clipping Optimization
Instead of parsing large country outline shapefiles globally during every single plotting loop, the script extracts your grid limits and utilizes a spatial bounding box operation (`shapely.geometry.box`) to pre-slice geopolitical borders:

$$\text{Domain Box} = \left[\text{Lon}_{\min}, \text{Lat}_{\min}, \text{Lon}_{\max}, \text{Lat}_{\max}\right]$$

This pre-clips the underlying vector layers to match your exact NetCDF grid boundaries, keeping map generation fast and scalable.

---

### 🎨 Statistical Annotation Cards
Each generated sub-panel automatically appends a localized bounding summary box to cross-verify visual patterns with rigorous numerical metrics:
*   **Absolute Map (MSWX):** Reports spatial domain core structures: Mean, Standard Deviation ($\sigma$), and absolute Maximum value.
*   **Bias Maps (Predicted $-$ Observed):** Computes precise error signatures using the **99th** percentile absolute spatial error (`99p Err`) alongside standard bias parameters. This prevents isolated boundary pixel anomalies from distorting your color scales, keeping map evaluations clear and balanced.

In [ ]:
"""
plot_etccdi_spatial_diff_all.py
===============================
Computes 11 ETCCDI indices and plots spatial bias maps for 5 models.
Saves all 11 figures as 300 DPI publication-ready PNGs into a single output folder.

Features:
  - 2x3 grid (MSWX absolute | CA bias | WT bias | SRCNN bias | DDPM bias | Flow Matching bias)
  - Custom India_Boundary.shp (data raster strictly clipped inside India)
  - Statistical summary boxes (Mean, Std, Max/99p Err calculated only over Indian land cells)
"""

import xarray as xr
import matplotlib.pyplot as plt
import numpy as np
import geopandas as gpd
import shapely
from shapely.geometry import box
import os
import warnings

# Suppress warnings for clean output
warnings.filterwarnings("ignore")

import matplotlib
matplotlib.rcParams.update({
    'font.family':       'serif',
    'font.serif':        ['Times New Roman', 'Times', 'DejaVu Serif'],
    'pdf.fonttype':      42,
    'ps.fonttype':       42,
})

# ==========================================
# 1. Define Paths & Output
# ==========================================
mswx_path  = "/path/to/data/MSWX_Pcp_Daily_Ind_HR.nc"
ddpm_path  = "/path/to/project/PhD_Precipitation/03_Code/precipitation-ddpm-india/scripts/ddpm/diffusr_climate/results/precip_ddpm_v5b/inference/ddpm_v5b_precipitation_test_set.nc"
srcnn_path = "/path/to/project/PhD_Precipitation/03_Code/results/srcnn_baseline/srcnn_pred_2011-2014.nc"
fm_path    = "/path/to/project/Flow_matching_downscaling/results/precip_fm_v1/precip_fm_v1_heun50_merged_test.nc"
ca_path    = "/path/to/project/PhD_Precipitation/03_Code/results/ca_baseline/ca_pred_2011-2014.nc"
wt_path    = "/path/to/project/PhD_Precipitation/03_Code/results/weather_typing_baseline/wt_pred_2011-2014.nc"

# India Boundary shapefile
shp_path   = "/path/to/project/raw-data/shapefiles/India_Boundary.shp"

# Single unified output directory for all figures
output_dir = "/path/to/project/PhD_Precipitation/03_Code/notebooks/results/precip_ddpm_v5b/etccdi_spatial/all_figures_300dpi"
os.makedirs(output_dir, exist_ok=True)

time_slice = slice('2011-01-01', '2014-12-31')
lat_slice  = slice(39.95, 5.05)
lon_slice  = slice(65.05, 99.95)

# ==========================================
# 2. Model Registry
# ==========================================
# Flow Matching placed last as the highlighted focal point
MODEL_ORDER = ['CA', 'WT', 'SRCNN', 'DDPM', 'Flow Matching']

MODELS_PATHS = {
    'CA':            (ca_path,    'pr_ca'),
    'WT':            (wt_path,    'pr_wt'),
    'SRCNN':         (srcnn_path, 'pr_srcnn'),
    'DDPM':          (ddpm_path,  'precipitation'),
    'Flow Matching': (fm_path,    'precipitation')
}

# ==========================================
# 3. Load Data & Create India Clipping Mask
# ==========================================
print("Loading NetCDF Data...")
ds_mswx = xr.open_dataset(mswx_path)
lat_min, lat_max = sorted([lat_slice.start, lat_slice.stop])
obs_lat = slice(lat_max, lat_min) if ds_mswx.lat[0] > ds_mswx.lat[-1] else slice(lat_min, lat_max)

da_obs = ds_mswx.sel(time=time_slice, lat=obs_lat, lon=lon_slice)['precipitation'].compute()

print("Loading India Boundary Shapefile and generating spatial mask...")
gdf = gpd.read_file(shp_path)
if gdf.crs is not None and gdf.crs.to_epsg() != 4326:
    gdf = gdf.to_crs(4326)

try:
    geom = gdf.union_all()
except AttributeError:      # For older geopandas versions
    geom = gdf.unary_union

# Build vectorized boolean mask array (True inside India, False outside)
LO, LA = np.meshgrid(da_obs.lon.values, da_obs.lat.values)
inside_mask = shapely.contains_xy(geom, LO.ravel(), LA.ravel()).reshape(LA.shape)
india_mask = xr.DataArray(inside_mask, coords=[da_obs.lat, da_obs.lon], dims=['lat', 'lon'])

models_data = {}
print("Aligning models to MSWX grid...")
for mname in MODEL_ORDER:
    path, varname = MODELS_PATHS[mname]
    with xr.open_dataset(path) as ds_m:
        da_m = ds_m.sel(time=time_slice)[varname]
        # Fast nearest-neighbor alignment
        da_m_aligned = da_m.sel(lat=da_obs.lat, lon=da_obs.lon, method='nearest').compute()
        models_data[mname] = da_m_aligned

# ==========================================
# 4. Vectorized ETCCDI Engines
# ==========================================
def compute_streak(precip_array, condition_func):
    """Calculates consecutive streaks (CDD/CWD)."""
    mask = condition_func(precip_array).astype(float)
    max_streak = np.zeros(mask.shape[1:], dtype=float)
    current_streak = np.zeros(mask.shape[1:], dtype=float)
    for t in range(mask.shape[0]):
        current_streak = (current_streak + mask[t]) * mask[t]
        max_streak = np.maximum(max_streak, current_streak)
    return max_streak

def calculate_all_indices(da, spatial_mask=None):
    """Calculates all 11 ETCCDI indices and clips strictly inside India."""
    wet_mask = da >= 1.0
    
    indices = {
        'CDD': xr.DataArray(compute_streak(da.values, lambda x: x < 1.0), coords=[da.lat, da.lon], dims=['lat', 'lon']),
        'CWD': xr.DataArray(compute_streak(da.values, lambda x: x >= 1.0), coords=[da.lat, da.lon], dims=['lat', 'lon']),
        'R1mm': (da >= 1.0).sum(dim='time').astype(float),
        'R10mm': (da >= 10.0).sum(dim='time').astype(float),
        'R20mm': (da >= 20.0).sum(dim='time').astype(float),
        'Rx1day': da.max(dim='time'),
        'Rx5day': da.rolling(time=5, min_periods=1).sum().max(dim='time'),
        'SDII': da.where(wet_mask).mean(dim='time'),
        'PRCPTOT': da.where(wet_mask).sum(dim='time'),
        'R95pTOT': da.where(da > da.where(wet_mask).quantile(0.95, dim='time')).sum(dim='time'),
        'R99pTOT': da.where(da > da.where(wet_mask).quantile(0.99, dim='time')).sum(dim='time')
    }
    
    # Clip raster values outside India to NaN
    if spatial_mask is not None:
        indices = {k: v.where(spatial_mask) for k, v in indices.items()}
        
    return indices

print("Computing full ETCCDI suite for MSWX (Observed)...")
obs_indices = calculate_all_indices(da_obs, spatial_mask=india_mask)

mod_indices = {}
for mname in MODEL_ORDER:
    print(f"Computing full ETCCDI suite for {mname}...")
    mod_indices[mname] = calculate_all_indices(models_data[mname], spatial_mask=india_mask)

# ==========================================
# 5. Stats Inset Generator
# ==========================================
def add_stats_box(ax, data, is_diff):
    """Adds a statistical summary box to the top right of the axis."""
    clean = data.values.flatten()
    clean = clean[~np.isnan(clean)]  # Automatically excludes cells outside India
    if len(clean) == 0: return
    
    mean_val, std_val = np.mean(clean), np.std(clean)
    if is_diff:
        p99_err = np.percentile(np.abs(clean), 99)
        stats_text = f"Bias: {mean_val:.1f}\nStd: {std_val:.1f}\n99p Err: {p99_err:.1f}"
    else:
        max_val = np.max(clean)
        stats_text = f"Mean: {mean_val:.1f}\nStd: {std_val:.1f}\nMax: {max_val:.1f}"
    
    ax.text(0.98, 0.98, stats_text, transform=ax.transAxes, fontsize=10, fontweight='500',
            verticalalignment='top', horizontalalignment='right', 
            bbox=dict(boxstyle="round,pad=0.4", fc="white", ec="gray", alpha=0.9), zorder=5)

# ==========================================
# 6. Master Plotting Loop
# ==========================================
# Format: { 'Index': ('Units', 'Sequential Colormap', 'Diverging Colormap for Error') }
index_metadata = {
    'CDD':     ('days',   'YlOrBr', 'RdBu_r'), # Reversed: Positive bias = too many dry days = Red
    'CWD':     ('days',   'YlGn',   'RdBu'),   # Positive bias = too wet = Blue
    'R1mm':    ('days',   'PuBu',   'RdBu'),
    'R10mm':   ('days',   'PuBu',   'RdBu'),
    'R20mm':   ('days',   'PuBu',   'RdBu'),
    'Rx1day':  ('mm',     'YlGnBu', 'RdBu'),
    'Rx5day':  ('mm',     'YlGnBu', 'RdBu'),
    'SDII':    ('mm/day', 'YlGnBu', 'RdBu'),
    'PRCPTOT': ('mm',     'Blues',  'RdBu'),
    'R95pTOT': ('mm',     'GnBu',   'RdBu'),
    'R99pTOT': ('mm',     'GnBu',   'RdBu')
}

print(f"\nSaving all 300 DPI figures to: {output_dir}\n" + "="*60)

for idx_name, (unit, cmap_name, diff_cmap) in index_metadata.items():
    print(f"Generating and saving spatial map for -> {idx_name} ...")
    
    obs_c = obs_indices[idx_name]
    
    # Calculate Differences
    diffs_c = {}
    for mname in MODEL_ORDER:
        sim_c = mod_indices[mname][idx_name]
        diffs_c[mname] = sim_c - obs_c
        
    # Dynamic Limits (Calculated strictly on India land data)
    vmax_obs = np.nanmax(obs_c.values)
    vmax_diff = max([np.nanpercentile(np.abs(diffs_c[m].values), 99) for m in MODEL_ORDER])
    if vmax_diff == 0 or np.isnan(vmax_diff): vmax_diff = 1.0 
    
    # Figure Setup: 2 rows, 3 columns
    fig, axes = plt.subplots(2, 3, figsize=(16, 11))
    fig.patch.set_facecolor('white')
    axes_flat = axes.flatten()
    
    # 1. Plot Observations (Absolute) on the first panel
    im_obs = obs_c.plot(ax=axes_flat[0], cmap=plt.get_cmap(cmap_name, 12), vmin=0, vmax=vmax_obs, add_colorbar=False)
    axes_flat[0].set_title(f"MSWX (Observed)\n{idx_name}", fontsize=14, fontweight='bold')
    add_stats_box(axes_flat[0], obs_c, False)
    
    # 2. Plot Model Biases (Differences) on the remaining panels
    im_diffs = []
    for i, mname in enumerate(MODEL_ORDER):
        ax = axes_flat[i+1]
        im_d = diffs_c[mname].plot(ax=ax, cmap=plt.get_cmap(diff_cmap, 11), vmin=-vmax_diff, vmax=vmax_diff, add_colorbar=False)
        im_diffs.append(im_d)
        
        # Highlight Flow Matching title
        title_color = '#8B0000' if mname == 'Flow Matching' else '#1A1A1A'
        ax.set_title(f"{mname} Bias\n(Predicted - Observed)", fontsize=14, fontweight='bold', color=title_color)
        add_stats_box(ax, diffs_c[mname], True)
        
    # Add Shapefile boundaries and format axes
    for ax in axes_flat:
        gdf.boundary.plot(ax=ax, color='black', lw=0.8, alpha=0.8, zorder=3)
        ax.set_axis_off()

    # Adjust layout to make room for colorbars at the bottom
    plt.subplots_adjust(bottom=0.15, hspace=0.2, wspace=0.05)

    # Colorbars
    # Colorbar 1: Observation (placed under the first column)
    cbar_ax1 = fig.add_axes([0.15, 0.08, 0.20, 0.02]) 
    cbar1 = fig.colorbar(im_obs, cax=cbar_ax1, orientation='horizontal', drawedges=True)
    cbar1.set_label(f"Absolute Value ({unit})", fontsize=12, fontweight='bold')
    cbar1.ax.tick_params(labelsize=10)
    
    # Colorbar 2: Bias (placed under the second and third columns)
    cbar_ax2 = fig.add_axes([0.45, 0.08, 0.40, 0.02])
    cbar2 = fig.colorbar(im_diffs[0], cax=cbar_ax2, orientation='horizontal', drawedges=True)
    cbar2.set_label(f"Spatial Bias ({unit})", fontsize=12, fontweight='bold')
    cbar2.ax.tick_params(labelsize=10)

    # Global Title
    fig.suptitle(f"Spatial Distribution & Model Biases of ETCCDI Index: {idx_name} (2011-2014)", 
                 fontsize=20, fontweight='black', y=0.95)
    
    # Save to unified folder at 300 DPI
    output_file = os.path.join(output_dir, f"{idx_name}_spatial_diff_analysis.png")
    plt.savefig(output_file, dpi=300, bbox_inches='tight', facecolor='white')
    plt.show() # Uncomment if running interactively in Jupyter
    plt.close()

ds_mswx.close()
print(f"\n✅ SUCCESS: All 11 ETCCDI spatial bias maps saved at 300 DPI to:\n  {output_dir}")

In [ ]:
"""
plot_return_levels_v2.py
========================
Regional GEV return-level figure for the precipitation downscaling manuscript.

Design choices:
  - A nonparametric bootstrap 90% envelope is drawn once, around the observed
    fit. Model curves that fall inside it agree with observations within
    sampling uncertainty; curves that fall outside do not. This shows the
    uncertainty of a 4-year extrapolation without cluttering the panel with
    five separate bands.
  - No in-plot text, labels, or callouts: the panels carry only data. All
    interpretation (empirical limit, extrapolation caveat, fitted shape
    parameter) belongs in the caption and text.
  - Return level on top, model/observed ratio below, shared log return-period
    axis. The region beyond the 10-year empirical limit is lightly shaded (no
    label) to flag extrapolation; the caption explains it.

FM field is the Heun-50 merged field, consistent with Sect. 4.3.
"""

import os
import warnings
import numpy as np
import xarray as xr
from scipy.stats import genextreme
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker

warnings.filterwarnings("ignore")

matplotlib.rcParams.update({
    'font.family':     'serif',
    'font.serif':      ['Times New Roman', 'Times', 'DejaVu Serif'],
    'font.size':       9,
    'axes.labelsize':  10,
    'xtick.labelsize': 9,
    'ytick.labelsize': 9,
    'legend.fontsize': 8.5,
    'figure.dpi':      150,
    'axes.linewidth':  0.8,
    'pdf.fonttype':    42,
    'ps.fonttype':     42,
})

# ============================================================ paths
obs_path   = "/path/to/data/MSWX_Pcp_Daily_Ind_HR.nc"
ddpm_path  = ("/path/to/project/PhD_Precipitation/"
              "03_Code/precipitation-ddpm-india/scripts/ddpm/diffusr_climate/"
              "results/precip_ddpm_v5b/inference/ddpm_v5b_precipitation_test_set.nc")
srcnn_path = ("/path/to/project/PhD_Precipitation/"
              "03_Code/results/srcnn_baseline/srcnn_pred_2011-2014.nc")
fm_path    = ("/path/to/project/Flow_matching_downscaling/"
              "results/precip_fm_v1/precip_fm_v1_heun50_merged_test.nc")
ca_path    = ("/path/to/project/PhD_Precipitation/"
              "03_Code/results/ca_baseline/ca_pred_2011-2014.nc")
wt_path    = ("/path/to/project/PhD_Precipitation/"
              "03_Code/results/weather_typing_baseline/wt_pred_2011-2014.nc")
output_dir = ("/path/to/project/PhD_Precipitation/"
              "03_Code/notebooks/results/precip_ddpm_v5b")

time_slice = slice('2011-01-01', '2014-12-31')
lat_slice  = slice(39.95, 5.05)
lon_slice  = slice(65.05, 99.95)

# ============================================================ config
MODEL_ORDER = ['CA', 'WT', 'SRCNN', 'DDPM', 'Flow Matching']
OBS_C = '#1A1A1A'
PALETTE = {'CA': '#A0C4C8', 'WT': '#5B9AA0', 'SRCNN': '#C68B59',
           'DDPM': '#8E44AD', 'Flow Matching': '#C0292B'}
LINESTYLE = {'CA': (0, (5, 2)), 'WT': (0, (2, 2)), 'SRCNN': (0, (5, 2, 1, 2)),
             'DDPM': (0, (4, 1.5)), 'Flow Matching': 'solid'}
LINEWIDTH = {'CA': 1.3, 'WT': 1.3, 'SRCNN': 1.3, 'DDPM': 1.6, 'Flow Matching': 1.9}
ZORDER    = {'CA': 5, 'WT': 5, 'SRCNN': 6, 'DDPM': 8, 'Flow Matching': 9}
MODELS_PATHS = {
    'Flow Matching': (fm_path, 'precipitation'), 'DDPM': (ddpm_path, 'precipitation'),
    'SRCNN': (srcnn_path, 'pr_srcnn'), 'CA': (ca_path, 'pr_ca'), 'WT': (wt_path, 'pr_wt'),
}

RETURN_PERIODS  = np.logspace(np.log10(1.01), np.log10(100), 200)
EMPIRICAL_LIMIT = 10.0     # years; beyond this is extrapolation for a 4-year record
N_BOOT, CI      = 400, 90

# ============================================================ GEV
def pooled_annual_maxima(da):
    am = da.resample(time='YE').max(dim='time').values.flatten()
    return am[np.isfinite(am) & (am > 0)]

def gev_levels(maxima, periods):
    c, loc, scale = genextreme.fit(maxima)
    return genextreme.ppf(1.0 - 1.0 / periods, c, loc=loc, scale=scale)

def bootstrap_envelope(maxima, periods, n_boot=N_BOOT, ci=CI, seed=0):
    rng = np.random.default_rng(seed)
    n = maxima.size
    boot = np.full((n_boot, periods.size), np.nan)
    for b in range(n_boot):
        s = maxima[rng.integers(0, n, n)]
        try:
            c, loc, scale = genextreme.fit(s)
            boot[b] = genextreme.ppf(1.0 - 1.0 / periods, c, loc=loc, scale=scale)
        except Exception:
            pass
    lo = np.nanpercentile(boot, (100 - ci) / 2, axis=0)
    hi = np.nanpercentile(boot, 100 - (100 - ci) / 2, axis=0)
    return lo, hi

def process_all_datasets():
    ds_obs = xr.open_dataset(obs_path)
    lo_, hi_ = sorted([lat_slice.start, lat_slice.stop])
    obs_lat = slice(hi_, lo_) if ds_obs.lat[0] > ds_obs.lat[-1] else slice(lo_, hi_)
    da_obs = ds_obs.sel(time=time_slice, lat=obs_lat, lon=lon_slice)['precipitation']
    obs_am = pooled_annual_maxima(da_obs)
    obs_levels = gev_levels(obs_am, RETURN_PERIODS)
    obs_ci = bootstrap_envelope(obs_am, RETURN_PERIODS)
    models = {}
    for m in MODEL_ORDER:
        path, var = MODELS_PATHS[m]
        with xr.open_dataset(path) as ds_m:
            da_m = ds_m.sel(time=time_slice)[var]
            try:
                xr.testing.assert_allclose(da_obs.lat, da_m.lat)
                xr.testing.assert_allclose(da_obs.lon, da_m.lon)
            except AssertionError:
                da_m = da_m.sel(lat=da_obs.lat, lon=da_obs.lon, method='nearest')
            models[m] = gev_levels(pooled_annual_maxima(da_m), RETURN_PERIODS)
    ds_obs.close()
    return obs_levels, obs_ci, models

# ============================================================ figure
def plot_return_levels(obs_levels, obs_ci, models, save=True):
    os.makedirs(output_dir, exist_ok=True)
    obs_lo, obs_hi = obs_ci
    x = RETURN_PERIODS
    fig, (ax1, ax2) = plt.subplots(
        2, 1, figsize=(7.0, 6.6), sharex=True,
        gridspec_kw={'height_ratios': [2.5, 1], 'hspace': 0.08})
    fig.patch.set_facecolor('white')

    def extrap(ax):
        ax.axvspan(EMPIRICAL_LIMIT, 100, color='#000000', alpha=0.035, lw=0, zorder=0)

    # -- (a) return levels --
    extrap(ax1)
    ax1.fill_between(x, obs_lo, obs_hi, color=OBS_C, alpha=0.14, lw=0, zorder=1)
    ax1.semilogx(x, obs_levels, color=OBS_C, lw=2.4, zorder=7, label='MSWX (observed)')
    for m in MODEL_ORDER:
        ax1.semilogx(x, models[m], color=PALETTE[m], ls=LINESTYLE[m],
                     lw=LINEWIDTH[m], zorder=ZORDER[m], label=m)
    ax1.set_ylim(0, 350)
    ax1.set_ylabel(r'Return level (mm day$^{-1}$)')
    ax1.grid(True, which='major', ls='-', lw=0.5, alpha=0.35, color='#CCCCCC')
    ax1.grid(True, which='minor', ls=':', lw=0.4, alpha=0.3, color='#E8E8E8')
    for s in ('top', 'right'): ax1.spines[s].set_visible(False)
    ax1.legend(loc='lower right', frameon=True, framealpha=0.92,
               edgecolor='#CCCCCC', fancybox=False, handlelength=2.6)
    ax1.text(-0.10, 1.02, 'a', transform=ax1.transAxes, fontweight='bold', fontsize=12)

    # -- (b) ratio to observed --
    extrap(ax2)
    ax2.fill_between(x, obs_lo / obs_levels, obs_hi / obs_levels,
                     color=OBS_C, alpha=0.14, lw=0, zorder=1)
    ax2.axhline(1.0, color=OBS_C, lw=2.0, zorder=7)
    for m in MODEL_ORDER:
        ax2.semilogx(x, models[m] / obs_levels, color=PALETTE[m], ls=LINESTYLE[m],
                     lw=LINEWIDTH[m], zorder=ZORDER[m])
    ax2.set_xlim(1.01, 100)
    ax2.set_ylim(0.0, 1.5)
    ax2.set_xlabel('Return period (years)')
    ax2.set_ylabel('Model / observed')
    ax2.set_xticks([2, 5, 10, 20, 50, 100])
    ax2.xaxis.set_major_formatter(ticker.FuncFormatter(lambda v, p: f"{int(v)}"))
    ax2.grid(True, which='major', ls='-', lw=0.5, alpha=0.35, color='#CCCCCC')
    ax2.grid(True, which='minor', ls=':', lw=0.4, alpha=0.3, color='#E8E8E8')
    for s in ('top', 'right'): ax2.spines[s].set_visible(False)
    ax2.text(-0.10, 1.05, 'b', transform=ax2.transAxes, fontweight='bold', fontsize=12)

    fig.tight_layout()
    if save:
        for ext in ('pdf', 'png'):
            fig.savefig(os.path.join(output_dir, f'fig_return_levels_gev.{ext}'),
                        dpi=300, bbox_inches='tight', facecolor='white')
    return fig

# ============================================================ main
if __name__ == '__main__':
    obs_levels, obs_ci, models = process_all_datasets()
    plot_return_levels(obs_levels, obs_ci, models, save=True)
    print("Done.")

In [ ]:
"""
plot_return_levels_v2.py
========================
Regional GEV return-level figure for the precipitation downscaling manuscript.

Design choices:
  - A nonparametric bootstrap 90% envelope is drawn once, around the observed
    fit. Model curves that fall inside it agree with observations within
    sampling uncertainty; curves that fall outside do not. This shows the
    uncertainty of a 4-year extrapolation without cluttering the panel with
    five separate bands.
  - No in-plot text, labels, or callouts: the panels carry only data. All
    interpretation (empirical limit, extrapolation caveat, fitted shape
    parameter) belongs in the caption and text.
  - Return level on top, model/observed ratio below, shared log return-period
    axis. The region beyond the 10-year empirical limit is lightly shaded (no
    label) to flag extrapolation; the caption explains it.

FM field is the Heun-50 merged field, consistent with Sect. 4.3.
"""

import os
import warnings
import numpy as np
import xarray as xr
from scipy.stats import genextreme
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker

warnings.filterwarnings("ignore")

matplotlib.rcParams.update({
    'font.family':     'serif',
    'font.serif':      ['Times New Roman', 'Times', 'DejaVu Serif'],
    'font.size':       9,
    'axes.labelsize':  10,
    'xtick.labelsize': 9,
    'ytick.labelsize': 9,
    'legend.fontsize': 8.5,
    'figure.dpi':      150,
    'axes.linewidth':  0.8,
    'pdf.fonttype':    42,
    'ps.fonttype':     42,
})

# ============================================================ paths
obs_path   = "/path/to/data/MSWX_Pcp_Daily_Ind_HR.nc"
ddpm_path  = ("/path/to/project/PhD_Precipitation/"
              "03_Code/precipitation-ddpm-india/scripts/ddpm/diffusr_climate/"
              "results/precip_ddpm_v5b/inference/ddpm_v5b_precipitation_test_set.nc")
srcnn_path = ("/path/to/project/PhD_Precipitation/"
              "03_Code/results/srcnn_baseline/srcnn_pred_2011-2014.nc")
fm_path    = ("/path/to/project/Flow_matching_downscaling/"
              "results/precip_fm_v1/precip_fm_v1_heun50_merged_test.nc")
ca_path    = ("/path/to/project/PhD_Precipitation/"
              "03_Code/results/ca_baseline/ca_pred_2011-2014.nc")
wt_path    = ("/path/to/project/PhD_Precipitation/"
              "03_Code/results/weather_typing_baseline/wt_pred_2011-2014.nc")
output_dir = ("/path/to/project/PhD_Precipitation/"
              "03_Code/notebooks/results/precip_ddpm_v5b")

time_slice = slice('2011-01-01', '2014-12-31')
lat_slice  = slice(39.95, 5.05)
lon_slice  = slice(65.05, 99.95)

# ============================================================ config
MODEL_ORDER = ['CA', 'WT', 'SRCNN', 'DDPM', 'Flow Matching']
OBS_C = '#1A1A1A'
PALETTE = {'CA': '#A0C4C8', 'WT': '#5B9AA0', 'SRCNN': '#C68B59',
           'DDPM': '#8E44AD', 'Flow Matching': '#C0292B'}
LINESTYLE = {'CA': (0, (5, 2)), 'WT': (0, (2, 2)), 'SRCNN': (0, (5, 2, 1, 2)),
             'DDPM': (0, (4, 1.5)), 'Flow Matching': 'solid'}
LINEWIDTH = {'CA': 1.3, 'WT': 1.3, 'SRCNN': 1.3, 'DDPM': 1.6, 'Flow Matching': 1.9}
ZORDER    = {'CA': 5, 'WT': 5, 'SRCNN': 6, 'DDPM': 8, 'Flow Matching': 9}
MODELS_PATHS = {
    'Flow Matching': (fm_path, 'precipitation'), 'DDPM': (ddpm_path, 'precipitation'),
    'SRCNN': (srcnn_path, 'pr_srcnn'), 'CA': (ca_path, 'pr_ca'), 'WT': (wt_path, 'pr_wt'),
}

RETURN_PERIODS  = np.logspace(np.log10(1.01), np.log10(100), 200)
EMPIRICAL_LIMIT = 10.0     # years; beyond this is extrapolation for a 4-year record
N_BOOT, CI      = 400, 90

# ============================================================ GEV
def pooled_annual_maxima(da):
    am = da.resample(time='YE').max(dim='time').values.flatten()
    return am[np.isfinite(am) & (am > 0)]

def gev_levels(maxima, periods):
    c, loc, scale = genextreme.fit(maxima)
    return genextreme.ppf(1.0 - 1.0 / periods, c, loc=loc, scale=scale)

def bootstrap_envelope(maxima, periods, n_boot=N_BOOT, ci=CI, seed=0):
    rng = np.random.default_rng(seed)
    n = maxima.size
    boot = np.full((n_boot, periods.size), np.nan)
    for b in range(n_boot):
        s = maxima[rng.integers(0, n, n)]
        try:
            c, loc, scale = genextreme.fit(s)
            boot[b] = genextreme.ppf(1.0 - 1.0 / periods, c, loc=loc, scale=scale)
        except Exception:
            pass
    lo = np.nanpercentile(boot, (100 - ci) / 2, axis=0)
    hi = np.nanpercentile(boot, 100 - (100 - ci) / 2, axis=0)
    return lo, hi

def process_all_datasets():
    ds_obs = xr.open_dataset(obs_path)
    lo_, hi_ = sorted([lat_slice.start, lat_slice.stop])
    obs_lat = slice(hi_, lo_) if ds_obs.lat[0] > ds_obs.lat[-1] else slice(lo_, hi_)
    da_obs = ds_obs.sel(time=time_slice, lat=obs_lat, lon=lon_slice)['precipitation']
    obs_am = pooled_annual_maxima(da_obs)
    obs_levels = gev_levels(obs_am, RETURN_PERIODS)
    obs_ci = bootstrap_envelope(obs_am, RETURN_PERIODS)
    models = {}
    for m in MODEL_ORDER:
        path, var = MODELS_PATHS[m]
        with xr.open_dataset(path) as ds_m:
            da_m = ds_m.sel(time=time_slice)[var]
            try:
                xr.testing.assert_allclose(da_obs.lat, da_m.lat)
                xr.testing.assert_allclose(da_obs.lon, da_m.lon)
            except AssertionError:
                da_m = da_m.sel(lat=da_obs.lat, lon=da_obs.lon, method='nearest')
            models[m] = gev_levels(pooled_annual_maxima(da_m), RETURN_PERIODS)
    ds_obs.close()
    return obs_levels, obs_ci, models

# ============================================================ figure
def plot_return_levels(obs_levels, obs_ci, models, save=True):
    os.makedirs(output_dir, exist_ok=True)
    obs_lo, obs_hi = obs_ci
    x = RETURN_PERIODS
    fig, (ax1, ax2) = plt.subplots(
        2, 1, figsize=(7.0, 6.6), sharex=True,
        gridspec_kw={'height_ratios': [2.5, 1], 'hspace': 0.08})
    fig.patch.set_facecolor('white')

    def extrap(ax):
        ax.axvspan(EMPIRICAL_LIMIT, 100, color='#000000', alpha=0.035, lw=0, zorder=0)

    # -- (a) return levels --
    extrap(ax1)
    ax1.fill_between(x, obs_lo, obs_hi, color=OBS_C, alpha=0.14, lw=0, zorder=1)
    ax1.semilogx(x, obs_levels, color=OBS_C, lw=2.4, zorder=7, label='MSWX (observed)')
    for m in MODEL_ORDER:
        ax1.semilogx(x, models[m], color=PALETTE[m], ls=LINESTYLE[m],
                     lw=LINEWIDTH[m], zorder=ZORDER[m], label=m)
    ax1.set_ylim(0, 350)
    ax1.set_ylabel(r'Return level (mm day$^{-1}$)')
    ax1.grid(True, which='major', ls='-', lw=0.5, alpha=0.35, color='#CCCCCC')
    ax1.grid(True, which='minor', ls=':', lw=0.4, alpha=0.3, color='#E8E8E8')
    for s in ('top', 'right'): ax1.spines[s].set_visible(False)
    ax1.legend(loc='lower right', frameon=True, framealpha=0.92,
               edgecolor='#CCCCCC', fancybox=False, handlelength=2.6)
    ax1.text(-0.10, 1.02, 'a', transform=ax1.transAxes, fontweight='bold', fontsize=12)

    # -- (b) ratio to observed --
    extrap(ax2)
    ax2.fill_between(x, obs_lo / obs_levels, obs_hi / obs_levels,
                     color=OBS_C, alpha=0.14, lw=0, zorder=1)
    ax2.axhline(1.0, color=OBS_C, lw=2.0, zorder=7)
    for m in MODEL_ORDER:
        ax2.semilogx(x, models[m] / obs_levels, color=PALETTE[m], ls=LINESTYLE[m],
                     lw=LINEWIDTH[m], zorder=ZORDER[m])
    ax2.set_xlim(1.01, 100)
    ax2.set_ylim(0.0, 1.5)
    ax2.set_xlabel('Return period (years)')
    ax2.set_ylabel('Model / observed')
    ax2.set_xticks([2, 5, 10, 20, 50, 100])
    ax2.xaxis.set_major_formatter(ticker.FuncFormatter(lambda v, p: f"{int(v)}"))
    ax2.grid(True, which='major', ls='-', lw=0.5, alpha=0.35, color='#CCCCCC')
    ax2.grid(True, which='minor', ls=':', lw=0.4, alpha=0.3, color='#E8E8E8')
    for s in ('top', 'right'): ax2.spines[s].set_visible(False)
    ax2.text(-0.10, 1.05, 'b', transform=ax2.transAxes, fontweight='bold', fontsize=12)

    fig.tight_layout()
    if save:
        for ext in ('pdf', 'png'):
            fig.savefig(os.path.join(output_dir, f'fig_return_levels_gev.{ext}'),
                        dpi=300, bbox_inches='tight', facecolor='white')
    return fig

# ============================================================ main
if __name__ == '__main__':
    obs_levels, obs_ci, models = process_all_datasets()
    plot_return_levels(obs_levels, obs_ci, models, save=True)
    print("Done.")

In [ ]:
"""
plot_spell_lengths.py
=====================
Nature Communications-style distribution of Dry and Wet Spell Lengths.

Evaluates the temporal autocorrelation (chronological persistence) of the models.
Exposes whether a model suffers from continuous "drizzle" (which physically 
collapses the length of dry spells) or correctly captures the long-tail 
durations of monsoon active/break sequences.

UPGRADE: Switched from Probability Density Function (PDF) to Exceedance 
Probability (1-CDF). This naturally smooths discrete integer counts, 
eliminating tail noise and producing perfectly clean, readable ratio comparisons.
Flow Matching highlighted as the primary model using a layered line-thickness strategy.
"""

import os
import warnings
import numpy as np
import xarray as xr
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker

warnings.filterwarnings("ignore")

matplotlib.rcParams.update({
    'font.family':       'serif',
    'font.serif':        ['Times New Roman', 'Times', 'DejaVu Serif'],
    'font.size':         9,
    'axes.labelsize':    10,
    'axes.titlesize':    11,
    'xtick.labelsize':   9,
    'ytick.labelsize':   9,
    'legend.fontsize':   8.5,
    'figure.dpi':        150,
    'axes.linewidth':    0.8,
    'pdf.fonttype':      42,
    'ps.fonttype':       42,
})

# ============================================================
# PATHS
# ============================================================
obs_path     = "/path/to/data/MSWX_Pcp_Daily_Ind_HR.nc"
ddpm_path    = ("/path/to/project/PhD_Precipitation/"
                "03_Code/precipitation-ddpm-india/scripts/ddpm/diffusr_climate/"
                "results/precip_ddpm_v5b/inference/ddpm_v5b_precipitation_test_set.nc")
srcnn_path   = ("/path/to/project/PhD_Precipitation/"
                "03_Code/results/srcnn_baseline/srcnn_pred_2011-2014.nc")
fm_path      = ("/path/to/project/Flow_matching_downscaling/"
                "results/precip_fm_v1/precip_fm_v1_heun50_merged_test.nc")
ca_path      = ("/path/to/project/PhD_Precipitation/"
                "03_Code/results/ca_baseline/ca_pred_2011-2014.nc")
wt_path      = ("/path/to/project/PhD_Precipitation/"
                "03_Code/results/weather_typing_baseline/wt_pred_2011-2014.nc")
output_dir   = ("/path/to/project/PhD_Precipitation/"
                "03_Code/notebooks/results/precip_ddpm_v5b")

time_slice = slice('2011-01-01', '2014-12-31')
lat_slice  = slice(39.95, 5.05)
lon_slice  = slice(65.05, 99.95)
WET_THR    = 1.0   # mm/day threshold

# ============================================================
# CONFIGURATION
# ============================================================
MODEL_ORDER = ['CA', 'WT', 'SRCNN', 'DDPM', 'Flow Matching']

PALETTE = {
    'CA':            '#A0C4C8',
    'WT':            '#78B7BC',
    'SRCNN':         '#C68B59',
    'DDPM':          '#8E44AD',  # Muted purple
    'Flow Matching': '#C0292B',  # Highlight red
}

# Layered linestyle strategy to reveal underlying lines
LINESTYLE = {
    'CA':            (0, (4, 2)),       
    'WT':            (0, (2, 2)),       
    'SRCNN':         (0, (4, 2, 1, 2)), 
    'DDPM':          (0, (3, 2)),       # Dashed for DDPM
    'Flow Matching': 'solid',           # Solid core for Flow Matching
}

# Layered linewidth strategy
LINEWIDTH = {
    'CA':            1.2,
    'WT':            1.2,
    'SRCNN':         1.2,
    'DDPM':          2.2,
    'Flow Matching': 1.4,
}

MODELS_PATHS = {
    'Flow Matching': (fm_path,    'precipitation'),
    'DDPM':          (ddpm_path,  'precipitation'),
    'SRCNN':         (srcnn_path, 'pr_srcnn'),
    'CA':            (ca_path,    'pr_ca'),
    'WT':            (wt_path,    'pr_wt'),
}

# ============================================================
# VECTORIZED SPELL LENGTH COMPUTATION
# ============================================================
def extract_spell_lengths(data_3d, condition='dry', threshold=1.0):
    """
    Ultra-fast vectorized computation of all continuous spell lengths.
    Treats each spatial pixel's time-series as an independent sequence.
    """
    n_time, n_lat, n_lon = data_3d.shape
    data_2d = data_3d.reshape(n_time, -1)
    
    if condition == 'dry':
        mask = (data_2d < threshold) & ~np.isnan(data_2d)
    else: # wet
        mask = (data_2d >= threshold) & ~np.isnan(data_2d)
        
    mask_t = mask.T
    n_pixels = mask_t.shape[0]
    
    padded = np.hstack([np.zeros((n_pixels, 1), dtype=bool),
                        mask_t,
                        np.zeros((n_pixels, 1), dtype=bool)])
                        
    diff = padded[:, 1:].astype(int) - padded[:, :-1].astype(int)
    
    starts = np.where(diff == 1)[1]
    ends = np.where(diff == -1)[1]
    
    lengths = ends - starts
    return lengths

def compute_exceedance(lengths, max_len=100):
    """
    Converts a flat array of spell lengths into an Exceedance Probability distribution P(L >= x).
    This is monotonically decreasing and perfectly smooths out discrete integer noise, 
    making tail comparisons (and ratios) crystal clear.
    """
    if len(lengths) == 0:
        return np.arange(1, max_len + 1), np.zeros(max_len)
        
    bins = np.arange(1, max_len + 2)
    counts, _ = np.histogram(lengths, bins=bins)
    
    # Reverse cumulative sum to get P(X >= x)
    exceedance = np.cumsum(counts[::-1])[::-1] / counts.sum()
    centers = bins[:-1]
    
    return centers, exceedance

# ============================================================
# DATA PROCESSING
# ============================================================
def load_and_process():
    print("Loading Observations (MSWX)...")
    ds_obs = xr.open_dataset(obs_path)
    lat_min, lat_max = sorted([lat_slice.start, lat_slice.stop])
    obs_lat = slice(lat_max, lat_min) if ds_obs.lat[0] > ds_obs.lat[-1] else slice(lat_min, lat_max)
                
    da_obs = ds_obs.sel(time=time_slice, lat=obs_lat, lon=lon_slice)['precipitation']
    obs_vals = da_obs.values
    
    print("Extracting MSWX spell lengths...")
    obs_dry_len = extract_spell_lengths(obs_vals, 'dry', WET_THR)
    obs_wet_len = extract_spell_lengths(obs_vals, 'wet', WET_THR)
    
    results = {
        'MSWX (Observed)': {
            'dry': compute_exceedance(obs_dry_len, 100),
            'wet': compute_exceedance(obs_wet_len, 40)
        }
    }

    print("\nProcessing Models...")
    for mname in MODEL_ORDER:
        path, varname = MODELS_PATHS[mname]
        print(f"  Computing spells for {mname}...")
        
        with xr.open_dataset(path) as ds_m:
            da_m = ds_m.sel(time=time_slice)[varname]
            try:
                xr.testing.assert_allclose(da_obs.lat, da_m.lat)
                xr.testing.assert_allclose(da_obs.lon, da_m.lon)
                da_m_aligned = da_m
            except AssertionError:
                da_m_aligned = da_m.sel(lat=da_obs.lat, lon=da_obs.lon, method='nearest')
                
            mod_vals = da_m_aligned.values
            
            dry_len = extract_spell_lengths(mod_vals, 'dry', WET_THR)
            wet_len = extract_spell_lengths(mod_vals, 'wet', WET_THR)
            
            results[mname] = {
                'dry': compute_exceedance(dry_len, 100),
                'wet': compute_exceedance(wet_len, 40)
            }

    ds_obs.close()
    return results

# ============================================================
# PLOTTING
# ============================================================
def plot_spell_distributions(results):
    os.makedirs(output_dir, exist_ok=True)

    # 2x2 Grid: Exceedance on top, Ratio on bottom
    fig, axes = plt.subplots(2, 2, figsize=(11, 7.5), gridspec_kw={'height_ratios': [2.5, 1]}, sharex='col')
    fig.patch.set_facecolor('white')
    
    ax1_top, ax2_top = axes[0]
    ax1_bot, ax2_bot = axes[1]

    def plot_curves(ax_top, ax_bot, spell_type, max_x):
        # ── TOP PANEL (Absolute Exceedance) ──
        x_obs, y_obs = results['MSWX (Observed)'][spell_type]
        valid_obs = y_obs > 0
        ax_top.semilogy(x_obs[valid_obs], y_obs[valid_obs], color='#1A1A1A', lw=3.2, 
                        label='MSWX (Observed)', zorder=4)
        
        # ── BOTTOM PANEL (Ratio Baseline) ──
        ax_bot.axhline(1.0, color='#1A1A1A', linestyle='-', linewidth=3.2, zorder=4)
        
        # Models
        for mname in MODEL_ORDER:
            x_m, y_m = results[mname][spell_type]
            is_fm = (mname == 'Flow Matching')
            is_ddpm = (mname == 'DDPM')
            valid_m = y_m > 0
            
            lw = LINEWIDTH[mname]
            
            # Z-order stacking: Obs(4) < CA/WT/SRCNN(6) < DDPM(9) < FM(10)
            if is_fm:
                z_ord = 10
                alpha = 1.0
            elif is_ddpm:
                z_ord = 9
                alpha = 0.95
            else:
                z_ord = 6
                alpha = 0.85
            
            # Plot Absolute
            ax_top.semilogy(x_m[valid_m], y_m[valid_m], 
                            color=PALETTE[mname], 
                            linestyle=LINESTYLE[mname],
                            lw=lw,
                            alpha=alpha,
                            zorder=z_ord,
                            label=mname)

            # Calculate and Plot Ratio
            with np.errstate(divide='ignore', invalid='ignore'):
                ratio = np.where(y_obs > 0, y_m / y_obs, np.nan)
            
            valid_ratio = ~np.isnan(ratio) & (ratio > 0)
            
            ax_bot.plot(x_m[valid_ratio], ratio[valid_ratio], 
                        color=PALETTE[mname], 
                        linestyle=LINESTYLE[mname],
                        lw=lw,
                        alpha=alpha,
                        zorder=z_ord)

        # Formatting Top
        ax_top.set_xlim(1, max_x)
        ax_top.set_ylim(1e-5, 1.0)
        ax_top.grid(True, which='major', linestyle='-', alpha=0.3, color='#CCCCCC')
        ax_top.grid(True, which='minor', linestyle=':', alpha=0.3, color='#E8E8E8')
        ax_top.spines['top'].set_visible(False)
        ax_top.spines['right'].set_visible(False)
        
        # Formatting Bottom
        ax_bot.set_xlim(1, max_x)
        ax_bot.set_ylim(0, 3.0)  # Capped at 3x deviation for readability
        ax_bot.xaxis.set_major_locator(ticker.MultipleLocator(10 if spell_type == 'dry' else 5))
        ax_bot.xaxis.set_minor_locator(ticker.MultipleLocator(5 if spell_type == 'dry' else 1))
        ax_bot.grid(True, which='major', linestyle='-', alpha=0.3, color='#CCCCCC')
        ax_bot.grid(True, which='minor', linestyle=':', alpha=0.3, color='#E8E8E8')
        ax_bot.spines['top'].set_visible(False)
        ax_bot.spines['right'].set_visible(False)

    # ── PANEL A: Dry Spells ─────────────────────────────
    plot_curves(ax1_top, ax1_bot, 'dry', max_x=60)
    ax1_top.set_title('(a) Dry Spell Lengths (< 1 mm/day)', fontsize=12, fontweight='bold')
    ax1_top.set_ylabel('Exceedance Probability $P(X \geq x)$', fontsize=10)
    ax1_bot.set_ylabel('Ratio (Model/Obs)', fontsize=10)
    ax1_bot.set_xlabel('Consecutive Dry Days', fontsize=10)
    
    # Add Insight Box to Dry Spells (Top Panel)
    # ax1_top.text(0.95, 0.85, 
    #         "Exposes the 'drizzle problem': classical\n"
    #         "models collapse the temporal distribution,\n"
    #         "failing to produce long monsoon breaks.",
    #         transform=ax1_top.transAxes, ha='right', va='top',
    #         fontsize=8, color='#666666', style='italic',
    #         bbox=dict(boxstyle='round,pad=0.4', fc='#F9F9F9', ec='none'))

    # ── PANEL B: Wet Spells ─────────────────────────────
    plot_curves(ax2_top, ax2_bot, 'wet', max_x=30)
    ax2_top.set_title('(b) Wet Spell Lengths (≥ 1 mm/day)', fontsize=12, fontweight='bold')
    ax2_bot.set_xlabel('Consecutive Wet Days', fontsize=10)
    
    # Legend in Panel B (Top)
    ax2_top.legend(loc='upper right', fontsize=9, frameon=True, framealpha=0.9, 
                   edgecolor='#CCCCCC', fancybox=False, handlelength=2.5)

    # Global Formatting
    # fig.suptitle('Exceedance Probability of Temporal Persistence (Spell Lengths)', 
    #              fontsize=14, fontweight='black', y=0.95)
    
    plt.tight_layout()

    # ── Save ─────────────────────────────────────────────
    # out_pdf = os.path.join(output_dir, 'fig_spell_lengths.pdf')
    # out_png = os.path.join(output_dir, 'fig_spell_lengths.png')
    # fig.savefig(out_pdf, dpi=300, bbox_inches='tight', facecolor='white')
    # fig.savefig(out_png, dpi=300, bbox_inches='tight', facecolor='white')
    # print(f"\nSaved plots to:\n  {out_pdf}\n  {out_png}")

    plt.show()
    plt.close(fig)

# ============================================================
# MAIN
# ============================================================
if __name__ == '__main__':
    results = load_and_process()
    plot_spell_distributions(results)
    print("Process complete.")

In [ ]:
"""
plot_spell_lengths_binned_fixed.py
Nature Communications Earth & Environment style
Dry and Wet Spell Length Distribution - Grouped Bar Chart
Wong (2011) colorblind-safe palette; OBS distinguished via hatch + outline.
Includes relative difference annotations on top of model bars.

Fixed vs. plot_spell_lengths_binned.py: masks out "dead" pixels (grid
cells whose entire 1461-day series is exactly 0.0) before spell
extraction, for every model, not just SRCNN. This mirrors the fix
already applied to Table 6 via mask_dead_pixels_and_recompute.py, so
the figure and the table are now built from the same corrected data.
Confirmed earlier: only SRCNN has any dead pixels (n=1046, all located
exactly at the domain edge); this masking is a no-op for MSWX/CA/WT/
DDPM/FM, so nothing else about the figure should change.

Everything else (bin edges, labels, bar layout, annotation logic,
styling) is unchanged from the original script.
"""

import os, warnings
import numpy as np
import xarray as xr
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
from matplotlib.patches import Patch

warnings.filterwarnings("ignore")

matplotlib.rcParams.update({
    'font.family':        'sans-serif',
    'font.sans-serif':    ['Arial', 'Helvetica Neue', 'DejaVu Sans'],
    'font.size':           8,
    'axes.labelsize':      9,
    'axes.titlesize':      9.5,
    'xtick.labelsize':     8,
    'ytick.labelsize':     8,
    'legend.fontsize':     8,
    'figure.dpi':          150,
    'axes.linewidth':      0.65,
    'xtick.major.size':    3.0,
    'ytick.major.size':    3.0,
    'xtick.major.width':   0.65,
    'ytick.major.width':   0.65,
    'xtick.minor.size':    1.8,
    'ytick.minor.size':    1.8,
    'xtick.minor.width':   0.5,
    'ytick.minor.width':   0.5,
    'pdf.fonttype':        42,
    'ps.fonttype':         42,
})

# ── Paths ──────────────────────────────────────────────────────────────────────
obs_path   = "/path/to/data/MSWX_Pcp_Daily_Ind_HR.nc"
ddpm_path  = ("/path/to/project/PhD_Precipitation/"
              "03_Code/precipitation-ddpm-india/scripts/ddpm/diffusr_climate/"
              "results/precip_ddpm_v5b/inference/ddpm_v5b_precipitation_test_set.nc")
srcnn_path = ("/path/to/project/PhD_Precipitation/"
              "03_Code/results/srcnn_baseline/srcnn_pred_2011-2014.nc")
fm_path    = ("/path/to/project/Flow_matching_downscaling/"
              "results/precip_fm_v1/precip_fm_v1_heun50_merged_test.nc")
ca_path    = ("/path/to/project/PhD_Precipitation/"
              "03_Code/results/ca_baseline/ca_pred_2011-2014.nc")
wt_path    = ("/path/to/project/PhD_Precipitation/"
              "03_Code/results/weather_typing_baseline/wt_pred_2011-2014.nc")
output_dir = ("/path/to/project/PhD_Precipitation/"
              "03_Code/notebooks/results/precip_ddpm_v5b")

time_slice = slice('2011-01-01', '2014-12-31')
lat_slice  = slice(39.95, 5.05)
lon_slice  = slice(65.05, 99.95)
WET_THR    = 1.0

MODEL_ORDER = ['MSWX (Observed)', 'CA', 'WT', 'SRCNN', 'DDPM', 'Flow Matching']

PALETTE = {
    'MSWX (Observed)': '#DDDDDD',
    'CA':              '#56B4E9',
    'WT':              '#009E73',
    'SRCNN':           '#E69F00',
    'DDPM':            '#0072B2',
    'Flow Matching':   '#D55E00',
}

OBS_EDGE   = '#333333'
OBS_HATCH  = '////'

MODELS_PATHS = {
    'Flow Matching': (fm_path,    'precipitation'),
    'DDPM':          (ddpm_path,  'precipitation'),
    'SRCNN':         (srcnn_path, 'pr_srcnn'),
    'CA':            (ca_path,    'pr_ca'),
    'WT':            (wt_path,    'pr_wt'),
}

DRY_BINS   = [1, 4, 8, 15, 30, np.inf]
DRY_LABELS = ['1–3', '4–7', '8–14', '15–29', '≥30']
WET_BINS   = [1, 4, 8, 15, np.inf]
WET_LABELS = ['1–3', '4–7', '8–14', '≥15']

# ── Spell extraction ───────────────────────────────────────────────────────────
def extract_spell_lengths(data_3d, condition='dry', threshold=1.0):
    n_time, n_lat, n_lon = data_3d.shape
    d2   = data_3d.reshape(n_time, -1)
    mask = ((d2 < threshold) if condition == 'dry' else (d2 >= threshold)) & ~np.isnan(d2)
    mt   = mask.T
    n_pix = mt.shape[0]
    pad  = np.hstack([np.zeros((n_pix, 1), bool), mt, np.zeros((n_pix, 1), bool)])
    diff = pad[:, 1:].astype(int) - pad[:, :-1].astype(int)
    starts = np.where(diff ==  1)[1]
    ends   = np.where(diff == -1)[1]
    return ends - starts

def binned_freq(lengths, bins):
    if not len(lengths):
        return np.zeros(len(bins) - 1)
    c, _ = np.histogram(lengths, bins=bins)
    return c / c.sum() * 100.0

def mask_dead_pixels(data_3d, label):
    """Grid cells whose entire test-period series is exactly 0.0 are the
    boundary-artifact signature identified via diagnose_max_spell.py and
    mask_dead_pixels_and_recompute.py (SRCNN, n=1046, all at the domain
    edge). Set them to NaN so extract_spell_lengths drops them entirely,
    matching the treatment already applied to Table 6. No-op for any
    model that has no dead pixels."""
    dead = np.all(data_3d == 0.0, axis=0)
    n_dead = int(dead.sum())
    if n_dead > 0:
        print(f"    {label}: masking {n_dead} dead (all-zero) pixels before spell extraction")
        data_3d = data_3d.copy()
        data_3d[:, dead] = np.nan
    return data_3d

# ── Data loading ───────────────────────────────────────────────────────────────
def load_and_process():
    print("Loading MSWX observations …")
    ds_obs  = xr.open_dataset(obs_path)
    lat_min, lat_max = sorted([lat_slice.start, lat_slice.stop])
    obs_lat = slice(lat_max, lat_min) if ds_obs.lat[0] > ds_obs.lat[-1] else slice(lat_min, lat_max)
    da_obs  = ds_obs.sel(time=time_slice, lat=obs_lat, lon=lon_slice)['precipitation']
    ov      = mask_dead_pixels(da_obs.values, 'MSWX (Observed)')

    results = {
        'MSWX (Observed)': {
            'dry': binned_freq(extract_spell_lengths(ov, 'dry', WET_THR), DRY_BINS),
            'wet': binned_freq(extract_spell_lengths(ov, 'wet', WET_THR), WET_BINS),
        }
    }

    for mname in MODEL_ORDER[1:]:
        path, var = MODELS_PATHS[mname]
        print(f"  Processing {mname} …")
        with xr.open_dataset(path) as ds:
            da = ds.sel(time=time_slice)[var]
            try:
                xr.testing.assert_allclose(da_obs.lat, da.lat)
                da_a = da
            except AssertionError:
                da_a = da.sel(lat=da_obs.lat, lon=da_obs.lon, method='nearest')
            v = mask_dead_pixels(da_a.values, mname)
            results[mname] = {
                'dry': binned_freq(extract_spell_lengths(v, 'dry', WET_THR), DRY_BINS),
                'wet': binned_freq(extract_spell_lengths(v, 'wet', WET_THR), WET_BINS),
            }
    ds_obs.close()
    return results

# ── Plotting ───────────────────────────────────────────────────────────────────
def _style_ax(ax):
    """Apply clean Nature Comms axis style."""
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.spines['left'].set_linewidth(0.65)
    ax.spines['bottom'].set_linewidth(0.65)
    ax.tick_params(axis='both', which='both', direction='out', width=0.65)
    ax.tick_params(axis='x', which='minor', bottom=False)
    ax.yaxis.set_minor_locator(mticker.AutoMinorLocator(2))
    ax.grid(axis='y', which='major', linestyle=':', linewidth=0.55,
            color='#C8C8C8', zorder=0)
    ax.set_axisbelow(True)


def plot_binned_spells(results):
    os.makedirs(output_dir, exist_ok=True)

    N     = len(MODEL_ORDER)
    BAR_W = 0.105
    STEP  = 0.118   # > BAR_W → hairline gap between bars within a group

    fig, axes = plt.subplots(2, 1, figsize=(7.09, 6.3), facecolor='white')

    panels = [
        (axes[0], 'dry', DRY_LABELS,
         'Dry spell length distribution  (<1 mm day⁻¹)',
         'Consecutive dry days', 'a'),
        (axes[1], 'wet', WET_LABELS,
         'Wet spell length distribution  (≥1 mm day⁻¹)',
         'Consecutive wet days', 'b'),
    ]

    for ax, spell, xlabels, title, xlabel, panel_tag in panels:
        x = np.arange(len(xlabels), dtype=float)

        max_freq = max([max(results[m][spell]) for m in MODEL_ORDER])
        ax.set_ylim(0, max_freq * 1.35)

        obs_freq = results['MSWX (Observed)'][spell]

        for i, mname in enumerate(MODEL_ORDER):
            freq   = results[mname][spell]
            offset = (i - N / 2 + 0.5) * STEP
            col    = PALETTE[mname]

            if mname == 'MSWX (Observed)':
                bars = ax.bar(x + offset, freq, width=BAR_W,
                              color=col, edgecolor=OBS_EDGE, linewidth=0.85,
                              hatch=OBS_HATCH, zorder=3)
            else:
                bars = ax.bar(x + offset, freq, width=BAR_W,
                              color=col, edgecolor='white', linewidth=0.3,
                              alpha=0.92, zorder=3)

                diff = freq - obs_freq
                for j, bar in enumerate(bars):
                    height = bar.get_height()
                    ax.text(bar.get_x() + bar.get_width() / 2, height + (max_freq * 0.02),
                            f"{diff[j]:+.2f}",
                            ha='center', va='bottom', fontsize=5.5, rotation=90,
                            color='#444444', zorder=4)

        ax.set_xticks(x)
        ax.set_xticklabels(xlabels)
        ax.set_ylabel('Relative frequency (%)', labelpad=5)
        ax.set_xlabel(xlabel, labelpad=4)
        ax.set_title(title, fontweight='bold', pad=6, loc='left')
        _style_ax(ax)

        ax.text(-0.09, 1.03, panel_tag, transform=ax.transAxes,
                fontsize=11, fontweight='bold', fontstyle='italic',
                va='bottom', ha='left')

    leg_handles = []
    for mname in MODEL_ORDER:
        if mname == 'MSWX (Observed)':
            p = Patch(facecolor=PALETTE[mname], edgecolor=OBS_EDGE,
                      hatch=OBS_HATCH, linewidth=0.85, label=mname)
        else:
            p = Patch(facecolor=PALETTE[mname], edgecolor='white',
                      linewidth=0.3, alpha=0.92, label=mname)
        leg_handles.append(p)

    fig.legend(handles=leg_handles, loc='lower center', ncol=3,
               bbox_to_anchor=(0.5, -0.02),
               fontsize=8, frameon=True, framealpha=0.97,
               edgecolor='#CCCCCC', fancybox=False,
               handlelength=1.6, handleheight=1.0,
               borderpad=0.7, columnspacing=1.2, handletextpad=0.5)

    plt.tight_layout(rect=[0, 0.09, 1, 1])

    base = 'precip_spell_lengths_binned_2011_2014'
    fig.savefig(os.path.join(output_dir, f'{base}.pdf'),
                dpi=300, bbox_inches='tight', facecolor='white')
    fig.savefig(os.path.join(output_dir, f'{base}.png'),
                dpi=300, bbox_inches='tight', facecolor='white')
    print(f"\nSaved →  {output_dir}/{base}.[pdf|png]")
    plt.show()
    plt.close(fig)


# ── Main ───────────────────────────────────────────────────────────────────────
if __name__ == '__main__':
    results = load_and_process()
    plot_binned_spells(results)
    print("Done.")

In [ ]:
"""
diagnose_max_spell.py
======================
Follow-up to analyse_spell_lengths.py.

Table 1D shows WT and SRCNN both hitting max = 1461 days for dry spells,
i.e. exactly the full length of the test period, at (at least) one grid
cell each. The manuscript text only calls this out for WT. Before that
sentence goes to print, this script answers three questions:

  1. Is WT's worst pixel the SAME location as SRCNN's worst pixel?
  2. Is SRCNN's worst pixel sitting on the domain edge? (classic CNN
     zero-padding artifact territory - SRCNN has no "regime" concept,
     so if it reproduces WT's exact ceiling value, it needs its own
     explanation, not WT's.)
  3. Is this a single rogue pixel per model, or a broad pattern across
     many pixels? A single pixel points to an artifact; many pixels
     points to a genuine methodological effect.

No plots, no CSVs overwritten - this only prints diagnostics.
"""

import numpy as np
import xarray as xr
import warnings

warnings.filterwarnings("ignore")

# ============================================================
# PATHS (identical to analyse_spell_lengths.py)
# ============================================================
obs_path   = "/path/to/data/MSWX_Pcp_Daily_Ind_HR.nc"
ddpm_path  = ("/path/to/project/PhD_Precipitation/"
              "03_Code/precipitation-ddpm-india/scripts/ddpm/diffusr_climate/"
              "results/precip_ddpm_v5b/inference/ddpm_v5b_precipitation_test_set.nc")
srcnn_path = ("/path/to/project/PhD_Precipitation/"
              "03_Code/results/srcnn_baseline/srcnn_pred_2011-2014.nc")
fm_path    = ("/path/to/project/Flow_matching_downscaling/"
              "results/precip_fm_v1/precip_fm_v1_heun50_merged_test.nc")
ca_path    = ("/path/to/project/PhD_Precipitation/"
              "03_Code/results/ca_baseline/ca_pred_2011-2014.nc")
wt_path    = ("/path/to/project/PhD_Precipitation/"
              "03_Code/results/weather_typing_baseline/wt_pred_2011-2014.nc")

time_slice = slice('2011-01-01', '2014-12-31')
lat_slice  = slice(39.95, 5.05)
lon_slice  = slice(65.05, 99.95)
WET_THR    = 1.0

MODELS_PATHS = {
    'Flow Matching': (fm_path,    'precipitation'),
    'DDPM':          (ddpm_path,  'precipitation'),
    'SRCNN':         (srcnn_path, 'pr_srcnn'),
    'CA':            (ca_path,    'pr_ca'),
    'WT':            (wt_path,    'pr_wt'),
}

# pixels within this many grid cells of the domain boundary count as "edge"
# adjust upward if SRCNN's receptive field is large (more conv layers / no padding)
EDGE_BUFFER = 3


# ============================================================
# spell extraction that also returns WHICH pixel each spell belongs to
# (same "runs" logic as analyse_spell_lengths.py, extended to track location)
# ============================================================
def extract_spells_with_location(data_3d, condition='dry', threshold=1.0):
    n_time, n_lat, n_lon = data_3d.shape
    data_2d = data_3d.reshape(n_time, -1)

    if condition == 'dry':
        mask = (data_2d < threshold) & ~np.isnan(data_2d)
    else:
        mask = (data_2d >= threshold) & ~np.isnan(data_2d)

    mask_t  = mask.T
    n_pixels = mask_t.shape[0]
    padded  = np.hstack([np.zeros((n_pixels, 1), dtype=bool),
                          mask_t,
                          np.zeros((n_pixels, 1), dtype=bool)])
    diff = padded[:, 1:].astype(int) - padded[:, :-1].astype(int)
    starts_rows, starts_cols = np.where(diff == 1)
    _,           ends_cols   = np.where(diff == -1)

    lengths = ends_cols - starts_cols
    pixel   = starts_rows          # flattened (lat, lon) index for each spell
    return lengths, pixel, n_lat, n_lon


def locate_worst_pixel(vals, lat_arr, lon_arr, condition='dry', threshold=1.0):
    lengths, pixel, n_lat, n_lon = extract_spells_with_location(vals, condition, threshold)
    i_max        = np.argmax(lengths)
    max_len      = int(lengths[i_max])
    flat_pix     = int(pixel[i_max])
    lat_i, lon_i = np.unravel_index(flat_pix, (n_lat, n_lon))
    is_edge = (lat_i < EDGE_BUFFER or lat_i >= n_lat - EDGE_BUFFER or
               lon_i < EDGE_BUFFER or lon_i >= n_lon - EDGE_BUFFER)
    return dict(max_len=max_len, lat_i=lat_i, lon_i=lon_i,
                lat=float(lat_arr[lat_i]), lon=float(lon_arr[lon_i]),
                n_lat=n_lat, n_lon=n_lon, is_edge_pixel=is_edge)


# ============================================================
# main
# ============================================================
def main():
    print("Loading MSWX...")
    ds_obs  = xr.open_dataset(obs_path)
    lat_min, lat_max = sorted([lat_slice.start, lat_slice.stop])
    obs_lat = (slice(lat_max, lat_min)
               if ds_obs.lat[0] > ds_obs.lat[-1]
               else slice(lat_min, lat_max))
    da_obs   = ds_obs.sel(time=time_slice, lat=obs_lat, lon=lon_slice)['precipitation']
    obs_vals = da_obs.values
    lat_arr, lon_arr = da_obs.lat.values, da_obs.lon.values

    all_vals = {'MSWX (Observed)': obs_vals}
    for mname, (path, varname) in MODELS_PATHS.items():
        print(f"  Loading {mname}...")
        with xr.open_dataset(path) as ds_m:
            da_m = ds_m.sel(time=time_slice)[varname]
            try:
                xr.testing.assert_allclose(da_obs.lat, da_m.lat)
            except AssertionError:
                da_m = da_m.sel(lat=da_obs.lat, lon=da_obs.lon, method='nearest')
            all_vals[mname] = da_m.values
    ds_obs.close()

    print(f"\nDomain grid: {len(lat_arr)} x {len(lon_arr)} "
          f"(edge buffer = {EDGE_BUFFER} pixels)\n")

    # ---- Step 1: find the pixel behind each model's max DRY spell ----
    print("=" * 100)
    print("WORST DRY-SPELL PIXEL PER MODEL")
    print("=" * 100)
    worst = {}
    for mname, vals in all_vals.items():
        info = locate_worst_pixel(vals, lat_arr, lon_arr, 'dry', WET_THR)
        worst[mname] = info
        print(f"{mname:18s} max={info['max_len']:5d}d  "
              f"lat={info['lat']:.2f} lon={info['lon']:.2f}  "
              f"grid=({info['lat_i']},{info['lon_i']}) of ({info['n_lat']},{info['n_lon']})  "
              f"EDGE_PIXEL={info['is_edge_pixel']}")

    # ---- Step 2: are WT and SRCNN's worst pixels the SAME location? ----
    wt_loc    = (worst['WT']['lat_i'],    worst['WT']['lon_i'])
    srcnn_loc = (worst['SRCNN']['lat_i'], worst['SRCNN']['lon_i'])
    print(f"\nWT worst pixel == SRCNN worst pixel? {wt_loc == srcnn_loc}  "
          f"(WT={wt_loc} @ lat={worst['WT']['lat']:.2f},lon={worst['WT']['lon']:.2f}  "
          f"SRCNN={srcnn_loc} @ lat={worst['SRCNN']['lat']:.2f},lon={worst['SRCNN']['lon']:.2f})")

    # ---- Step 3: raw time series at each flagged model's worst pixel,
    #      cross-referenced against every other model at that SAME location ----
    for target in ['WT', 'SRCNN']:
        lat_i, lon_i = worst[target]['lat_i'], worst[target]['lon_i']
        print(f"\n{'-' * 100}")
        print(f"Raw precipitation at {target}'s worst pixel "
              f"(lat={worst[target]['lat']:.2f}, lon={worst[target]['lon']:.2f}) "
              f"across ALL models:")
        print(f"{'-' * 100}")
        for mname, vals in all_vals.items():
            series   = vals[:, lat_i, lon_i]
            n_nan    = int(np.isnan(series).sum())
            n_zero   = int(np.sum(series == 0))
            n_lt_thr = int(np.sum((series < WET_THR) & ~np.isnan(series)))
            print(f"  {mname:18s} min={np.nanmin(series):8.3f}  "
                  f"max={np.nanmax(series):8.3f}  mean={np.nanmean(series):8.3f}  "
                  f"n_NaN={n_nan:5d}  n_exact0={n_zero:5d}  "
                  f"n_below_{WET_THR}mm={n_lt_thr:5d}/{len(series)}")

    # ---- Step 4: how many pixels (not just the single worst) exceed a
    #      1-year dry spell in each model? Many pixels => broad/genuine
    #      pattern. One or two pixels => localized artifact more likely. ----
    print(f"\n{'-' * 100}")
    print("Pixels with a >=365-day dry spell (breadth check):")
    print(f"{'-' * 100}")
    for mname, vals in all_vals.items():
        lengths, pixel, n_lat, n_lon = extract_spells_with_location(vals, 'dry', WET_THR)
        long_pixels = np.unique(pixel[lengths >= 365])
        print(f"  {mname:18s} n_pixels_with_365d+_dry_spell = {len(long_pixels)}")

    # ---- Step 5: same breadth check but on the WET side, for WT and SRCNN
    #      only. WT's wet max (722d) is already far above everyone else's
    #      (~200d) in Table 1W - if that also traces to many pixels, it
    #      supports "WT genuinely gets stuck in regimes" as a general
    #      property, not a one-off. SRCNN's wet max (208d) was unremarkable
    #      in Table 1W already, so this is mostly a confirmation check. ----
    print(f"\n{'-' * 100}")
    print("Pixels with a >=180-day WET spell (WT vs SRCNN breadth check):")
    print(f"{'-' * 100}")
    for mname in ['WT', 'SRCNN']:
        lengths, pixel, n_lat, n_lon = extract_spells_with_location(
            all_vals[mname], 'wet', WET_THR)
        long_pixels = np.unique(pixel[lengths >= 180])
        print(f"  {mname:18s} n_pixels_with_180d+_wet_spell = {len(long_pixels)}")

    print("\nDone.")


if __name__ == '__main__':
    main()

In [ ]:
"""
mask_dead_pixels_and_recompute.py
===================================
Follow-up to diagnose_max_spell.py.

What diagnose_max_spell.py already established:
  - SRCNN's worst dry-spell pixel is the domain's literal NW corner
    (grid (0,0)), EXACT 0.000 mm/day, ZERO variance, for all 1461 days.
    MSWX shows real rain (13-20mm max) at that same location, so this
    cannot be a genuine climate signal.
  - SRCNN is ALSO exact 0.000 at WT's worst pixel (a different lat/lon),
    meaning it isn't one unlucky corner cell, it looks like a dead band.
  - WT's own worst pixel, by contrast, has small but REAL fluctuating
    values (0.006-0.885 mm, never exactly 0) that simply never cross
    1 mm under any of its regime states. That is a genuine (if extreme)
    methodological signature, not an artifact.
  - Breadth check: SRCNN has 1315 pixels with a >=365-day dry spell;
    WT has 5265. Different orders of a similar-looking number, but
    likely different mechanisms behind them.

This script turns "likely a boundary artifact" into a precise, testable
claim: it finds every pixel per model whose ENTIRE 1461-day series is
EXACTLY 0.0 (the strongest, least ambiguous artifact signature - real
precipitation fields essentially never do this over 4 years at a land
grid cell), reports how tightly those pixels cluster near the domain
edge, and recomputes the core Table 6 dry-spell statistics with them
masked out, so you can see exactly how much (if at all) SRCNN's
reported numbers move.

No plots, no CSVs overwritten - prints only.
"""

import numpy as np
import pandas as pd
import xarray as xr
from scipy import stats as scipy_stats
from scipy.stats import ks_2samp, wasserstein_distance
import warnings

warnings.filterwarnings("ignore")

# ============================================================
# PATHS (identical to analyse_spell_lengths.py)
# ============================================================
obs_path   = "/path/to/data/MSWX_Pcp_Daily_Ind_HR.nc"
ddpm_path  = ("/path/to/project/PhD_Precipitation/"
              "03_Code/precipitation-ddpm-india/scripts/ddpm/diffusr_climate/"
              "results/precip_ddpm_v5b/inference/ddpm_v5b_precipitation_test_set.nc")
srcnn_path = ("/path/to/project/PhD_Precipitation/"
              "03_Code/results/srcnn_baseline/srcnn_pred_2011-2014.nc")
fm_path    = ("/path/to/project/Flow_matching_downscaling/"
              "results/precip_fm_v1/precip_fm_v1_heun50_merged_test.nc")
ca_path    = ("/path/to/project/PhD_Precipitation/"
              "03_Code/results/ca_baseline/ca_pred_2011-2014.nc")
wt_path    = ("/path/to/project/PhD_Precipitation/"
              "03_Code/results/weather_typing_baseline/wt_pred_2011-2014.nc")

time_slice = slice('2011-01-01', '2014-12-31')
lat_slice  = slice(39.95, 5.05)
lon_slice  = slice(65.05, 99.95)
WET_THR    = 1.0

MODEL_ORDER  = ['CA', 'WT', 'SRCNN', 'DDPM', 'Flow Matching']
MODELS_PATHS = {
    'Flow Matching': (fm_path,    'precipitation'),
    'DDPM':          (ddpm_path,  'precipitation'),
    'SRCNN':         (srcnn_path, 'pr_srcnn'),
    'CA':            (ca_path,    'pr_ca'),
    'WT':            (wt_path,    'pr_wt'),
}
DRY_THRESHOLDS = [10, 30]   # matches R10 / R30 in manuscript Table 6


def extract_spell_lengths(data_3d, condition='dry', threshold=1.0):
    n_time, n_lat, n_lon = data_3d.shape
    data_2d = data_3d.reshape(n_time, -1)
    if condition == 'dry':
        mask = (data_2d < threshold) & ~np.isnan(data_2d)
    else:
        mask = (data_2d >= threshold) & ~np.isnan(data_2d)
    mask_t  = mask.T
    n_pixels = mask_t.shape[0]
    padded  = np.hstack([np.zeros((n_pixels, 1), dtype=bool),
                          mask_t,
                          np.zeros((n_pixels, 1), dtype=bool)])
    diff = padded[:, 1:].astype(int) - padded[:, :-1].astype(int)
    starts = np.where(diff == 1)[1]
    ends   = np.where(diff == -1)[1]
    return ends - starts


def core_stats(lengths, thresholds, obs_lengths):
    n = len(lengths)
    pcts = np.percentile(lengths, [90, 99])
    mean_len = float(np.mean(lengths))
    bins = np.arange(1, 51)
    counts, _ = np.histogram(lengths, bins=np.append(bins, bins[-1] + 1))
    probs = counts / n
    valid = probs > 0
    if valid.sum() >= 5:
        res = scipy_stats.linregress(bins[valid], np.log(probs[valid]))
        decay_slope = float(res.slope)
    else:
        decay_slope = np.nan

    rng = np.random.default_rng(42)
    max_n = 500_000
    o = obs_lengths if len(obs_lengths) <= max_n else rng.choice(obs_lengths, max_n, replace=False)
    m = lengths if len(lengths) <= max_n else rng.choice(lengths, max_n, replace=False)
    ks_stat, _ = ks_2samp(o.astype(float), m.astype(float))
    w_dist = wasserstein_distance(o.astype(float), m.astype(float))

    out = dict(n_spells=n, mean=mean_len, std=float(np.std(lengths)),
               P90=float(pcts[0]), P99=float(pcts[1]), max=float(np.max(lengths)),
               frac_1day_pct=float(np.mean(lengths == 1) * 100),
               decay_slope=decay_slope, KS_stat=float(ks_stat),
               Wasserstein_days=float(w_dist))
    for t in thresholds:
        p_obs = float(np.mean(obs_lengths >= t))
        p_mod = float(np.mean(lengths >= t))
        out[f'R{t}'] = float(p_mod / p_obs) if p_obs > 0 else np.nan
    return out


def main():
    print("Loading MSWX...")
    ds_obs  = xr.open_dataset(obs_path)
    lat_min, lat_max = sorted([lat_slice.start, lat_slice.stop])
    obs_lat = (slice(lat_max, lat_min)
               if ds_obs.lat[0] > ds_obs.lat[-1]
               else slice(lat_min, lat_max))
    da_obs   = ds_obs.sel(time=time_slice, lat=obs_lat, lon=lon_slice)['precipitation']
    obs_vals = da_obs.values
    ds_obs.close()

    all_vals = {'MSWX (Observed)': obs_vals}
    for mname in MODEL_ORDER:
        path, varname = MODELS_PATHS[mname]
        print(f"  Loading {mname}...")
        with xr.open_dataset(path) as ds_m:
            da_m = ds_m.sel(time=time_slice)[varname]
            try:
                xr.testing.assert_allclose(da_obs.lat, da_m.lat)
            except AssertionError:
                da_m = da_m.sel(lat=da_obs.lat, lon=da_obs.lon, method='nearest')
            all_vals[mname] = da_m.values

    n_lat, n_lon = obs_vals.shape[1], obs_vals.shape[2]
    lat_idx_grid, lon_idx_grid = np.meshgrid(np.arange(n_lat), np.arange(n_lon), indexing='ij')
    dist_to_edge = np.minimum.reduce([lat_idx_grid, n_lat - 1 - lat_idx_grid,
                                       lon_idx_grid, n_lon - 1 - lon_idx_grid])

    # ---- Step 1: find every "dead" pixel per model: exact 0.0 for all 1461 days ----
    print(f"\n{'=' * 100}")
    print("DEAD-PIXEL CHECK (entire 1461-day series is EXACTLY 0.000 mm/day)")
    print("Real precipitation fields essentially never do this over 4 years at a land")
    print("pixel, so any hits here are the artifact signature, not climate.")
    print(f"{'=' * 100}")
    dead_masks = {}
    for mname, vals in all_vals.items():
        dead = np.all(vals == 0.0, axis=0)
        dead_masks[mname] = dead
        n_dead = int(dead.sum())
        if n_dead > 0:
            edge_dists = dist_to_edge[dead]
            print(f"  {mname:18s} n_dead_pixels={n_dead:6d}  "
                  f"dist_to_edge: min={edge_dists.min()} max={edge_dists.max()} "
                  f"median={int(np.median(edge_dists))}  "
                  f"(<=5px from edge: {int((edge_dists <= 5).sum())}/{n_dead})")
        else:
            print(f"  {mname:18s} n_dead_pixels=     0")

    # ---- Step 2: recompute core DRY-spell Table 6 stats with dead pixels masked ----
    print(f"\n{'=' * 100}")
    print("DRY-SPELL TABLE 6 STATS: ORIGINAL vs. DEAD-PIXEL-MASKED")
    print("(each model's own dead-pixel mask applied to itself only; MSWX untouched)")
    print(f"{'=' * 100}")
    obs_lengths = extract_spell_lengths(obs_vals, 'dry', WET_THR)

    rows = []
    ordered = ['SRCNN'] + [m for m in MODEL_ORDER if m != 'SRCNN']
    for mname in ordered:
        vals = all_vals[mname]
        orig = core_stats(extract_spell_lengths(vals, 'dry', WET_THR), DRY_THRESHOLDS, obs_lengths)
        orig['variant'] = f'{mname} (original)'
        rows.append(orig)

        dead = dead_masks[mname]
        if dead.sum() > 0:
            vals_masked = vals.copy()
            vals_masked[:, dead] = np.nan
            masked = core_stats(extract_spell_lengths(vals_masked, 'dry', WET_THR),
                                 DRY_THRESHOLDS, obs_lengths)
            masked['variant'] = f'{mname} (masked, n_dead={int(dead.sum())})'
            rows.append(masked)

    df = pd.DataFrame(rows).set_index('variant')
    cols = ['n_spells', 'mean', 'std', 'P90', 'P99', 'max',
            'frac_1day_pct', 'decay_slope', 'KS_stat', 'Wasserstein_days', 'R10', 'R30']
    with pd.option_context('display.float_format', '{:.4f}'.format,
                           'display.max_columns', 15, 'display.width', 160):
        print(df[cols].to_string())

    print("\nDone.")


if __name__ == '__main__':
    main()

In [ ]:
"""
plot_qq_analysis.py
===================
Nature Communications-style Quantile-Quantile (Q-Q) Plot.

Plots the quantiles of each downscaling model against the MSWX observations.
A perfect model lies exactly on the 1:1 diagonal. This clearly visualizes 
systematic biases across the entire distribution, specifically exposing 
the inability of classical models to reach extreme precipitation intensities.

Flow Matching highlighted as the primary proposed model.
"""

import os
import warnings
import numpy as np
import xarray as xr
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker

warnings.filterwarnings("ignore")

matplotlib.rcParams.update({
    'font.family':       'serif',
    'font.serif':        ['Times New Roman', 'Times', 'DejaVu Serif'],
    'font.size':         9,
    'axes.labelsize':    10,
    'axes.titlesize':    11,
    'xtick.labelsize':   9,
    'ytick.labelsize':   9,
    'legend.fontsize':   8.5,
    'figure.dpi':        150,
    'axes.linewidth':    0.8,
    'pdf.fonttype':      42,
    'ps.fonttype':       42,
})

# ============================================================
# PATHS
# ============================================================
obs_path     = "/path/to/data/MSWX_Pcp_Daily_Ind_HR.nc"
ddpm_path    = ("/path/to/project/PhD_Precipitation/"
                "03_Code/precipitation-ddpm-india/scripts/ddpm/diffusr_climate/"
                "results/precip_ddpm_v5b/inference/ddpm_v5b_precipitation_test_set.nc")
srcnn_path   = ("/path/to/project/PhD_Precipitation/"
                "03_Code/results/srcnn_baseline/srcnn_pred_2011-2014.nc")
fm_path      = ("/path/to/project/Flow_matching_downscaling/"
                "results/precip_fm_v1/precip_fm_v1_heun50_merged_test.nc")
ca_path      = ("/path/to/project/PhD_Precipitation/"
                "03_Code/results/ca_baseline/ca_pred_2011-2014.nc")
wt_path      = ("/path/to/project/PhD_Precipitation/"
                "03_Code/results/weather_typing_baseline/wt_pred_2011-2014.nc")
output_dir   = ("/path/to/project/PhD_Precipitation/"
                "03_Code/notebooks/results/precip_ddpm_v5b")

time_slice = slice('2011-01-01', '2014-12-31')
lat_slice  = slice(39.95, 5.05)
lon_slice  = slice(65.05, 99.95)
WET_THR    = 1.0   # mm/day

# ============================================================
# CONFIGURATION
# ============================================================
# Reordered to place Flow Matching last as the primary proposed model
MODEL_ORDER = ['CA', 'WT', 'SRCNN', 'DDPM', 'Flow Matching']

PALETTE = {
    'CA':            '#A0C4C8',
    'WT':            '#78B7BC',
    'SRCNN':         '#C68B59',
    'DDPM':          '#8E44AD',  # Muted purple
    'Flow Matching': '#C0292B',  # Highlight red
}

# Layered linestyle strategy
LINESTYLE = {
    'CA':            (0, (4, 2)),       
    'WT':            (0, (2, 2)),       
    'SRCNN':         (0, (4, 2, 1, 2)), 
    'DDPM':          (0, (3, 2)),       # Dashed for DDPM
    'Flow Matching': 'solid',           # Solid core for Flow Matching
}

LINEWIDTH = {
    'CA':            1.2,
    'WT':            1.2,
    'SRCNN':         1.2,
    'DDPM':          1.8,
    'Flow Matching': 2.5,
}

MODELS_PATHS = {
    'Flow Matching': (fm_path,    'precipitation'),
    'DDPM':          (ddpm_path,  'precipitation'),
    'SRCNN':         (srcnn_path, 'pr_srcnn'),
    'CA':            (ca_path,    'pr_ca'),
    'WT':            (wt_path,    'pr_wt'),
}

# ============================================================
# DATA PROCESSING
# ============================================================
def extract_quantiles():
    print("Loading Observations (MSWX)...")
    ds_obs = xr.open_dataset(obs_path)
    lat_min, lat_max = sorted([lat_slice.start, lat_slice.stop])
    obs_lat = slice(lat_max, lat_min) if ds_obs.lat[0] > ds_obs.lat[-1] else slice(lat_min, lat_max)
               
    da_obs = ds_obs.sel(time=time_slice, lat=obs_lat, lon=lon_slice)['precipitation']
    
    # Extract wet days
    obs_flat = da_obs.values.flatten()
    obs_wet = obs_flat[np.isfinite(obs_flat) & (obs_flat >= WET_THR)]
    
    # We use high-resolution quantiles to accurately capture the extreme tail (up to 99.99th)
    q_levels = np.linspace(0.01, 0.9999, 1000)
    print("Computing MSWX quantiles...")
    obs_quantiles = np.quantile(obs_wet, q_levels)

    model_quantiles = {}
    print("\nProcessing Models...")
    for mname in MODEL_ORDER:
        path, varname = MODELS_PATHS[mname]
        print(f"  Computing quantiles for {mname}...")
        
        with xr.open_dataset(path) as ds_m:
            da_m = ds_m.sel(time=time_slice)[varname]
            # Fast nearest-neighbor grid alignment
            try:
                xr.testing.assert_allclose(da_obs.lat, da_m.lat)
                xr.testing.assert_allclose(da_obs.lon, da_m.lon)
                da_m_aligned = da_m
            except AssertionError:
                da_m_aligned = da_m.sel(lat=da_obs.lat, lon=da_obs.lon, method='nearest')
                
            mod_flat = da_m_aligned.values.flatten()
            mod_wet = mod_flat[np.isfinite(mod_flat) & (mod_flat >= WET_THR)]
            
            model_quantiles[mname] = np.quantile(mod_wet, q_levels)

    ds_obs.close()
    return obs_quantiles, model_quantiles

# ============================================================
# PLOTTING
# ============================================================
def plot_qq(obs_quantiles, model_quantiles):
    os.makedirs(output_dir, exist_ok=True)

    # Q-Q plots should be square to accurately reflect the 1:1 diagonal
    fig, ax = plt.subplots(figsize=(6, 6))
    fig.patch.set_facecolor('white')

    plot_max = 250 # mm/day capping limit

    # ── 1:1 Reference Line ──────────────────────────────
    ax.plot([WET_THR, plot_max], [WET_THR, plot_max], 
            color='#555555', linestyle='--', linewidth=1.5, zorder=1, label='Perfect Agreement (1:1)')

    # ── Model Quantiles ─────────────────────────────────
    for mname in MODEL_ORDER:
        is_fm = (mname == 'Flow Matching')
        is_ddpm = (mname == 'DDPM')
        
        lw = LINEWIDTH[mname]
        
        # Z-order stacking for layered rendering
        if is_fm:
            z_ord = 10
            alpha = 1.0
        elif is_ddpm:
            z_ord = 9
            alpha = 0.95
        else:
            z_ord = 5
            alpha = 0.85
        
        ax.plot(obs_quantiles, model_quantiles[mname], 
                color=PALETTE[mname], 
                linestyle=LINESTYLE[mname],
                linewidth=lw, 
                zorder=z_ord, 
                alpha=alpha,
                label=mname)

    # ── Axis Formatting ─────────────────────────────────
    ax.set_xlim(WET_THR, plot_max)
    ax.set_ylim(WET_THR, plot_max)
    
    ax.set_xlabel('MSWX (Observed) Quantiles (mm/day)', fontsize=10, fontweight='bold')
    ax.set_ylabel('Model Quantiles (mm/day)', fontsize=10, fontweight='bold')
    ax.set_title('Quantile-Quantile (Q-Q) Precipitation Plot', fontsize=13, fontweight='bold', pad=12)

    ax.xaxis.set_major_locator(ticker.MultipleLocator(50))
    ax.xaxis.set_minor_locator(ticker.MultipleLocator(10))
    ax.yaxis.set_major_locator(ticker.MultipleLocator(50))
    ax.yaxis.set_minor_locator(ticker.MultipleLocator(10))

    ax.grid(True, which='major', linestyle='-', alpha=0.3, color='#CCCCCC')
    ax.grid(True, which='minor', linestyle=':', alpha=0.3, color='#E8E8E8')

    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.spines['left'].set_linewidth(1.0)
    ax.spines['bottom'].set_linewidth(1.0)

    # ── Legend ──────────────────────────────────────────
    ax.legend(loc='upper left', fontsize=9, frameon=True, framealpha=0.9, 
              edgecolor='#CCCCCC', fancybox=False, handlelength=2.5)

    # ── Insight Annotation ──────────────────────────────
    ax.text(0.95, 0.05, 
            "Deviation below the 1:1 line indicates\nunderestimation of extreme event intensity.",
            transform=ax.transAxes, ha='right', va='bottom',
            fontsize=8, color='#666666', style='italic',
            bbox=dict(boxstyle='round,pad=0.4', fc='#F9F9F9', ec='none'))

    plt.tight_layout()

    # ── Save ─────────────────────────────────────────────
    # out_pdf = os.path.join(output_dir, 'fig_qq_plot.pdf')
    # out_png = os.path.join(output_dir, 'fig_qq_plot.png')
    # fig.savefig(out_pdf, dpi=300, bbox_inches='tight', facecolor='white')
    # fig.savefig(out_png, dpi=300, bbox_inches='tight', facecolor='white')
    # print(f"\nSaved plots to:\n  {out_pdf}\n  {out_png}")

    plt.show()
    plt.close(fig)

# ============================================================
# MAIN
# ============================================================
if __name__ == '__main__':
    obs_q, mod_q = extract_quantiles()
    plot_qq(obs_q, mod_q)
    print("Process complete.")

In [ ]:
"""
plot_lat_lon_profiles.py
========================
Computes and visualizes the mean daily precipitation [mm/d] profiles across
longitude (Zonal Mean) and latitude (Meridional Mean) for MSWX observations
and 5 downscaling models over India (2011–2014).

Visual Enhancements for Nature Communications / JAMES Style:
  - Clean Aesthetics: Removed all point markers, top/right spines, and legend boxes.
  - Strict Hierarchy: Observations (Thick Black), Flow Matching (Red), DDPM (Blue). 
    Baselines are demoted to thin, muted, dashed/dotted lines.
  - Typography: Left-aligned, simplified, bold alphabetical panel titles (a, b, c, d).
  - Uncertainty: Added $\pm 1\sigma$ interannual variability shading to the MSWX reference.
  - Layout: Employs matplotlib's constrained_layout for flawless spacing.
"""

import os
import warnings
import numpy as np
import xarray as xr
import geopandas as gpd
import shapely
from shapely.geometry import box
import matplotlib
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore")

# ============================================================
# MATPLOTLIB PUBLICATION STYLE (Nature / JAMES)
# ============================================================
matplotlib.rcParams.update({
    'font.family':       'serif',
    'font.serif':        ['Times New Roman', 'Times', 'DejaVu Serif'],
    'font.size':         11,
    'axes.labelsize':    12,
    'axes.titlesize':    13,
    'axes.titleweight':  'bold',
    'xtick.labelsize':   10,
    'ytick.labelsize':   10,
    'figure.dpi':        300,  # High res for publication
    'axes.linewidth':    0.8,
    'pdf.fonttype':      42,
    'ps.fonttype':       42,
})

# ============================================================
# PATHS
# ============================================================
obs_path     = "/path/to/data/MSWX_Pcp_Daily_Ind_HR.nc"
ddpm_path    = ("/path/to/project/PhD_Precipitation/"
                "03_Code/precipitation-ddpm-india/scripts/ddpm/diffusr_climate/"
                "results/precip_ddpm_v5b/inference/ddpm_v5b_precipitation_test_set.nc")
srcnn_path   = ("/path/to/project/PhD_Precipitation/"
                "03_Code/results/srcnn_baseline/srcnn_pred_2011-2014.nc")
fm_path      = ("/path/to/project/Flow_matching_downscaling/"
                "results/precip_fm_v1/precip_fm_v1_heun50_merged_test.nc")
ca_path      = ("/path/to/project/PhD_Precipitation/"
                "03_Code/results/ca_baseline/ca_pred_2011-2014.nc")
wt_path      = ("/path/to/project/PhD_Precipitation/"
                "03_Code/results/weather_typing_baseline/wt_pred_2011-2014.nc")
shp_path     = ("/path/to/project/raw-data/shapefiles/ne_10m_admin_0_countries_ind/"
                "ne_10m_admin_0_countries_ind.shp")
output_dir   = ("/path/to/project/PhD_Precipitation/"
                "03_Code/notebooks/results/precip_ddpm_v5b/lat_lon_profiles")

os.makedirs(output_dir, exist_ok=True)

time_slice = slice('2011-01-01', '2014-12-31')
lat_slice  = slice(39.95, 5.05)
lon_slice  = slice(65.05, 99.95)

MODEL_ORDER = ['CA', 'WT', 'SRCNN', 'DDPM', 'Flow Matching']

MODELS_PATHS = {
    'DDPM':          (ddpm_path,  'precipitation'),
    'SRCNN':         (srcnn_path, 'pr_srcnn'),
    'Flow Matching': (fm_path,    'precipitation'),
    'CA':            (ca_path,    'pr_ca'),
    'WT':            (wt_path,    'pr_wt'),
}

# ============================================================
# RESTRAINED VISUAL HIERARCHY PALETTE (No Markers)
# ============================================================
STYLE_PALETTE = {
    'MSWX (Observed)': {'color': 'black',   'ls': '-',  'lw': 2.5, 'zorder': 10, 'label': 'MSWX (Observed)'},
    'Flow Matching':   {'color': '#E64B35', 'ls': '-',  'lw': 1.8, 'zorder': 9,  'label': 'Flow Matching'},
    'DDPM':            {'color': '#3182BD', 'ls': '-',  'lw': 1.8, 'zorder': 8,  'label': 'DDPM'},
    'SRCNN':           {'color': '#4DAF4A', 'ls': '--', 'lw': 1.2, 'zorder': 4,  'label': 'SRCNN'},
    'CA':              {'color': '#969696', 'ls': '-.', 'lw': 1.2, 'zorder': 3,  'label': 'CA'},
    'WT':              {'color': '#BDBDBD', 'ls': ':',  'lw': 1.2, 'zorder': 2,  'label': 'WT'},
}

# ============================================================
# 1. LOAD DATA & BUILD LAND MASK
# ============================================================
print("Loading MSWX Observations...")
ds_obs = xr.open_dataset(obs_path)

lat_min, lat_max = sorted([lat_slice.start, lat_slice.stop])
obs_lat = slice(lat_max, lat_min) if ds_obs.lat[0] > ds_obs.lat[-1] else slice(lat_min, lat_max)
da_obs = ds_obs.sel(time=time_slice, lat=obs_lat, lon=lon_slice)['precipitation'].compute()

print("Loading Shapefile & Building Vectorized Land Mask...")
gdf = gpd.read_file(shp_path)
if gdf.crs is not None and gdf.crs.to_epsg() != 4326:
    gdf = gdf.to_crs(4326)

domain_box = box(float(da_obs.lon.min()), float(da_obs.lat.min()), float(da_obs.lon.max()), float(da_obs.lat.max()))
gdf = gpd.clip(gdf, domain_box)
geom = gdf.union_all() if hasattr(gdf, 'union_all') else gdf.unary_union

LO, LA = np.meshgrid(da_obs.lon.values, da_obs.lat.values)
inside_mask = shapely.contains_xy(geom, LO.ravel(), LA.ravel()).reshape(LA.shape)
india_mask = xr.DataArray(inside_mask, coords=[da_obs.lat, da_obs.lon], dims=['lat', 'lon'])

da_obs_masked = da_obs.where(india_mask)

# ============================================================
# 2. COMPUTE SPATIAL, TEMPORAL PROFILES & UNCERTAINTY
# ============================================================
def compute_profiles(da):
    """Computes spatial means and interannual standard deviations."""
    # 1. Overall Temporal Mean
    mean_time = da.mean(dim='time', skipna=True)
    lat_prof = mean_time.mean(dim='lon', skipna=True).sortby('lat')
    lon_prof = mean_time.mean(dim='lat', skipna=True).sortby('lon')
    
    # 2. Interannual Variability (for uncertainty envelope)
    da_yearly = da.groupby('time.year').mean(dim='time', skipna=True)
    lat_std = da_yearly.mean(dim='lon', skipna=True).std(dim='year').sortby('lat')
    lon_std = da_yearly.mean(dim='lat', skipna=True).std(dim='year').sortby('lon')
    
    return lat_prof, lon_prof, lat_std, lon_std

print("Computing Lat/Lon profiles for Observations...")
lat_profiles, lon_profiles = {}, {}
lat_std_dict, lon_std_dict = {}, {}

lat_prof, lon_prof, lat_std, lon_std = compute_profiles(da_obs_masked)
lat_profiles['MSWX (Observed)'] = lat_prof
lon_profiles['MSWX (Observed)'] = lon_prof
lat_std_dict['MSWX (Observed)'] = lat_std
lon_std_dict['MSWX (Observed)'] = lon_std

for mname in MODEL_ORDER:
    print(f"Computing Lat/Lon profiles for {mname}...")
    path, varname = MODELS_PATHS[mname]
    with xr.open_dataset(path) as ds_m:
        da_m = ds_m.sel(time=time_slice)[varname]
        da_m_aligned = da_m.sel(lat=da_obs.lat, lon=da_obs.lon, method='nearest').compute()
        da_m_masked = da_m_aligned.where(india_mask)
        
        lat_prof, lon_prof, _, _ = compute_profiles(da_m_masked)
        lat_profiles[mname] = lat_prof
        lon_profiles[mname] = lon_prof

ds_obs.close()

# ============================================================
# 3. GENERATE 2x2 PUBLICATION FIGURE
# ============================================================
print("\nGenerating Nature/JAMES styled 2x2 publication figure...")

# Use constrained_layout for automatic flawless margins
fig, axes = plt.subplots(2, 2, figsize=(12, 8), sharex='col', constrained_layout=True)
fig.patch.set_facecolor('white')

((ax_lat_abs, ax_lon_abs), (ax_lat_bias, ax_lon_bias)) = axes
plot_order = ['MSWX (Observed)'] + MODEL_ORDER

# --- Apply Journal Aesthetic Tweaks to All Axes ---
for ax in axes.flat:
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.grid(True, linestyle=':', alpha=0.6, color='#B0B0B0')

# --- Row 1: Absolute Profiles ---
# Add Observation Uncertainty Envelopes first (so lines draw over them)
obs_lat_p = lat_profiles['MSWX (Observed)']
obs_lon_p = lon_profiles['MSWX (Observed)']
obs_lat_s = lat_std_dict['MSWX (Observed)']
obs_lon_s = lon_std_dict['MSWX (Observed)']

ax_lat_abs.fill_between(obs_lat_p.lat.values, obs_lat_p.values - obs_lat_s.values, obs_lat_p.values + obs_lat_s.values,
                        color='black', alpha=0.12, zorder=1, lw=0, label='Obs. $\pm1\\sigma$ (Interannual)')
ax_lon_abs.fill_between(obs_lon_p.lon.values, obs_lon_p.values - obs_lon_s.values, obs_lon_p.values + obs_lon_s.values,
                        color='black', alpha=0.12, zorder=1, lw=0)

for name in plot_order:
    sty = STYLE_PALETTE[name]
    
    # Latitude Absolute
    prof_lat = lat_profiles[name]
    ax_lat_abs.plot(prof_lat.lat.values, prof_lat.values, 
                    label=sty['label'], color=sty['color'], linestyle=sty['ls'], 
                    linewidth=sty['lw'], zorder=sty['zorder'])
    
    # Longitude Absolute
    prof_lon = lon_profiles[name]
    ax_lon_abs.plot(prof_lon.lon.values, prof_lon.values, 
                    label=sty['label'], color=sty['color'], linestyle=sty['ls'], 
                    linewidth=sty['lw'], zorder=sty['zorder'])

ax_lat_abs.set_title("a  Meridional profile", loc='left')
ax_lat_abs.set_ylabel("Precipitation (mm day⁻¹)")
ax_lat_abs.set_xlim(8, 38)

ax_lon_abs.set_title("b  Zonal profile", loc='left')
ax_lon_abs.set_xlim(68, 97)

# --- Row 2: Bias Profiles (Model - Observed) ---
# Bold Zero-Reference lines
ax_lat_bias.axhline(0, color='black', linestyle='-', linewidth=1.2, alpha=0.8, zorder=5)
ax_lon_bias.axhline(0, color='black', linestyle='-', linewidth=1.2, alpha=0.8, zorder=5)

for mname in MODEL_ORDER:
    sty = STYLE_PALETTE[mname]
    
    # Latitude Bias
    bias_lat = lat_profiles[mname] - lat_profiles['MSWX (Observed)']
    ax_lat_bias.plot(bias_lat.lat.values, bias_lat.values, 
                     color=sty['color'], linestyle=sty['ls'], 
                     linewidth=sty['lw'], zorder=sty['zorder'])
    
    # Longitude Bias
    bias_lon = lon_profiles[mname] - lon_profiles['MSWX (Observed)']
    ax_lon_bias.plot(bias_lon.lon.values, bias_lon.values, 
                     color=sty['color'], linestyle=sty['ls'], 
                     linewidth=sty['lw'], zorder=sty['zorder'])

ax_lat_bias.set_title("c  Meridional bias", loc='left')
ax_lat_bias.set_xlabel("Latitude (°N)")
ax_lat_bias.set_ylabel("Bias (mm day⁻¹)")

ax_lon_bias.set_title("d  Zonal bias", loc='left')
ax_lon_bias.set_xlabel("Longitude (°E)")

# --- Clean Frame-less Legend ---
# Extract handles from one of the absolute axes to include the uncertainty patch
handles, labels = ax_lat_abs.get_legend_handles_labels()
fig.legend(handles, labels, loc='lower center', bbox_to_anchor=(0.5, -0.06),
           ncol=4, frameon=False, fontsize=11, columnspacing=2.0)

# ============================================================
# 4. SAVE AT 300 DPI
# ============================================================
out_png = os.path.join(output_dir, "mean_precip_lat_lon_profiles_enhanced_2011_2014.png")
out_pdf = os.path.join(output_dir, "mean_precip_lat_lon_profiles_enhanced_2011_2014.pdf")

# bbox_inches='tight' works alongside constrained_layout to preserve the legend at the bottom
fig.savefig(out_png, dpi=300, bbox_inches='tight', facecolor='white')
fig.savefig(out_pdf, dpi=300, bbox_inches='tight', facecolor='white')

print(f"✅ SUCCESS: Enhanced high-distinction 2x2 figure saved to:\n  {out_png}\n  {out_pdf}")
plt.show()

In [ ]:
"""
analyse_lat_lon_profiles.py
===========================
Statistical counterpart to plot_lat_lon_profiles.py.
Produces manuscript-ready numerical analysis of zonal (longitudinal)
and meridional (latitudinal) precipitation profiles over India (2011–2014).

No plots. No hallucinations. Pure numerical rigor.

Outputs generated:
  1. Summary terminal tables with RMSE, MAE, MBE, Correlation, and Peak Errors
  2. Complete CSV exports of absolute 1D profiles (lat & lon)
  3. Complete CSV exports of model bias profiles (lat & lon)
  4. Model ranking CSV based on 1D profile spatial fidelity
"""

import os
import warnings
import numpy as np
import pandas as pd
import xarray as xr
import geopandas as gpd
import shapely
from shapely.geometry import box
from scipy.stats import pearsonr

warnings.filterwarnings("ignore")

# ============================================================
# PATHS
# ============================================================
obs_path     = "/path/to/data/MSWX_Pcp_Daily_Ind_HR.nc"
ddpm_path    = ("/path/to/project/PhD_Precipitation/"
                "03_Code/precipitation-ddpm-india/scripts/ddpm/diffusr_climate/"
                "results/precip_ddpm_v5b/inference/ddpm_v5b_precipitation_test_set.nc")
srcnn_path   = ("/path/to/project/PhD_Precipitation/"
                "03_Code/results/srcnn_baseline/srcnn_pred_2011-2014.nc")
fm_path      = ("/path/to/project/Flow_matching_downscaling/"
                "results/precip_fm_v1/precip_fm_v1_heun50_merged_test.nc")
ca_path      = ("/path/to/project/PhD_Precipitation/"
                "03_Code/results/ca_baseline/ca_pred_2011-2014.nc")
wt_path      = ("/path/to/project/PhD_Precipitation/"
                "03_Code/results/weather_typing_baseline/wt_pred_2011-2014.nc")
shp_path     = ("/path/to/project/raw-data/shapefiles/ne_10m_admin_0_countries_ind/"
                "ne_10m_admin_0_countries_ind.shp")
output_dir   = ("/path/to/project/PhD_Precipitation/"
                "03_Code/notebooks/results/precip_ddpm_v5b/lat_lon_profiles")

os.makedirs(output_dir, exist_ok=True)

time_slice = slice('2011-01-01', '2014-12-31')
lat_slice  = slice(39.95, 5.05)
lon_slice  = slice(65.05, 99.95)

MODEL_ORDER = ['CA', 'WT', 'SRCNN', 'DDPM', 'Flow Matching']

MODELS_PATHS = {
    'DDPM':          (ddpm_path,  'precipitation'),
    'SRCNN':         (srcnn_path, 'pr_srcnn'),
    'Flow Matching': (fm_path,    'precipitation'),
    'CA':            (ca_path,    'pr_ca'),
    'WT':            (wt_path,    'pr_wt'),
}

# ============================================================
# 1. LOAD DATA & BUILD LAND MASK
# ============================================================
def load_and_compute_profiles():
    print("Loading MSWX Observations...")
    ds_obs = xr.open_dataset(obs_path)

    lat_min, lat_max = sorted([lat_slice.start, lat_slice.stop])
    obs_lat = slice(lat_max, lat_min) if ds_obs.lat[0] > ds_obs.lat[-1] else slice(lat_min, lat_max)
    da_obs = ds_obs.sel(time=time_slice, lat=obs_lat, lon=lon_slice)['precipitation'].compute()

    print("Loading Shapefile & Building Vectorized Land Mask...")
    gdf = gpd.read_file(shp_path)
    if gdf.crs is not None and gdf.crs.to_epsg() != 4326:
        gdf = gdf.to_crs(4326)

    domain_box = box(float(da_obs.lon.min()), float(da_obs.lat.min()), float(da_obs.lon.max()), float(da_obs.lat.max()))
    gdf = gpd.clip(gdf, domain_box)
    geom = gdf.union_all() if hasattr(gdf, 'union_all') else gdf.unary_union

    LO, LA = np.meshgrid(da_obs.lon.values, da_obs.lat.values)
    inside_mask = shapely.contains_xy(geom, LO.ravel(), LA.ravel()).reshape(LA.shape)
    india_mask = xr.DataArray(inside_mask, coords=[da_obs.lat, da_obs.lon], dims=['lat', 'lon'])

    da_obs_masked = da_obs.where(india_mask)

    def compute_profiles(da):
        mean_time = da.mean(dim='time', skipna=True)
        lat_prof = mean_time.mean(dim='lon', skipna=True).sortby('lat')
        lon_prof = mean_time.mean(dim='lat', skipna=True).sortby('lon')
        return lat_prof, lon_prof

    print("Computing Lat/Lon profiles for Observations...")
    lat_profiles, lon_profiles = {}, {}
    lat_profiles['MSWX (Observed)'], lon_profiles['MSWX (Observed)'] = compute_profiles(da_obs_masked)

    for mname in MODEL_ORDER:
        print(f"Computing Lat/Lon profiles for {mname}...")
        path, varname = MODELS_PATHS[mname]
        with xr.open_dataset(path) as ds_m:
            da_m = ds_m.sel(time=time_slice)[varname]
            da_m_aligned = da_m.sel(lat=da_obs.lat, lon=da_obs.lon, method='nearest').compute()
            da_m_masked = da_m_aligned.where(india_mask)
            
            lat_prof, lon_prof = compute_profiles(da_m_masked)
            lat_profiles[mname] = lat_prof
            lon_profiles[mname] = lon_prof

    ds_obs.close()
    return lat_profiles, lon_profiles

# ============================================================
# 2. STATISTICAL METRIC ENGINE
# ============================================================
def compute_profile_metrics(prof_mod, prof_obs, coord_name):
    """
    Computes RMSE, MAE, MBE (Mean Bias Error), Pearson Correlation,
    and identifies the location and magnitude of the peak localized error.
    """
    # Ensure exact coordinate alignment
    prof_mod, prof_obs = xr.align(prof_mod, prof_obs)
    
    valid = ~np.isnan(prof_mod.values) & ~np.isnan(prof_obs.values)
    m = prof_mod.values[valid]
    o = prof_obs.values[valid]
    coords = prof_mod[coord_name].values[valid]
    
    rmse = float(np.sqrt(np.mean((m - o)**2)))
    mae  = float(np.mean(np.abs(m - o)))
    mbe  = float(np.mean(m - o))
    corr, _ = pearsonr(m, o)
    
    # Locate peak localized error
    diff = m - o
    max_idx = np.argmax(np.abs(diff))
    max_err_val = float(diff[max_idx])
    max_coord   = float(coords[max_idx])
    
    return rmse, mae, mbe, float(corr), max_err_val, max_coord

# ============================================================
# 3. RUN ANALYSIS & PRINT MANUSCRIPT TABLES
# ============================================================
def run_statistical_analysis(lat_profiles, lon_profiles):
    sep = "=" * 110
    
    lat_obs = lat_profiles['MSWX (Observed)']
    lon_obs = lon_profiles['MSWX (Observed)']
    
    lat_stats, lon_stats = [], []
    
    for mname in MODEL_ORDER:
        # Meridional (Lat) Metrics
        r_lat, mae_lat, mbe_lat, c_lat, pk_val_lat, pk_loc_lat = compute_profile_metrics(
            lat_profiles[mname], lat_obs, 'lat'
        )
        lat_stats.append({
            'Model': mname,
            'RMSE (mm/d)': r_lat,
            'MAE (mm/d)': mae_lat,
            'MBE (mm/d)': mbe_lat,
            'Pearson r': c_lat,
            'Peak Bias (mm/d)': pk_val_lat,
            'Peak Lat (°N)': pk_loc_lat
        })
        
        # Zonal (Lon) Metrics
        r_lon, mae_lon, mbe_lon, c_lon, pk_val_lon, pk_loc_lon = compute_profile_metrics(
            lon_profiles[mname], lon_obs, 'lon'
        )
        lon_stats.append({
            'Model': mname,
            'RMSE (mm/d)': r_lon,
            'MAE (mm/d)': mae_lon,
            'MBE (mm/d)': mbe_lon,
            'Pearson r': c_lon,
            'Peak Bias (mm/d)': pk_val_lon,
            'Peak Lon (°E)': pk_loc_lon
        })
        
    df_lat = pd.DataFrame(lat_stats).set_index('Model')
    df_lon = pd.DataFrame(lon_stats).set_index('Model')
    
    # ---- Table 1: Meridional Profile Accuracy ----
    print(f"\n{sep}")
    print("TABLE 1 — MERIDIONAL PROFILE (LATITUDE) STATISTICAL FIDELITY (2011–2014)")
    print("Evaluates how well models capture the South-to-North precipitation gradient over Indian landmass")
    print(sep)
    with pd.option_context('display.float_format', '{:+.4f}'.format, 'display.width', 130):
        # Custom format for coordinates and non-signed metrics
        fmt_df = df_lat.copy()
        for col in ['RMSE (mm/d)', 'MAE (mm/d)', 'Pearson r', 'Peak Lat (°N)']:
            fmt_df[col] = fmt_df[col].map('{:.4f}'.format)
        print(fmt_df.to_string())

    # ---- Table 2: Zonal Profile Accuracy ----
    print(f"\n{sep}")
    print("TABLE 2 — ZONAL PROFILE (LONGITUDE) STATISTICAL FIDELITY (2011–2014)")
    print("Evaluates how well models capture the West-to-East moisture transport across India")
    print(sep)
    with pd.option_context('display.float_format', '{:+.4f}'.format, 'display.width', 130):
        fmt_df_lon = df_lon.copy()
        for col in ['RMSE (mm/d)', 'MAE (mm/d)', 'Pearson r', 'Peak Lon (°E)']:
            fmt_df_lon[col] = fmt_df_lon[col].map('{:.4f}'.format)
        print(fmt_df_lon.to_string())

    # ---- Table 3: Combined Spatial Profile Ranking ----
    print(f"\n{sep}")
    print("TABLE 3 — COMBINED 1D SPATIAL PROFILE MODEL RANKING")
    print("Lower Combined RMSE and Higher Combined Correlation = Superior Spatial Representation")
    print(sep)
    
    rank_df = pd.DataFrame(index=MODEL_ORDER)
    rank_df['Mean Profile RMSE'] = (df_lat['RMSE (mm/d)'] + df_lon['RMSE (mm/d)']) / 2.0
    rank_df['Mean Profile MAE']  = (df_lat['MAE (mm/d)']  + df_lon['MAE (mm/d)'])  / 2.0
    rank_df['Mean Correlation']  = (df_lat['Pearson r']   + df_lon['Pearson r'])   / 2.0
    
    rank_df['Rank_RMSE'] = rank_df['Mean Profile RMSE'].rank(ascending=True).astype(int)
    rank_df['Rank_Corr'] = rank_df['Mean Correlation'].rank(ascending=False).astype(int)
    rank_df['Overall Profile Rank'] = rank_df[['Rank_RMSE', 'Rank_Corr']].mean(axis=1).rank(ascending=True).astype(int)
    
    with pd.option_context('display.float_format', '{:.4f}'.format, 'display.width', 130):
        print(rank_df.to_string())
        
    # ============================================================
    # 4. EXPORT ALL DATA TO CSVS
    # ============================================================
    # Export summary tables
    df_lat.to_csv(os.path.join(output_dir, "meridional_profile_metrics.csv"))
    df_lon.to_csv(os.path.join(output_dir, "zonal_profile_metrics.csv"))
    rank_df.to_csv(os.path.join(output_dir, "spatial_profile_ranking.csv"))
    
    # Build and export full 1D curve dataframes
    df_lat_curves = pd.DataFrame({'Latitude (°N)': lat_obs.lat.values, 'MSWX (Observed)': lat_obs.values})
    df_lon_curves = pd.DataFrame({'Longitude (°E)': lon_obs.lon.values, 'MSWX (Observed)': lon_obs.values})
    
    df_lat_bias = pd.DataFrame({'Latitude (°N)': lat_obs.lat.values})
    df_lon_bias = pd.DataFrame({'Longitude (°E)': lon_obs.lon.values})
    
    for mname in MODEL_ORDER:
        df_lat_curves[mname] = lat_profiles[mname].values
        df_lon_curves[mname] = lon_profiles[mname].values
        
        df_lat_bias[f"{mname} Bias"] = lat_profiles[mname].values - lat_obs.values
        df_lon_bias[f"{mname} Bias"] = lon_profiles[mname].values - lon_obs.values
        
    df_lat_curves.to_csv(os.path.join(output_dir, "lat_profiles_absolute_1D.csv"), index=False)
    df_lon_curves.to_csv(os.path.join(output_dir, "lon_profiles_absolute_1D.csv"), index=False)
    df_lat_bias.to_csv(os.path.join(output_dir, "lat_profiles_bias_1D.csv"), index=False)
    df_lon_bias.to_csv(os.path.join(output_dir, "lon_profiles_bias_1D.csv"), index=False)
    
    print(f"\n✅ SUCCESS: Exported 3 metric CSVs and 4 full 1D curve CSVs to:\n  {output_dir}")

# ============================================================
# MAIN
# ============================================================
if __name__ == '__main__':
    print("\n" + "="*70)
    print("Zonal & Meridional Precipitation Profile Statistical Analysis")
    print("="*70)
    
    lat_profs, lon_profs = load_and_compute_profiles()
    run_statistical_analysis(lat_profs, lon_profs)
    
    print("\nProcess complete.")